<div style="direction: rtl; white-space: normal; line-height: 1.8;">
<h2>Notebook 13 — Golden Dataset v2 Migration</h2>
<p>این نسخه برای Golden Dataset v2 فریز‌شده با 3813 رکورد ساخته شده است.</p>
<p>Golden v2 بدون تغییر از artifacts/phase2/golden_dataset_v2 خوانده می‌شود و یک compatibility layer فقط در حافظه برای قراردادهای NB13 ساخته می‌شود.</p>
<p>وابستگی تاریخی به Golden v1، 28 seed قدیمی، integrity erratum و lineage نوت‌بوک‌های 12e/12f از مسیر ارزیابی حذف شده است.</p>
<p>نتایج Benchmark همچنان زیر namespace مبتنی بر SHA-256 دیتاست فعال ذخیره می‌شوند.</p>
</div>

In [1]:
# Code Cell 00
# Golden Dataset v2 bootstrap + NB13 compatibility adapter

from pathlib import Path
import hashlib
import json
import re

EXPECTED_GOLDEN_V2_RECORD_COUNT = 3813
EXPECTED_GOLDEN_V2_SHA256 = (
    "b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2"
)

cwd = Path.cwd().resolve()
project_root_candidates = [
    p for p in [cwd, *cwd.parents]
    if (p / "artifacts").is_dir() and (p / "notebooks").is_dir()
]
assert project_root_candidates, f"Project root not found from: {cwd}"
project_root = project_root_candidates[0]

golden_dataset_path = (
    project_root
    / "artifacts/phase2/golden_dataset_v2/golden_dataset_v2.jsonl"
)
golden_dataset_report_path = (
    project_root
    / "artifacts/phase2/golden_dataset_v2/golden_dataset_v2_final_report.json"
)

assert golden_dataset_path.is_file(), (
    f"Golden Dataset v2 not found: {golden_dataset_path}"
)

dataset_hash = hashlib.sha256(
    golden_dataset_path.read_bytes()
).hexdigest()

assert dataset_hash == EXPECTED_GOLDEN_V2_SHA256, (
    "Golden Dataset v2 hash mismatch. "
    f"Expected {EXPECTED_GOLDEN_V2_SHA256}, found {dataset_hash}"
)

raw_golden_dataset = [
    json.loads(line)
    for line in golden_dataset_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

assert len(raw_golden_dataset) == EXPECTED_GOLDEN_V2_RECORD_COUNT, (
    f"Expected {EXPECTED_GOLDEN_V2_RECORD_COUNT} records, "
    f"found {len(raw_golden_dataset)}"
)

golden_dataset = []

for record in raw_golden_dataset:
    evaluation_content = record.get("evaluation_content")
    assert isinstance(evaluation_content, dict), (
        f"Invalid evaluation_content: {record.get('candidate_id')}"
    )

    question = evaluation_content.get("question")
    expected_answer = evaluation_content.get("expected_answer")
    source_seed_slot_id = record.get("source_seed_slot_id")
    candidate_id = record.get("candidate_id")

    assert candidate_id
    assert source_seed_slot_id
    assert isinstance(question, str) and question.strip()
    assert isinstance(expected_answer, str) and expected_answer.strip()

    adapted = dict(record)

    # NB13 compatibility aliases. Golden v2 on disk is never modified.
    adapted["question"] = question
    adapted["expected_answer"] = expected_answer
    adapted["question_intent"] = evaluation_content.get("question_intent")
    adapted["source_seed_id"] = candidate_id
    adapted["source_lineage"] = source_seed_slot_id

    evidence_quotes = record.get("evidence_quotes") or []
    adapted["source_evidence"] = "\n\n".join(
        str(item).strip()
        for item in evidence_quotes
        if str(item).strip()
    )

    golden_dataset.append(adapted)

assert len({r["candidate_id"] for r in golden_dataset}) == len(golden_dataset)
assert all(r["source_evidence"] for r in golden_dataset), (
    "At least one Golden v2 record has no evidence quote."
)

golden_dataset_context = {
    "path": golden_dataset_path.as_posix(),
    "sha256": dataset_hash,
    "record_count": len(golden_dataset),
    "dataset_version": "golden_dataset_v2",
    "compatibility_adapter": "NB13_GOLDEN_V2_V1",
}

print("Active Golden Dataset:", golden_dataset_context["path"])
print("SHA-256:", golden_dataset_context["sha256"])
print("Record count:", golden_dataset_context["record_count"])
print("Compatibility adapter:", golden_dataset_context["compatibility_adapter"])
print("\nACTIVE GOLDEN DATASET V2: PASS")

Active Golden Dataset: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/golden_dataset_v2/golden_dataset_v2.jsonl
SHA-256: b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2
Record count: 3813
Compatibility adapter: NB13_GOLDEN_V2_V1

ACTIVE GOLDEN DATASET V2: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h1>Notebook 13 — Embedding Benchmark</h1>

<h2>مقایسۀ مدل‌های Embedding بر اساس Golden Dataset فعال</h2>

<p>
هدف این Notebook انتخاب مدل Embedding بر اساس Benchmark واقعی روی Golden Dataset فعال پروژه است.
انتخاب مدل بر مبنای شهرت، تنظیم پیش‌فرض یا حدس انجام نمی‌شود.
</p>

<p>
در این مرحله، Candidate Modelها همراه با Candidateهای Chunking Freeze‌شده NB11
ارزیابی می‌شوند تا کیفیت بازیابی، هزینه، Latency و Failureهای واقعی هر ترکیب اندازه‌گیری شود.
</p>

<h3>جایگاه در Pipeline</h3>

<p>
این Notebook پس از Golden Dataset و پیش از ساخت Dense Index قرار دارد.
</p>

<h3>Inputهای اصلی</h3>

<ul>
<li>متغیر مرکزی <code>golden_dataset</code></li>
<li>اطلاعات همراه آن در <code>golden_dataset_context</code></li>
<li>Candidateهای Chunking Freeze‌شده NB11</li>
<li>Candidate Modelهای Embedding که در مراحل بعد Discovery خواهند شد</li>
</ul>

<h3>Outputهای مورد انتظار</h3>

<ul>
<li>Registry مدل‌های Candidate</li>
<li>Config Benchmark نسخه‌دار</li>
<li>Metricهای واقعی Retrieval برای هر ترکیب Model و Chunking</li>
<li>تحلیل کیفیت، Latency، Failure و Cost</li>
<li>Evidence لازم برای انتخاب Embedding پیش از Dense Index</li>
</ul>

<h3>خارج از Scope</h3>

<ul>
<li>ساخت Dense Index نهایی</li>
<li>انتخاب نهایی Chunking</li>
<li>Sparse Retrieval، Hybrid Retrieval و Reranking</li>
<li>Generation، Citation و Safety</li>
<li>هرگونه تغییر در Golden Dataset</li>
</ul>

<h3>معیار موفقیت نهایی</h3>

<p>
Notebook فقط زمانی کامل است که تمام Candidateها با Golden Dataset فعال Benchmark شوند،
Metricها واقعاً اندازه‌گیری و ثبت شوند و
<code>Restart Kernel → Run All → Final Health PASS</code>
با موفقیت انجام شود.
</p>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>Starting Point / Execution Plan</h2>

<p>
Code Cell 00، Golden Dataset فعال را با موفقیت بارگذاری کرده است.
اکنون باید پیش از شناسایی مدل‌ها یا خواندن Candidateهای Chunking، وضعیت واقعی همین Dataset و Report همراه آن بررسی شود.
</p>

<p>
این Step فقط Discovery و Preflight است؛ هیچ Dataset، Index، Model، Config یا Artifact جدیدی نمی‌سازد یا تغییر نمی‌دهد.
</p>

<h3>کاری که انجام می‌شود</h3>

<ul>
<li>بررسی ساختار واقعی رکوردهای <code>golden_dataset</code></li>
<li>بررسی یکتایی شناسه‌های موجود بدون فرض‌کردن نام Field</li>
<li>خواندن Report همراه Dataset از روی Disk</li>
<li>ثبت Fieldها، Countها و Hashهای واقعی برای ادامه Benchmark</li>
</ul>

<h3>Validation</h3>

<ul>
<li>Golden Dataset فعال خالی نباشد.</li>
<li>تمام رکوردها JSON Object باشند.</li>
<li>Report همراه Dataset وجود داشته و قابل Parse باشد.</li>
<li>هیچ Path، Count، Hash یا Schema از قبل فرض نشود.</li>
</ul>

<p>
خروجی موفق این Step، Preflight واقعی Golden Dataset فعال برای ورود به Discovery ورودی‌های NB13 است.
</p>

</div>

In [2]:
# Code Cell 01

from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import hashlib
import json
import platform
import subprocess
import time

In [3]:
# Code Cell 02
# Golden v2 preflight

assert golden_dataset
assert len(golden_dataset) == 3813
assert golden_dataset_report_path.is_file(), (
    f"Golden Dataset v2 final report not found: {golden_dataset_report_path}"
)

golden_dataset_report = json.loads(
    golden_dataset_report_path.read_text(encoding="utf-8")
)
assert isinstance(golden_dataset_report, dict)

reported_dataset_sha256 = golden_dataset_context["sha256"]

field_counts = Counter()
for record in golden_dataset:
    field_counts.update(record.keys())

candidate_id_field = "candidate_id"
identifier_values = [record[candidate_id_field] for record in golden_dataset]
duplicate_identifier_count = len(identifier_values) - len(set(identifier_values))

assert duplicate_identifier_count == 0
assert all(record["question"].strip() for record in golden_dataset)
assert all(record["expected_answer"].strip() for record in golden_dataset)

print("=== NB13 GOLDEN V2 PREFLIGHT ===")
print("Dataset path:", golden_dataset_context["path"])
print("Dataset SHA-256:", golden_dataset_context["sha256"])
print("Dataset records:", golden_dataset_context["record_count"])
print("Report path:", golden_dataset_report_path.name)
print("Identifier field:", candidate_id_field)
print("Duplicate identifiers:", duplicate_identifier_count)
print("\nNB13 GOLDEN V2 PREFLIGHT: PASS")

=== NB13 GOLDEN V2 PREFLIGHT ===
Dataset path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/golden_dataset_v2/golden_dataset_v2.jsonl
Dataset SHA-256: b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2
Dataset records: 3813
Report path: golden_dataset_v2_final_report.json
Identifier field: candidate_id
Duplicate identifiers: 0

NB13 GOLDEN V2 PREFLIGHT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>اعتبارسنجی Golden Dataset فعال با Report و Integrity Erratum</h2>

<p>
Golden Dataset فعال از نظر نام فایل، تعداد رکورد و Target Size با Report همراه خود تطبیق دارد،
اما Hash فایل فعال با Hash ثبت‌شده در Report اولیه یکسان نیست.
</p>

<p>
در کنار Dataset، یک Integrity Erratum رسمی نیز وجود دارد. پیش از هر تصمیم،
باید محتوای واقعی این Erratum از روی Disk خوانده شود تا مشخص شود آیا تفاوت Hash
مستند، قابل‌ردیابی و پذیرفته‌شده است یا خیر.
</p>

<p>
این Step فقط Root Cause را بررسی می‌کند و هیچ فایل، Dataset، Report یا Artifactی را تغییر نمی‌دهد.
</p>

<h3>معیار PASS این مرحله</h3>

<p>
Erratum باید وجود داشته و قابل Parse باشد. سپس در Step بعد، Hash فعال فقط در صورتی پذیرفته می‌شود
که Erratum آن را به‌صورت صریح و قابل‌اعتبارسنجی پوشش دهد.
</p>

</div>

In [4]:
# Code Cell 03
# Golden v2 does not use the legacy v1 integrity-erratum contract.

golden_dataset_erratum_path = None
golden_dataset_erratum = {
    "artifact": golden_dataset_path.name,
    "actual_sha256": golden_dataset_context["sha256"],
    "legacy_report_sha256": None,
    "legacy_report_path": None,
    "records": golden_dataset_context["record_count"],
    "dataset_matches_git_head": None,
    "dataset_content_modified": False,
    "integrity_status": "PASS_FROZEN_GOLDEN_V2",
}
erratum_hash = None

print("=== GOLDEN V2 INTEGRITY CONTRACT ===")
print("Legacy integrity erratum required: False")
print("Frozen dataset SHA-256:", golden_dataset_context["sha256"])
print("Records:", golden_dataset_context["record_count"])
print("Integrity status:", golden_dataset_erratum["integrity_status"])
print("\nGOLDEN V2 INTEGRITY CONTRACT: PASS")

=== GOLDEN V2 INTEGRITY CONTRACT ===
Legacy integrity erratum required: False
Frozen dataset SHA-256: b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2
Records: 3813
Integrity status: PASS_FROZEN_GOLDEN_V2

GOLDEN V2 INTEGRITY CONTRACT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>پذیرش نهایی Integrity Golden Dataset فعال</h2>

<p>
Integrity Erratum رسمی تأیید کرد که تفاوت Hash میان Golden Dataset فعال و Report اولیه
ناشی از Metadata هاردکدشدهٔ قدیمی در Report است، نه تغییر در محتوای Dataset.
</p>

<p>
در این Step، Dataset فعال، Report اولیه و Erratum رسمی با هم Reconcilie می‌شوند.
پس از PASS شدن این Gate، Golden Dataset فعال به‌عنوان ورودی معتبر NB13 پذیرفته می‌شود.
</p>

<p>
این Step هیچ Artifactی را تغییر نمی‌دهد و فقط وضعیت Integrity را در حافظهٔ Notebook ثبت می‌کند.
</p>

<h3>معیار PASS</h3>

<ul>
<li>Hash فعال دقیقاً با Hash ثبت‌شده در Erratum برابر باشد.</li>
<li>Hash قدیمی Report با مقدار Legacy ثبت‌شده در Erratum برابر باشد.</li>
<li>تعداد رکوردها در Dataset و Erratum برابر باشد.</li>
<li>Erratum صراحتاً عدم تغییر محتوا را تأیید کند.</li>
<li>Erratum صراحتاً تطابق Dataset با Git Head را تأیید کند.</li>
</ul>

</div>

In [5]:
# Code Cell 04
# Frozen Golden v2 integrity reconciliation.

integrity_gates = {
    "dataset_hash_is_frozen_golden_v2": (
        golden_dataset_context["sha256"]
        == EXPECTED_GOLDEN_V2_SHA256
    ),
    "record_count_is_3813": (
        golden_dataset_context["record_count"]
        == EXPECTED_GOLDEN_V2_RECORD_COUNT
    ),
    "final_report_exists": golden_dataset_report_path.is_file(),
    "candidate_ids_unique": duplicate_identifier_count == 0,
}

failed_integrity_gates = [
    name for name, passed in integrity_gates.items() if not passed
]

print("=== ACTIVE GOLDEN V2 INTEGRITY RECONCILIATION ===")
for name, passed in integrity_gates.items():
    print(f"- {name}: {'PASS' if passed else 'FAIL'}")

assert not failed_integrity_gates, failed_integrity_gates

golden_dataset_integrity = {
    "status": "PASS",
    "active_dataset_sha256": golden_dataset_context["sha256"],
    "integrity_status": "PASS_FROZEN_GOLDEN_V2",
    "report_status": "GOLDEN_V2_FINAL_REPORT_PRESENT",
}

print("\nACTIVE GOLDEN DATASET INTEGRITY: PASS")

=== ACTIVE GOLDEN V2 INTEGRITY RECONCILIATION ===
- dataset_hash_is_frozen_golden_v2: PASS
- record_count_is_3813: PASS
- final_report_exists: PASS
- candidate_ids_unique: PASS

ACTIVE GOLDEN DATASET INTEGRITY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>کشف Artifactهای Freeze‌شدهٔ Chunking از NB11</h2>

<p>
Golden Dataset فعال با موفقیت پذیرفته شد. برای Benchmark Embedding، ورودی دوم
Candidateهای Chunking هستند که در NB11 با وضعیت
<code>FROZEN_FOR_EVALUATION</code>
ثبت شده‌اند.
</p>

<p>
در این Step ابتدا Manifest واقعی Freeze از روی Disk پیدا و خوانده می‌شود.
هدف فقط کشف Contract، وضعیت Freeze و مسیرهای ثبت‌شده در Manifest است.
</p>

<p>
این مرحله هیچ Chunk، Dataset، Model، Index یا Artifact جدیدی ایجاد یا تغییر نمی‌دهد.
</p>

<h3>معیار PASS</h3>

<ul>
<li>Repository Root بدون مسیر شخصی Hard-codeشده کشف شود.</li>
<li>Manifest Freeze یکتای NB11 روی Disk وجود داشته باشد.</li>
<li>Manifest قابل Parse باشد.</li>
<li>Contract و فیلدهای واقعی آن از روی فایل نمایش داده شوند.</li>
</ul>

</div>

In [6]:
# Code Cell 05

current_path = Path.cwd().resolve()

project_root = next(
    (
        candidate
        for candidate in (current_path, *current_path.parents)
        if (candidate / ".git").exists()
        and (candidate / "notebooks" / "phase2").exists()
    ),
    None,
)

assert project_root is not None, (
    "Project root could not be discovered."
)

freeze_manifest_matches = sorted(
    project_root.rglob(
        "evaluation_freeze_manifest_v1.json"
    )
)

assert len(freeze_manifest_matches) == 1, (
    "Expected exactly one NB11 evaluation freeze manifest, "
    f"found {len(freeze_manifest_matches)}."
)

chunking_freeze_manifest_path = freeze_manifest_matches[0]

chunking_freeze_manifest = json.loads(
    chunking_freeze_manifest_path.read_text(
        encoding="utf-8"
    )
)

assert isinstance(
    chunking_freeze_manifest,
    dict,
), "NB11 evaluation freeze manifest is not a JSON object."

manifest_hash = hashlib.sha256(
    chunking_freeze_manifest_path.read_bytes()
).hexdigest()

print("=== NB11 CHUNKING FREEZE DISCOVERY ===")
print(
    "Manifest path:",
    chunking_freeze_manifest_path.relative_to(
        project_root
    ),
)
print("Manifest SHA-256:", manifest_hash)

print("\nManifest fields:")
for field_name in sorted(chunking_freeze_manifest):
    print("-", field_name)

print("\nManifest preview:")
print(
    json.dumps(
        chunking_freeze_manifest,
        ensure_ascii=False,
        indent=2,
    )[:5000]
)

print("\nNB11 CHUNKING FREEZE DISCOVERY: PASS")

=== NB11 CHUNKING FREEZE DISCOVERY ===
Manifest path: reports/phase2/chunking/evaluation_freeze_manifest_v1.json
Manifest SHA-256: c29c4f1840e1fbd313c4f27b3556ff84f2041cce86ba0d5e27ff278c450106cc

Manifest fields:
- accepted_corpus
- artifacts
- candidate_reread_validation
- final_production_strategy
- final_selection_status
- freeze_scope
- measurement_tokenizer
- nb12_handoff
- next_notebook
- not_production_selection
- notebook
- retrieval_metrics
- status
- structural_shortlist

Manifest preview:
{
  "accepted_corpus": {
    "path": "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl",
    "records": 19456,
    "sha256": "9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40"
  },
  "artifacts": {
    "baseline_phase1_char_safe": {
      "artifact_type": "chunk_candidate",
      "bytes": 37213608,
      "freeze_status": "FROZEN_FOR_EVALUATION",
      "path": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
      "records": 3488,
   

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>اعتبارسنجی مستقل Artifactهای Freeze‌شدهٔ NB11</h2>

<p>
Manifest Freeze وجود و ساختار Candidateهای Chunking را تأیید کرد.
اکنون همهٔ Artifactهای ثبت‌شده در همان Manifest مستقیماً از روی Disk بررسی می‌شوند.
</p>

<p>
این Step تضمین می‌کند NB13 دقیقاً روی Candidateهایی اجرا شود که در NB11 Benchmark و
<code>FROZEN_FOR_EVALUATION</code>
شده‌اند؛ نه روی فایل ناقص، تغییرکرده یا نامرتبط.
</p>

<h3>بررسی‌ها</h3>

<ul>
<li>وجود تمام Artifactهای ثبت‌شده</li>
<li>تطابق اندازهٔ فایل با Manifest</li>
<li>تطابق SHA-256 فایل با Manifest</li>
<li>اعتبار وضعیت Freeze</li>
<li>استخراج Registry Candidateهای دارای نقش Retrieval برای مراحل بعد</li>
</ul>

<p>
این مرحله هیچ Candidate، Manifest یا فایل دیگری را تغییر نمی‌دهد.
</p>

</div>

In [7]:
# Code Cell 06

assert (
    chunking_freeze_manifest.get("status")
    == "FROZEN_FOR_EVALUATION"
), "NB11 manifest does not have FROZEN_FOR_EVALUATION status."

assert (
    chunking_freeze_manifest.get("final_production_strategy")
    in {None, "NOT_SELECTED"}
), "NB11 manifest incorrectly declares a final production strategy."

manifest_artifacts = chunking_freeze_manifest.get("artifacts")

assert isinstance(
    manifest_artifacts,
    dict,
), "NB11 manifest artifacts field is not a JSON object."

artifact_validation_results = {}
artifact_validation_failures = []

for artifact_name, descriptor in sorted(manifest_artifacts.items()):
    relative_path = descriptor.get("path")
    expected_sha256 = descriptor.get("sha256")
    expected_bytes = descriptor.get("bytes")

    assert isinstance(relative_path, str), (
        f"Artifact path is invalid: {artifact_name}"
    )

    artifact_path = project_root / relative_path

    result = {
        "path": relative_path,
        "exists": artifact_path.is_file(),
        "size_matches": False,
        "sha256_matches": False,
    }

    if artifact_path.is_file():
        actual_bytes = artifact_path.stat().st_size

        artifact_digest = hashlib.sha256()

        with artifact_path.open("rb") as handle:
            for chunk in iter(
                lambda: handle.read(1024 * 1024),
                b"",
            ):
                artifact_digest.update(chunk)

        actual_sha256 = artifact_digest.hexdigest()

        result["actual_bytes"] = actual_bytes
        result["actual_sha256"] = actual_sha256
        result["size_matches"] = (
            actual_bytes == expected_bytes
        )
        result["sha256_matches"] = (
            actual_sha256 == expected_sha256
        )

    artifact_validation_results[artifact_name] = result

    if not (
        result["exists"]
        and result["size_matches"]
        and result["sha256_matches"]
    ):
        artifact_validation_failures.append(
            artifact_name
        )

chunking_candidates = {
    artifact_name: {
        "path": descriptor["path"],
        "records": descriptor["records"],
        "role": descriptor["role"],
        "sha256": descriptor["sha256"],
    }
    for artifact_name, descriptor in manifest_artifacts.items()
    if descriptor.get("artifact_type") == "chunk_candidate"
}

print("=== NB11 FROZEN ARTIFACT VALIDATION ===")
print("Manifest artifacts:", len(manifest_artifacts))
print("Chunking candidates:", len(chunking_candidates))

print("\nCandidate registry:")
for candidate_name, candidate in sorted(
    chunking_candidates.items()
):
    print(
        f"- {candidate_name}: "
        f"role={candidate['role']}, "
        f"records={candidate['records']}, "
        f"path={candidate['path']}"
    )

print("\nFailed artifact validations:", len(artifact_validation_failures))

if artifact_validation_failures:
    print(artifact_validation_failures)

assert not artifact_validation_failures, (
    "NB11 frozen artifact validation failed: "
    f"{artifact_validation_failures}"
)

print("\nNB11 FROZEN ARTIFACT VALIDATION: PASS")

=== NB11 FROZEN ARTIFACT VALIDATION ===
Manifest artifacts: 15
Chunking candidates: 5

Candidate registry:
- baseline_phase1_char_safe: role=retrieval, records=3488, path=data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl
- parent_child_children: role=retrieval_child, records=2721, path=data/phase2/chunks/nb11_benchmark_v1/parent_child_children_v1.jsonl
- parent_child_parents: role=context_parent, records=1447, path=data/phase2/chunks/nb11_benchmark_v1/parent_child_parents_v1.jsonl
- structure_aware_medium: role=retrieval, records=1799, path=data/phase2/chunks/nb11_benchmark_v1/structure_aware_medium_v1.jsonl
- structure_aware_small: role=retrieval, records=2526, path=data/phase2/chunks/nb11_benchmark_v1/structure_aware_small_v1.jsonl

Failed artifact validations: 0

NB11 FROZEN ARTIFACT VALIDATION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>Discovery ساختار واقعی Candidateهای Chunking</h2>

<p>
Artifactهای Freeze‌شده از نظر فایل، Size و Hash معتبر هستند.
اکنون باید ساختار واقعی Recordهای JSONL هر Candidate بررسی شود.
</p>

<p>
هدف این Step کشف Fieldهای واقعی Chunk، Metadata، Locator، Lineage و شناسه‌ها است.
نام هیچ Fieldی از قبل فرض نمی‌شود.
</p>

<p>
این مرحله فقط اولین Record هر Candidate را می‌خواند و هیچ فایل یا Artifactی را تغییر نمی‌دهد.
</p>

<h3>خروجی مورد انتظار</h3>

<ul>
<li>Fieldهای واقعی هر Candidate</li>
<li>نوع دادهٔ هر Field در نمونهٔ واقعی</li>
<li>مقایسهٔ ابتدایی Contract میان Candidateها</li>
</ul>

</div>

In [8]:
# Code Cell 07

chunking_candidate_samples = {}

print("=== CHUNKING CANDIDATE RECORD DISCOVERY ===")

for candidate_name, candidate in sorted(
    chunking_candidates.items()
):
    candidate_path = project_root / candidate["path"]

    first_record = None

    with candidate_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                first_record = json.loads(line)
                break

    assert isinstance(
        first_record,
        dict,
    ), (
        f"First non-empty record is not a JSON object: "
        f"{candidate_name}"
    )

    field_types = {
        field_name: type(value).__name__
        for field_name, value in sorted(
            first_record.items()
        )
    }

    chunking_candidate_samples[candidate_name] = {
        "fields": sorted(first_record.keys()),
        "field_types": field_types,
    }

    print(f"\n{candidate_name}")
    print("Role:", candidate["role"])
    print("Fields:")

    for field_name, field_type in field_types.items():
        print(f"  - {field_name}: {field_type}")

print("\nCHUNKING CANDIDATE RECORD DISCOVERY: PASS")

=== CHUNKING CANDIDATE RECORD DISCOVERY ===

baseline_phase1_char_safe
Role: retrieval
Fields:
  - attribution_context: dict
  - boundary_metadata: dict
  - char_count: int
  - chunk_contract_version: str
  - chunk_count: int
  - chunk_id: str
  - chunk_index: int
  - content: str
  - content_origin: list
  - lineage: dict
  - locator_end: dict
  - locator_start: dict
  - member_document_ids: list
  - member_locators: list
  - parent_chunk_id: NoneType
  - rights: dict
  - role: str
  - source_id: str
  - source_type: str
  - strategy_id: str
  - strategy_version: str
  - token_count: int

parent_child_children
Role: retrieval_child
Fields:
  - attribution_context: dict
  - boundary_metadata: dict
  - char_count: int
  - chunk_contract_version: str
  - chunk_count: int
  - chunk_id: str
  - chunk_index: int
  - content: str
  - content_origin: list
  - lineage: dict
  - locator_end: dict
  - locator_start: dict
  - member_document_ids: list
  - member_locators: list
  - parent_chunk_id

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>اعتبارسنجی کامل Recordهای Candidateهای Chunking</h2>

<p>
Discovery ساختار نشان داد تمام Candidateها Contract یکسانی دارند و Fieldهای اصلی Chunk
در همه وجود دارند.
</p>

<p>
اکنون هر فایل Candidate به‌طور کامل خوانده می‌شود تا مشخص شود تمام Recordها JSON معتبر،
دارای <code>chunk_id</code> یکتا، محتوای غیرخالی و Fieldهای سازگار با Contract کشف‌شده هستند.
</p>

<p>
در پایان، فقط Candidateهایی که نقش Retrieval دارند در متغیر
<code>retrieval_chunking_candidates</code>
برای Benchmark Embedding ثبت می‌شوند.
</p>

<p>
این مرحله هیچ Chunk یا Artifactی را تغییر نمی‌دهد.
</p>

</div>

In [9]:
# Code Cell 08

reference_fields = set(
    next(
        iter(
            chunking_candidate_samples.values()
        )
    )["fields"]
)

candidate_record_validation = {}
candidate_record_validation_failures = []

for candidate_name, candidate in sorted(
    chunking_candidates.items()
):
    candidate_path = project_root / candidate["path"]

    record_count = 0
    json_failures = []
    field_contract_failures = []
    empty_chunk_ids = []
    duplicate_chunk_ids = []
    empty_content_chunk_ids = []
    chunk_ids = set()

    with candidate_path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                json_failures.append(
                    {
                        "line": line_number,
                        "message": str(error),
                    }
                )
                continue

            record_count += 1

            if not isinstance(record, dict):
                field_contract_failures.append(
                    {
                        "line": line_number,
                        "reason": "record_is_not_object",
                    }
                )
                continue

            if set(record.keys()) != reference_fields:
                field_contract_failures.append(
                    {
                        "line": line_number,
                        "reason": "field_contract_mismatch",
                    }
                )

            chunk_id = record.get("chunk_id")

            if not isinstance(chunk_id, str) or not chunk_id.strip():
                empty_chunk_ids.append(line_number)
            elif chunk_id in chunk_ids:
                duplicate_chunk_ids.append(chunk_id)
            else:
                chunk_ids.add(chunk_id)

            content = record.get("content")

            if not isinstance(content, str) or not content.strip():
                empty_content_chunk_ids.append(chunk_id)

    result = {
        "records": record_count,
        "expected_records": candidate["records"],
        "json_failures": len(json_failures),
        "field_contract_failures": len(field_contract_failures),
        "empty_chunk_ids": len(empty_chunk_ids),
        "duplicate_chunk_ids": len(duplicate_chunk_ids),
        "empty_content_chunk_ids": len(
            empty_content_chunk_ids
        ),
    }

    candidate_record_validation[candidate_name] = result

    candidate_passed = (
        record_count == candidate["records"]
        and not json_failures
        and not field_contract_failures
        and not empty_chunk_ids
        and not duplicate_chunk_ids
        and not empty_content_chunk_ids
    )

    if not candidate_passed:
        candidate_record_validation_failures.append(
            candidate_name
        )

retrieval_chunking_candidates = {
    candidate_name: candidate
    for candidate_name, candidate in chunking_candidates.items()
    if candidate["role"] in {
        "retrieval",
        "retrieval_child",
    }
}

print("=== FULL CHUNKING CANDIDATE RECORD VALIDATION ===")

for candidate_name, result in sorted(
    candidate_record_validation.items()
):
    print(f"\n{candidate_name}")
    for field_name, value in result.items():
        print(f"  {field_name}: {value}")

print(
    "\nRetrieval candidates:",
    sorted(retrieval_chunking_candidates),
)

print(
    "\nFailed candidates:",
    candidate_record_validation_failures,
)

assert not candidate_record_validation_failures, (
    "Chunking candidate record validation failed: "
    f"{candidate_record_validation_failures}"
)

print(
    "\nFULL CHUNKING CANDIDATE RECORD VALIDATION: PASS"
)

=== FULL CHUNKING CANDIDATE RECORD VALIDATION ===

baseline_phase1_char_safe
  records: 3488
  expected_records: 3488
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

parent_child_children
  records: 2721
  expected_records: 2721
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

parent_child_parents
  records: 1447
  expected_records: 1447
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

structure_aware_medium
  records: 1799
  expected_records: 1799
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

structure_aware_small
  records: 2526
  expected_records: 2526
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

Retrieval 

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>Discovery ساختار Gold Evidence در Golden Dataset فعال</h2>

<p>
Candidateهای Chunking آمادهٔ Benchmark هستند، اما پیش از اجرای هر مدل Embedding باید مشخص شود
هر سؤال Golden Dataset چگونه به Source Evidence، Lineage، Seed و Locator متصل شده است.
</p>

<p>
این Step ساختار واقعی Fieldهای
<code>source_evidence</code>،
<code>source_lineage</code>
و
<code>source_seed_id</code>
را در کل Golden Dataset بررسی می‌کند.
</p>

<p>
هدف این است که در مراحل بعد، معیارهای Retrieval دقیقاً براساس Gold Evidence واقعی ساخته شوند
و هیچ نگاشت Question به Chunk از روی حدس انجام نشود.
</p>

<p>
این مرحله فقط Discovery است و هیچ Dataset، Chunk، Model یا Artifactی را تغییر نمی‌دهد.
</p>

</div>

In [10]:
# Code Cell 09
# Golden v2 evidence contract discovery.

def canonical_exact_text(text):
    return " ".join(str(text).split())

def canonical_comparison_text(text):
    return canonical_exact_text(text)

source_evidence_structures = Counter()
source_lineage_structures = Counter()
source_seed_id_types = Counter()

for record in golden_dataset:
    source_evidence_structures[type(record.get("source_evidence")).__name__] += 1
    source_lineage_structures[type(record.get("source_lineage")).__name__] += 1
    source_seed_id_types[type(record.get("source_seed_id")).__name__] += 1

print("=== GOLDEN V2 EVIDENCE STRUCTURE DISCOVERY ===")
print("source_seed_id types:", dict(source_seed_id_types))
print("source_evidence types:", dict(source_evidence_structures))
print("source_lineage types:", dict(source_lineage_structures))
print("\nGOLDEN V2 EVIDENCE STRUCTURE DISCOVERY: PASS")

=== GOLDEN V2 EVIDENCE STRUCTURE DISCOVERY ===
source_seed_id types: {'str': 3813}
source_evidence types: {'str': 3813}
source_lineage types: {'str': 3813}

GOLDEN V2 EVIDENCE STRUCTURE DISCOVERY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>ساخت Registry واقعی Gold Evidence و Seedها</h2>

<p>
Discovery نشان داد Gold Evidence به‌صورت متن و Source Seed در Golden Dataset ذخیره شده است.
اکنون پیش از نگاشت Evidence به Chunkها، باید رابطهٔ واقعی میان Candidateها، Seedها و متن Evidence
به‌طور کامل اندازه‌گیری شود.
</p>

<p>
این Step مشخص می‌کند هر Seed چند Candidate Golden دارد، هر متن Evidence چند بار استفاده شده است
و آیا Evidence خالی یا ناسازگار وجود دارد یا نه.
</p>

<p>
هیچ Locator، Document ID یا Gold Chunk جدیدی در این مرحله ساخته نمی‌شود.
</p>

</div>

In [11]:
# Code Cell 10
# Golden v2 query/evidence registry.

gold_evidence_registry = {}
seed_candidate_counts = Counter()
lineage_counts = Counter()
normalized_questions = Counter()

for record in golden_dataset:
    candidate_id = record["candidate_id"]
    source_seed_id = record["source_seed_id"]
    source_evidence = record["source_evidence"]
    source_lineage = record["source_lineage"]
    question = record["question"]

    seed_candidate_counts[source_seed_id] += 1
    lineage_counts[source_lineage] += 1
    normalized_questions[canonical_exact_text(question)] += 1

    evidence_hash = hashlib.sha256(
        source_evidence.encode("utf-8")
    ).hexdigest()

    gold_evidence_registry[evidence_hash] = {
        "source_evidence": source_evidence,
        "source_seed_ids": {source_seed_id},
        "source_lineages": {source_lineage},
        "candidate_ids": [candidate_id],
    }

duplicate_question_count = sum(
    count - 1 for count in normalized_questions.values() if count > 1
)

print("=== GOLDEN V2 QUERY / EVIDENCE REGISTRY ===")
print("Golden candidates:", len(golden_dataset))
print("Evaluation identities:", len(seed_candidate_counts))
print("Evidence payloads:", len(gold_evidence_registry))
print("Duplicate normalized questions:", duplicate_question_count)
print("\nGOLDEN V2 QUERY / EVIDENCE REGISTRY: PASS")

=== GOLDEN V2 QUERY / EVIDENCE REGISTRY ===
Golden candidates: 3813
Evaluation identities: 3813
Evidence payloads: 3594
Duplicate normalized questions: 78

GOLDEN V2 QUERY / EVIDENCE REGISTRY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>نگاشت اولیهٔ Gold Evidence به Candidateهای Chunking با تطابق متن خام</h2>

<p>
Golden Dataset دارای ۲۸ متن Evidence یکتا است. اکنون این متن‌ها مستقیماً با محتوای Chunkهای
Candidateهای Retrieval مقایسه می‌شوند.
</p>

<p>
این مرحله فقط تطابق دقیق متن خام را می‌سنجد. اگر بخشی از Evidence به‌دلیل تفاوت فاصله،
نیم‌فاصله، ساختار Chunk یا Boundary پیدا نشود، هنوز هیچ نگاشت تقریبی یا اصلاحی انجام نمی‌دهیم.
</p>

<p>
هدف، اندازه‌گیری پایه‌ای Coverage Evidence برای هر Candidate است.
</p>

</div>

In [12]:
# Code Cell 11
# Golden v2 migration note: legacy v1 evidence-root-cause scan retired.
# Positive retrieval labels are constructed directly from Golden v2 document_ids
# in Code Cell 34, avoiding the old 28-seed / author-review coupling.

print("LEGACY V1 EVIDENCE SCAN RETIRED FOR GOLDEN V2: PASS")

LEGACY V1 EVIDENCE SCAN RETIRED FOR GOLDEN V2: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>بررسی تطابق سطحی Gold Evidence پس از Canonical Comparison</h2>

<p>
تطابق خام Gold Evidence با Chunkها صفر بود. این نتیجه به‌تنهایی ثابت نمی‌کند که Evidence در Corpus وجود ندارد،
زیرا تفاوت‌های Unicode، فاصله، نیم‌فاصله، نویسه‌های Formatting و شکل عربی/فارسی حروف می‌توانند
تطابق خام را از بین ببرند.
</p>

<p>
در این Step یک Normalizer فقط برای <strong>مقایسه</strong> ساخته می‌شود.
این تابع هیچ Dataset یا Chunkی را تغییر نمی‌دهد و جایگزین Policy نرمال‌سازی رسمی NB08 نیست.
</p>

<h3>قواعد Comparison</h3>

<ul>
<li>Unicode NFC</li>
<li>حذف Soft Hyphen و نویسه‌های Formatting</li>
<li>یکسان‌سازی فاصله‌ها</li>
<li>یکسان‌سازی صرفاً مقایسه‌ای ی/ک عربی و فارسی</li>
<li>حفظ متن اصلی Dataset و Chunk بدون تغییر</li>
</ul>

<p>
اگر بعد از این مرحله نیز Coverage صفر یا ناکافی باشد، باید Root Cause در Lineage و منبع Gold Evidence بررسی شود.
</p>

</div>

In [13]:
# Code Cell 12
# Golden v2 migration note: legacy v1 evidence-root-cause scan retired.
# Positive retrieval labels are constructed directly from Golden v2 document_ids
# in Code Cell 34, avoiding the old 28-seed / author-review coupling.

print("LEGACY V1 EVIDENCE SCAN RETIRED FOR GOLDEN V2: PASS")

LEGACY V1 EVIDENCE SCAN RETIRED FOR GOLDEN V2: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>تحلیل واژگانی Root Cause برای نگاشت Gold Evidence به Chunkها</h2>

<p>
تطابق خام و Canonical هر دو صفر بودند. بنابراین Gold Evidence و Chunkها فقط اختلاف
فرمت سطحی ندارند و باید ارتباط آن‌ها به‌صورت واژگانی اندازه‌گیری شود.
</p>

<p>
در این Step برای هر متن Evidence، بیشترین هم‌پوشانی واژگانی با Chunkهای هر Candidate محاسبه می‌شود.
این یک Retrieval Benchmark نیست و هیچ مدل Embeddingی استفاده نمی‌شود؛ فقط یک تحلیل تشخیصی برای
کشف امکان یا عدم امکان ساخت Gold Mapping است.
</p>

<p>
این مرحله هیچ Gold Chunk را به‌طور قطعی انتخاب نمی‌کند و هیچ Artifactی را تغییر نمی‌دهد.
</p>

</div>

In [14]:
# Code Cell 13
# Golden v2 migration note: legacy v1 evidence-root-cause scan retired.
# Positive retrieval labels are constructed directly from Golden v2 document_ids
# in Code Cell 34, avoiding the old 28-seed / author-review coupling.

print("LEGACY V1 EVIDENCE SCAN RETIRED FOR GOLDEN V2: PASS")

LEGACY V1 EVIDENCE SCAN RETIRED FOR GOLDEN V2: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>کشف Artifactهای معتبر برای Gold Provenance Mapping</h2>

<p>
تحلیل واژگانی نشان داد Evidenceهای Golden Dataset با Chunkها مرتبط‌اند، اما متن آن‌ها دقیقاً یکسان نیست.
بنابراین انتخاب خودکار بهترین Chunk بر اساس شباهت واژگانی، به‌تنهایی Ground Truth معتبر ایجاد نمی‌کند.
</p>

<p>
در این Step، Artifactهای موجود پروژه که ممکن است ارتباط معتبر میان ۲۸ Seed،
Source Evidence، Locator و Pilot Review را نگه داشته باشند، از روی Disk کشف می‌شوند.
</p>

<p>
هدف، بازیابی Mapping واقعی از Artifactهای قبلی است؛ نه ساختن Mapping جدید بر اساس حدس.
</p>

<p>
این مرحله فقط Discovery است و هیچ Golden Label، Chunk، Dataset یا Artifactی را تغییر نمی‌دهد.
</p>

</div>

In [15]:
# Code Cell 14
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 14 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 14 skipped: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>Discovery ساختار Registry بازبینی انسانی و Source Packet</h2>

<p>
Artifactهای معتبر Provenance پیدا شدند. اکنون باید ساختار واقعی Registry اصلی Human Review
و Source Packet مربوط به Pilotهای ۲۳ تا ۲۸ بررسی شود.
</p>

<p>
هدف این Step پیدا کردن Fieldهای واقعیِ Case، Pilot، Seed، Target Evidence، Source، Locator
و تصمیم بازبینی است تا Gold Mapping از دادهٔ موجود بازیابی شود.
</p>

<p>
این مرحله فقط Read Only است و هیچ Registry، Source Packet، Golden Dataset یا Chunkی را تغییر نمی‌دهد.
</p>

</div>

In [16]:
# Code Cell 15
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 15 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 15 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۴. کشف ساختار رکوردهای منشأ

در سل قبل مشخص شد که منشأ ۲۸ پایلوت در دو منبع نگهداری شده است:

- ۲۲ پایلوت تکمیل‌شده در رجیستری بازبینی انسانی؛
- ۶ پایلوت تکمیلی در بستهٔ منبع.

در این مرحله فقط قرارداد داده‌ای هر رکورد را بررسی می‌کنیم: شمارهٔ پایلوت، شناسه‌ها، متن منبع، و هر ارجاعی که بتواند بعداً به چانک‌ها وصل شود.

هنوز هیچ نگاشتی ساخته نمی‌شود و هیچ شباهت متنی به‌عنوان حقیقت در نظر گرفته نمی‌شود.

</div>

In [17]:
# Code Cell 16
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 16 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 16 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۵. کشف جزئیات شواهد و ادعاهای منبع

در این مرحله، ساختار داخلی شواهد منبع در پرونده‌های بازبینی انسانی و ادعاهای Source Packet بررسی می‌شود.

هدف، یافتن ارجاع‌های صریح و قابل‌ردیابی است؛ مانند شناسهٔ سند، شناسهٔ منبع، locator یا متن منبع.  
شباهت متنی به‌تنهایی برای ساخت نگاشت حقیقت استفاده نخواهد شد.

</div>

In [18]:
# Code Cell 17
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 17 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 17 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۶. ممیزی اتصال صریح پایلوت‌ها به پرونده‌های منشأ

Golden Dataset هر منبع را با نام‌هایی مانند `pilot_1` تا `pilot_28` معرفی می‌کند.

در این مرحله بررسی می‌کنیم آیا رجیستری بازبینی انسانی نیز شمارهٔ پایلوت یا شناسهٔ seed را به‌صورت صریح نگه داشته است یا نه.  
فقط اتصال‌های صریح پذیرفته می‌شوند؛ ترتیب رکوردها یا شباهت متن مبنای نگاشت نخواهد بود.

</div>

In [19]:
# Code Cell 18
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 18 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 18 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۷. کشف رجیسترهای تاریخچه و نگاشت اصلی

برای ۲۲ پروندهٔ اول، اتصال شمارهٔ پایلوت به `pilot_case_id` هنوز صریحاً پیدا نشده است.

در این مرحله بخش‌های تاریخچه، وضعیت ادامهٔ کار، خلاصهٔ بازبینی و بازبینی مؤلف در رجیستری بررسی می‌شوند.  
هدف فقط یافتن نگاشت ثبت‌شده است؛ ترتیب رکوردها مبنای نتیجه‌گیری نخواهد بود.

</div>

In [20]:
# Code Cell 19
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 19 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 19 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۸. کشف قرارداد ۲۸ پروندهٔ آمادهٔ بازبینی مؤلف

رجیستری یک فهرست مستقل از ۲۸ پروندهٔ آمادهٔ بازبینی مؤلف دارد.

این فهرست محتمل‌ترین محل برای نگاشت رسمی میان شمارهٔ پایلوت، شناسهٔ پرونده و منشأ شواهد است.  
در این مرحله فقط ساختار آن بررسی می‌شود و هیچ نگاشتی ساخته یا تغییر داده نمی‌شود.

</div>

In [21]:
# Code Cell 20
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 20 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 20 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۹. آزمون تطابق دقیق منشأ Golden با شواهد بازبینی

برای بازیابی نگاشت پایلوت‌های ۱ تا ۲۲، متن شواهد Golden فقط با تطابق دقیقِ نرمال‌شده بررسی می‌شود.

تطابق تقریبی، رتبه‌بندی شباهت یا ترتیب رکوردها در این مرحله استفاده نمی‌شود.  
فقط تطابق دقیق و یکتا می‌تواند به‌عنوان اتصال ثبت‌شده پذیرفته شود.

</div>

In [22]:
# Code Cell 21
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 21 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 21 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۰. کشف ساختار فایل رسمی بازبینی ۲۸ پایلوت

تطابق دقیق متن Golden با قطعات شواهد، هیچ اتصال قابل‌قبولی ایجاد نکرد.

در این مرحله فایل رسمی بازبینی ۲۸ پایلوت بررسی می‌شود تا ببینیم آیا شمارهٔ پرونده، شمارهٔ پایلوت یا `pilot_case_id` به‌صورت صریح در آن ثبت شده است یا نه.

</div>

In [23]:
# Code Cell 22
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 22 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 22 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۱. مشاهدهٔ ساختار واقعی فایل بازبینی مؤلف

جست‌وجوی مستقیم برای واژه‌های پایلوت و پرونده در فایل بازبینی نتیجه‌ای نداشت.

در این مرحله فقط تیترها، سرستون‌های جدول و ابتدای ساختار متن نمایش داده می‌شود تا روش ثبت هر مورد مشخص شود.  
این مشاهده هیچ تغییری در داده‌ها ایجاد نمی‌کند.

</div>

In [24]:
# Code Cell 23
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 23 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 23 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۲. اتصال دقیق موارد فایل بازبینی به پرونده‌های رجیستری

فایل رسمی بازبینی، ۲۸ مورد شماره‌دار و متن منبع هر مورد را ثبت کرده است.

در این مرحله، متن منبع هر مورد فقط با تطابق دقیقِ نرمال‌شده به شواهد پرونده‌های رجیستری وصل می‌شود.  
اگر تطابق دقیق و یکتا نباشد، آن مورد بدون نگاشت باقی می‌ماند.

</div>

In [25]:
# Code Cell 24
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 24 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 24 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۳. تکمیل اتصال دقیق با همهٔ پاراگراف‌های هر مورد

برخی موارد با پاراگراف نخست فایل بازبینی به شواهد رجیستری وصل نشدند.

در این مرحله همهٔ پاراگراف‌های هر مورد بررسی می‌شوند.  
فقط تطابق دقیقِ متن با شواهد ثبت‌شده پذیرفته می‌شود و هنوز از ترتیب برای ساخت نگاشت استفاده نخواهد شد.

</div>

In [26]:
# Code Cell 25
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 25 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 25 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۴. آزمون اتصال دقیق موارد بازبینی به seedهای Golden

برای تکمیل زنجیرهٔ منشأ، سؤال‌های Golden با متن کامل هر مورد در فایل رسمی بازبینی مقایسه می‌شوند.

فقط سؤال‌هایی پذیرفته می‌شوند که به‌صورت دقیق و نرمال‌شده در متن همان مورد وجود داشته باشند.  
این آزمون، اتصال «مورد بازبینی → source_seed_id» را مستقل از ترتیب فهرست بررسی می‌کند.

</div>

In [27]:
# Code Cell 26
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 26 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 26 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۵. کشف نگاشت ثبت‌شده در نوت‌بوک یکپارچه‌سازی بازخورد مؤلف

تطابق دقیق متن منبع و سؤال، اتصال مستقیمی میان موارد بازبینی و seedهای Golden ایجاد نکرد.

در این مرحله نوت‌بوک یکپارچه‌سازی بازخورد مؤلف فقط خوانده و جست‌وجو می‌شود تا نگاشت ثبت‌شده میان پرونده‌های بازبینی، `source_seed_id` و پایلوت‌ها پیدا شود.

</div>

In [28]:
# Code Cell 27
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 27 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 27 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۶. بازیابی Lineage واقعی از نوت‌بوک تولید Golden

نوت‌بوک یکپارچه‌سازی بازخورد مؤلف نشان داد که منطق اصلی اتصال seedها به رکوردهای Golden در نوت‌بوک تولید Golden قرار دارد.

در این مرحله فقط سل‌های مرتبط NB12e خوانده می‌شوند تا مسیر واقعی
`Seed → Generated Records → Golden Dataset`
از خود کد تولید بازیابی شود.

</div>

In [29]:
# Code Cell 28
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 28 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 28 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۷. ساخت رجیستری قطعی منشأ Golden Dataset

منطق تولید NB12e مسیر زیر را به‌صورت مستقیم ثبت کرده است:

`مورد N بازبینی → pilot_number N → source_seed_id N → source_lineage pilot_N`

در این سل، برای هر seed فعال Golden Dataset، منشأ مستند آن شامل شناسهٔ پرونده، شناسهٔ سندها و نوع منبع ساخته می‌شود.

این رجیستری از Golden Dataset فعال ساخته می‌شود و در صورت ناسازگاری lineage، اجرای ارزیابی را متوقف می‌کند.

</div>

In [30]:
# Code Cell 29
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 29 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 29 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۸. بررسی پوشش سندهای منشأ در کاندیدهای چانکینگ

رجیستری منشأ برای هر seed، شناسهٔ سندهای معتبر را مشخص کرده است.

در این مرحله بررسی می‌شود که هر شناسهٔ سند در کدام چانک‌های هر کاندید چانکینگ وجود دارد.  
Seedهایی که شناسهٔ سند ندارند یا هیچ چانک متناظری ندارند، فعلاً وارد محاسبهٔ معیارهای retrieval نمی‌شوند.

</div>

In [31]:
# Code Cell 30
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 30 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 30 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۹. بازیابی شناسهٔ منشأ برای seed بدون document_id

۲۷ seed دارای شناسهٔ سند، در همهٔ کاندیدهای چانکینگ پوشش کامل دارند.

در این مرحله فقط شواهد ثبت‌شدهٔ seed بدون document_id بررسی می‌شود تا مشخص شود آیا شناسهٔ معتبر دیگری مانند `source_id` یا `source_path` برای آن وجود دارد یا نه.

</div>

In [32]:
# Code Cell 31
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 31 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 31 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۰. جست‌وجوی دقیق مسیر transcript در metadata چانک‌ها

برای seed 16، مسیر فایل transcript و بازهٔ خطوط منشأ ثبت شده است، اما `document_id` ثبت نشده است.

در این مرحله همین مسیر به‌صورت دقیق در رکوردهای چانک جست‌وجو می‌شود تا شناسهٔ سند متناظر، در صورت وجود، از metadata واقعی بازیابی شود.

</div>

In [33]:
# Code Cell 32
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 32 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 32 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۱. بازیابی چانک‌های مثبت دقیق برای seed 16

مسیر منشأ seed 16 در metadata چانک‌ها پیدا شد، اما کل فایل transcript برای ارزیابی بسیار گسترده است.

در این مرحله فقط چانک‌هایی مثبت محسوب می‌شوند که متن بلندِ شاهد انسانی seed 16 را به‌صورت دقیق و نرمال‌شده در خود دارند.

</div>

In [34]:
# Code Cell 33
# Golden v2 migration: historical v1 28-pilot provenance step retired.
# Golden v2 carries document_ids/evidence_ids/evidence_locators directly.

print("GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 33 skipped: PASS")

GOLDEN V2 DIRECT PROVENANCE MODE — legacy step 33 skipped: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۲. ساخت رجیستری نهایی ارزیابی Retrieval

برای ۲۷ seed، چانک‌های مثبت از شناسهٔ سندهای ثبت‌شده ساخته شده‌اند.

برای seed 16، فقط در روش `structure_aware_medium` یک چانک مثبت با تطابق دقیق شاهد انسانی وجود دارد.  
بنابراین هر روش فقط با seedهایی ارزیابی می‌شود که برای همان روش، چانک مثبت معتبر دارند.

</div>

In [35]:
# Code Cell 34
# Golden v2 direct provenance -> positive chunk registry.

gold_records_by_seed = {
    record["source_seed_id"]: record
    for record in golden_dataset
}
active_seed_ids = sorted(gold_records_by_seed)

assert len(active_seed_ids) == golden_dataset_context["record_count"]

candidate_document_to_chunk_ids = {}
candidate_chunk_count_by_name = {}
retrieval_evaluation_registry = {}

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])
    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    assert candidate_path.is_file(), (
        f"Chunk candidate file is missing: {candidate_path}"
    )

    document_to_chunk_ids = defaultdict(set)
    chunk_count = 0

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)
            chunk_count += 1
            chunk_id = chunk["chunk_id"]

            document_ids = {
                value
                for value in chunk.get("member_document_ids", [])
                if isinstance(value, str) and value
            }

            if isinstance(chunk.get("source_id"), str) and chunk["source_id"]:
                document_ids.add(chunk["source_id"])

            for document_id in document_ids:
                document_to_chunk_ids[document_id].add(chunk_id)

    candidate_document_to_chunk_ids[candidate_name] = document_to_chunk_ids
    candidate_chunk_count_by_name[candidate_name] = chunk_count

    seed_labels = {}

    for source_seed_id in active_seed_ids:
        record = gold_records_by_seed[source_seed_id]
        expected_document_ids = [
            value
            for value in record.get("document_ids", [])
            if isinstance(value, str) and value
        ]

        matched_document_ids = [
            document_id
            for document_id in expected_document_ids
            if document_id in document_to_chunk_ids
        ]

        positive_chunk_ids = set()
        for document_id in matched_document_ids:
            positive_chunk_ids.update(document_to_chunk_ids[document_id])

        eligible = bool(positive_chunk_ids)

        seed_labels[source_seed_id] = {
            "eligible": eligible,
            "positive_chunk_ids": positive_chunk_ids,
            "label_basis": "GOLDEN_V2_DOCUMENT_IDS",
            "expected_document_ids": expected_document_ids,
            "matched_document_ids": matched_document_ids,
        }

    retrieval_evaluation_registry[candidate_name] = seed_labels

print("=== GOLDEN V2 RETRIEVAL EVALUATION REGISTRY ===")
print("Golden v2 evaluation identities:", len(active_seed_ids))

for candidate_name, seed_labels in retrieval_evaluation_registry.items():
    eligible = sum(1 for x in seed_labels.values() if x["eligible"])
    excluded = len(seed_labels) - eligible
    positives = sum(len(x["positive_chunk_ids"]) for x in seed_labels.values())

    print(f"\n{candidate_name}")
    print("- Chunk records:", candidate_chunk_count_by_name[candidate_name])
    print("- Eligible queries:", eligible)
    print("- Excluded queries:", excluded)
    print("- Positive chunk labels:", positives)

    assert eligible > 0, (
        f"{candidate_name} has no Golden v2 queries with positive chunk labels."
    )

print("\nGOLDEN V2 RETRIEVAL EVALUATION REGISTRY: PASS")

=== GOLDEN V2 RETRIEVAL EVALUATION REGISTRY ===
Golden v2 evaluation identities: 3813

baseline_phase1_char_safe
- Chunk records: 3488
- Eligible queries: 3813
- Excluded queries: 0
- Positive chunk labels: 7267

parent_child_children
- Chunk records: 2721
- Eligible queries: 3813
- Excluded queries: 0
- Positive chunk labels: 4798

structure_aware_medium
- Chunk records: 1799
- Eligible queries: 3813
- Excluded queries: 0
- Positive chunk labels: 4150

structure_aware_small
- Chunk records: 2526
- Eligible queries: 3813
- Excluded queries: 0
- Positive chunk labels: 4728

GOLDEN V2 RETRIEVAL EVALUATION REGISTRY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۳. بررسی محیط اجرای Embedding

رجیستری ارزیابی retrieval آماده است.

در این مرحله فقط محیط اجرای مدل‌های embedding بررسی می‌شود: کتابخانه‌های نصب‌شده، نسخه‌ها و دسترسی سخت‌افزاری.  
هیچ مدل embedding دانلود یا اجرا نمی‌شود.

</div>

In [36]:
# Code Cell 35

import importlib.util
import sys

embedding_runtime_packages = [
    "numpy",
    "torch",
    "transformers",
    "sentence_transformers",
    "faiss",
    "sklearn",
]

print("=== EMBEDDING RUNTIME DISCOVERY ===")
print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())

available_packages = {}

for package_name in embedding_runtime_packages:
    package_spec = importlib.util.find_spec(package_name)
    available = package_spec is not None
    available_packages[package_name] = available

    if available:
        package = __import__(package_name)
        package_version = getattr(package, "__version__", "version unavailable")
        print(f"- {package_name}: AVAILABLE ({package_version})")
    else:
        print(f"- {package_name}: NOT AVAILABLE")

if available_packages["torch"]:
    import torch

    print("\nPyTorch runtime:")
    print("- CUDA available:", torch.cuda.is_available())
    print(
        "- MPS available:",
        bool(
            hasattr(torch.backends, "mps")
            and torch.backends.mps.is_available()
        ),
    )

    if torch.cuda.is_available():
        print("- CUDA device:", torch.cuda.get_device_name(0))

assert available_packages["numpy"], (
    "numpy is required for embedding benchmark preparation."
)

print("\nEMBEDDING RUNTIME DISCOVERY: PASS")

=== EMBEDDING RUNTIME DISCOVERY ===
Python version: 3.11.3
Platform: macOS-12.7.6-x86_64-i386-64bit
- numpy: AVAILABLE (1.24.3)
- torch: AVAILABLE (2.2.2)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- transformers: AVAILABLE (4.53.2)
- sentence_transformers: AVAILABLE (5.0.0)
- faiss: NOT AVAILABLE
- sklearn: AVAILABLE (1.6.1)

PyTorch runtime:
- CUDA available: False
- MPS available: False

EMBEDDING RUNTIME DISCOVERY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۴. کشف مدل‌های embedding موجود در cache محلی

محیط لازم برای اجرای embedding فراهم است و اجرای benchmark روی CPU انجام خواهد شد.

در این مرحله فقط مدل‌های موجود در cache محلی بررسی می‌شوند تا پیش از هر دانلود یا انتخاب مدل، وضعیت واقعی دستگاه مشخص باشد.

</div>

In [37]:
# Code Cell 36

huggingface_cache_roots = [
    Path.home() / ".cache" / "huggingface" / "hub",
    Path.home() / "Library" / "Caches" / "huggingface" / "hub",
]

print("=== LOCAL EMBEDDING MODEL CACHE DISCOVERY ===")

local_model_snapshots = []

for cache_root in huggingface_cache_roots:
    print("\nCache root:", cache_root)
    print("Exists:", cache_root.is_dir())

    if not cache_root.is_dir():
        continue

    for model_directory in sorted(cache_root.glob("models--*")):
        snapshot_directory = model_directory / "snapshots"

        if not snapshot_directory.is_dir():
            continue

        for snapshot_path in sorted(snapshot_directory.iterdir()):
            if snapshot_path.is_dir():
                local_model_snapshots.append(
                    {
                        "model_cache_name": model_directory.name,
                        "snapshot_id": snapshot_path.name,
                        "path": snapshot_path,
                        "has_config": (snapshot_path / "config.json").is_file(),
                        "has_sentence_transformers_config": (
                            snapshot_path / "modules.json"
                        ).is_file(),
                    }
                )

print("\nLocal model snapshots found:", len(local_model_snapshots))

for model_snapshot in local_model_snapshots:
    print(
        "- "
        f"{model_snapshot['model_cache_name']} | "
        f"snapshot={model_snapshot['snapshot_id']} | "
        f"config={model_snapshot['has_config']} | "
        f"sentence_transformers={model_snapshot['has_sentence_transformers_config']}"
    )

print("\nLOCAL EMBEDDING MODEL CACHE DISCOVERY: PASS")

=== LOCAL EMBEDDING MODEL CACHE DISCOVERY ===

Cache root: /Users/macbookpro/.cache/huggingface/hub
Exists: True

Cache root: /Users/macbookpro/Library/Caches/huggingface/hub
Exists: False

Local model snapshots found: 3
- models--intfloat--multilingual-e5-small | snapshot=614241f622f53c4eeff9890bdc4f31cfecc418b3 | config=True | sentence_transformers=True
- models--intfloat--multilingual-e5-small | snapshot=c007d7ef6fd86656326059b28395a7a03a7c5846 | config=True | sentence_transformers=True
- models--sentence-transformers--all-MiniLM-L6-v2 | snapshot=c9745ed1d9f207416be6d2e6f8de32d1f16199bf | config=True | sentence_transformers=True

LOCAL EMBEDDING MODEL CACHE DISCOVERY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۵. اعتبارسنجی مدل‌های embedding محلی

دو مدل محلی برای benchmark در دسترس هستند.

در این مرحله هر مدل فقط با چند متن کوتاه بارگذاری و اعتبارسنجی می‌شود: بُعد embedding، شکل خروجی و نبود مقدار نامعتبر.  
هیچ جست‌وجو یا معیار retrieval محاسبه نمی‌شود.

</div>

In [38]:
# Code Cell 37

import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model_registry = {
    "multilingual_e5_small": {
        "model_id": "intfloat/multilingual-e5-small",
        "query_prefix": "query: ",
        "passage_prefix": "passage: ",
    },
    "all_minilm_l6_v2": {
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "query_prefix": "",
        "passage_prefix": "",
    },
}

embedding_models = {}

sanity_query = "چگونه می‌توان هزینه‌های غیرضروری را مدیریت کرد؟"
sanity_passages = [
    "مدیریت خرج یعنی هزینه کردن برای نیازها و هدف‌های واقعی.",
    "بازار سهام محل معامله اوراق بهادار است.",
]

print("=== LOCAL EMBEDDING MODEL VALIDATION ===")

for model_name, model_config in embedding_model_registry.items():
    print(f"\nLoading: {model_name}")
    print("Model ID:", model_config["model_id"])

    model = SentenceTransformer(
        model_config["model_id"],
        local_files_only=True,
        device="cpu",
    )

    encoded_query = model.encode(
        model_config["query_prefix"] + sanity_query,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    encoded_passages = model.encode(
        [
            model_config["passage_prefix"] + passage
            for passage in sanity_passages
        ],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    assert encoded_query.ndim == 1, (
        f"{model_name} query embedding must be one-dimensional."
    )
    assert encoded_passages.ndim == 2, (
        f"{model_name} passage embeddings must be two-dimensional."
    )
    assert encoded_passages.shape[0] == len(sanity_passages), (
        f"{model_name} returned an unexpected passage count."
    )
    assert encoded_query.shape[0] == encoded_passages.shape[1], (
        f"{model_name} query and passage dimensions do not match."
    )
    assert not np.isnan(encoded_query).any(), (
        f"{model_name} query embedding contains NaN."
    )
    assert not np.isnan(encoded_passages).any(), (
        f"{model_name} passage embeddings contain NaN."
    )

    embedding_models[model_name] = model

    print("Embedding dimension:", encoded_query.shape[0])
    print("Passage embedding shape:", encoded_passages.shape)

print("\nLOCAL EMBEDDING MODEL VALIDATION: PASS")

=== LOCAL EMBEDDING MODEL VALIDATION ===

Loading: multilingual_e5_small
Model ID: intfloat/multilingual-e5-small
Embedding dimension: 384
Passage embedding shape: (2, 384)

Loading: all_minilm_l6_v2
Model ID: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
Passage embedding shape: (2, 384)

LOCAL EMBEDDING MODEL VALIDATION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۶. بررسی امن تنظیمات Embedding اروان

مدل‌های محلی با موفقیت اعتبارسنجی شدند.

در این مرحله فقط وجود تنظیمات محلی اروان بررسی می‌شود: کلید API، آدرس Gateway و کتابخانهٔ لازم.  
هیچ مقدار محرمانه چاپ نمی‌شود و هیچ درخواست شبکه‌ای ارسال نخواهد شد.

</div>

In [39]:
# Code Cell 38

import os
from dotenv import load_dotenv
from urllib.parse import urlparse

arvan_env_path = project_root / ".env"

print("=== ARVAN EMBEDDING CONFIGURATION PREFLIGHT ===")
print("Environment file exists:", arvan_env_path.is_file())

if arvan_env_path.is_file():
    load_dotenv(arvan_env_path, override=False)

arvan_api_key = os.getenv("ARVAN_AI_API_KEY")
arvan_embedding_url = os.getenv("ARVAN_EMBEDDING_URL")

openai_package_available = importlib.util.find_spec("openai") is not None

print("OpenAI-compatible client available:", openai_package_available)
print("Arvan API key available:", bool(arvan_api_key))
print("Arvan embedding gateway available:", bool(arvan_embedding_url))

if arvan_embedding_url:
    parsed_gateway_url = urlparse(arvan_embedding_url)
    print("Gateway scheme:", parsed_gateway_url.scheme)
    print("Gateway host:", parsed_gateway_url.netloc)
    print("Gateway path configured:", bool(parsed_gateway_url.path))

assert openai_package_available, (
    "The openai package is required for the Arvan-compatible client."
)
assert arvan_api_key, (
    "ARVAN_AI_API_KEY is not available in the current environment."
)
assert arvan_embedding_url, (
    "ARVAN_EMBEDDING_URL is not available in the current environment."
)

print("\nARVAN EMBEDDING CONFIGURATION PREFLIGHT: PASS")

=== ARVAN EMBEDDING CONFIGURATION PREFLIGHT ===
Environment file exists: True
OpenAI-compatible client available: True
Arvan API key available: True
Arvan embedding gateway available: True
Gateway scheme: https
Gateway host: arvancloudai.ir
Gateway path configured: True

ARVAN EMBEDDING CONFIGURATION PREFLIGHT: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۷. اعتبارسنجی اتصال واقعی Embedding اروان

تنظیمات محلی اروان با موفقیت بررسی شد.

در این مرحله یک درخواست کوتاه به مدل قبلی ارسال می‌شود تا اتصال واقعی، شکل خروجی و بُعد embedding تأیید شود.  
هنوز هیچ چانک یا پرسش Golden Dataset به‌صورت انبوه ارسال نمی‌شود.

</div>

In [40]:
# Code Cell 39

from openai import OpenAI

arvan_embedding_model_id = "text-embedding-3-large"

arvan_embedding_client = OpenAI(
    api_key=arvan_api_key,
    base_url=arvan_embedding_url,
    timeout=30.0,
    max_retries=0,
)

print("=== ARVAN EMBEDDING LIVE CONNECTION VALIDATION ===")
print("Model:", arvan_embedding_model_id)

arvan_test_response = arvan_embedding_client.embeddings.create(
    model=arvan_embedding_model_id,
    input=sanity_query,
)

arvan_test_embedding = np.asarray(
    arvan_test_response.data[0].embedding,
    dtype=np.float32,
)

assert arvan_test_embedding.ndim == 1, (
    "Arvan embedding response must be one-dimensional."
)
assert arvan_test_embedding.size > 0, (
    "Arvan embedding response is empty."
)
assert not np.isnan(arvan_test_embedding).any(), (
    "Arvan embedding response contains NaN."
)

embedding_model_registry["arvan_text_embedding_3_large"] = {
    "model_id": arvan_embedding_model_id,
    "provider": "arvan",
    "embedding_dimension": int(arvan_test_embedding.size),
}

print("Embedding dimension:", arvan_test_embedding.size)
print("Response items:", len(arvan_test_response.data))
print("\nARVAN EMBEDDING LIVE CONNECTION VALIDATION: PASS")

=== ARVAN EMBEDDING LIVE CONNECTION VALIDATION ===
Model: text-embedding-3-large
Embedding dimension: 3072
Response items: 1

ARVAN EMBEDDING LIVE CONNECTION VALIDATION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۸. برآورد حجم اجرای Benchmark

سه مدل آمادهٔ ارزیابی هستند: دو مدل محلی و یک مدل اروان.

در این مرحله حجم واقعی دادهٔ benchmark بررسی می‌شود تا تعداد درخواست‌ها و حجم تقریبی ورودی پیش از اجرای انبوه مشخص باشد.  
هیچ embedding جدیدی ساخته یا به سرویس اروان ارسال نمی‌شود.

</div>

In [41]:
# Code Cell 40

benchmark_query_records = [
    {
        "candidate_id": record["candidate_id"],
        "source_seed_id": record["source_seed_id"],
        "question": canonical_exact_text(record["question"]),
    }
    for record in golden_dataset
]

assert len(benchmark_query_records) == golden_dataset_context["record_count"], (
    "Benchmark query count must match the active Golden Dataset."
)

candidate_workload_summary = {}
all_canonical_chunk_texts = set()

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])

    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    canonical_chunk_texts = []

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)
            canonical_chunk_text = canonical_exact_text(chunk["content"])
            canonical_chunk_texts.append(canonical_chunk_text)
            all_canonical_chunk_texts.add(canonical_chunk_text)

    candidate_workload_summary[candidate_name] = {
        "chunk_count": len(canonical_chunk_texts),
        "unique_chunk_text_count": len(set(canonical_chunk_texts)),
        "character_count": sum(
            len(text)
            for text in canonical_chunk_texts
        ),
    }

unique_query_texts = {
    record["question"]
    for record in benchmark_query_records
}

print("=== EMBEDDING BENCHMARK WORKLOAD DISCOVERY ===")
print("Golden query records:", len(benchmark_query_records))
print("Unique Golden query texts:", len(unique_query_texts))
print(
    "Golden query characters:",
    sum(len(question) for question in unique_query_texts),
)

print("\nChunk workload by candidate:")

for candidate_name, summary in candidate_workload_summary.items():
    print(f"\n{candidate_name}")
    print("- Chunk records:", summary["chunk_count"])
    print("- Unique chunk texts within candidate:", summary["unique_chunk_text_count"])
    print("- Chunk characters:", summary["character_count"])

print("\nCross-candidate unique chunk texts:", len(all_canonical_chunk_texts))
print(
    "Cross-candidate unique chunk characters:",
    sum(len(text) for text in all_canonical_chunk_texts),
)

estimated_remote_inputs = (
    len(unique_query_texts)
    + len(all_canonical_chunk_texts)
)

print("\nEstimated remote embedding inputs with text-level reuse:")
print(estimated_remote_inputs)

print("\nEMBEDDING BENCHMARK WORKLOAD DISCOVERY: PASS")

=== EMBEDDING BENCHMARK WORKLOAD DISCOVERY ===
Golden query records: 3813
Unique Golden query texts: 3735
Golden query characters: 187388

Chunk workload by candidate:

baseline_phase1_char_safe
- Chunk records: 3488
- Unique chunk texts within candidate: 3476
- Chunk characters: 3645591

parent_child_children
- Chunk records: 2721
- Unique chunk texts within candidate: 2709
- Chunk characters: 3139519

structure_aware_medium
- Chunk records: 1799
- Unique chunk texts within candidate: 1787
- Chunk characters: 3140441

structure_aware_small
- Chunk records: 2526
- Unique chunk texts within candidate: 2514
- Chunk characters: 3139714

Cross-candidate unique chunk texts: 7698
Cross-candidate unique chunk characters: 10066444

Estimated remote embedding inputs with text-level reuse:
11433

EMBEDDING BENCHMARK WORKLOAD DISCOVERY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۹. آماده‌سازی cache اجرای Benchmark

Benchmark embedding برای Golden Dataset فعال اجرا خواهد شد.

در این مرحله محل نگهداری خروجی‌ها، شناسهٔ اجرای فعلی و manifest قابل‌ردیابی ساخته می‌شود تا اجرای مدل‌ها قابل ادامه باشد و هر Golden Dataset فعال، خروجی مستقل خود را داشته باشد.

</div>

In [42]:
# Code Cell 41

embedding_benchmark_root = (
    project_root
    / "artifacts"
    / "phase2"
    / "embedding_benchmark"
)

active_benchmark_run_directory = (
    embedding_benchmark_root
    / golden_dataset_context["sha256"]
)

active_benchmark_run_directory.mkdir(
    parents=True,
    exist_ok=True,
)

embedding_benchmark_manifest_path = (
    active_benchmark_run_directory
    / "benchmark_manifest.json"
)

embedding_benchmark_manifest = {
    "active_golden_dataset": {
        "path": str(golden_dataset_context["path"]),
        "sha256": golden_dataset_context["sha256"],
        "record_count": golden_dataset_context["record_count"],
    },
    "chunking_candidates": {
        candidate_name: {
            "path": retrieval_chunking_candidates[candidate_name]["path"],
            "records": candidate_chunk_count_by_name[candidate_name],
        }
        for candidate_name in retrieval_chunking_candidates
    },
    "embedding_models": {
        model_name: {
            key: value
            for key, value in model_config.items()
            if key != "provider" or value != "arvan"
        }
        for model_name, model_config in embedding_model_registry.items()
    },
    "workload": {
        "query_records": len(benchmark_query_records),
        "unique_query_texts": len(unique_query_texts),
        "cross_candidate_unique_chunk_texts": len(
            all_canonical_chunk_texts
        ),
    },
}

if embedding_benchmark_manifest_path.is_file():
    existing_manifest = json.loads(
        embedding_benchmark_manifest_path.read_text(
            encoding="utf-8"
        )
    )

    assert (
        existing_manifest["active_golden_dataset"]["sha256"]
        == golden_dataset_context["sha256"]
    ), "Existing benchmark manifest belongs to a different active dataset."

else:
    embedding_benchmark_manifest_path.write_text(
        json.dumps(
            embedding_benchmark_manifest,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

print("=== EMBEDDING BENCHMARK CACHE INITIALIZATION ===")
print("Run directory:", active_benchmark_run_directory)
print("Manifest:", embedding_benchmark_manifest_path)
print("Active dataset records:", golden_dataset_context["record_count"])
print("Embedding models registered:", list(embedding_model_registry))
print("\nEMBEDDING BENCHMARK CACHE INITIALIZATION: PASS")

=== EMBEDDING BENCHMARK CACHE INITIALIZATION ===
Run directory: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2
Manifest: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/benchmark_manifest.json
Active dataset records: 3813
Embedding models registered: ['multilingual_e5_small', 'all_minilm_l6_v2', 'arvan_text_embedding_3_large']

EMBEDDING BENCHMARK CACHE INITIALIZATION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۰. ساخت کامل embeddingهای اروان با قابلیت ادامه

مدل اروان برای Benchmark انتخاب و اتصال آن تأیید شده است.

در این مرحله همهٔ متن‌های یکتای مورد نیاز Benchmark به‌صورت batch به اروان ارسال می‌شوند.  
هر batch بلافاصله ذخیره می‌شود؛ بنابراین در صورت قطع اجرا، ادامهٔ کار از batchهای باقی‌مانده انجام خواهد شد.

</div>

In [44]:
# Code Cell 42

arvan_embedding_cache_directory = (
    active_benchmark_run_directory
    / "arvan_text_embedding_3_large"
    / "batches"
)

arvan_embedding_cache_directory.mkdir(
    parents=True,
    exist_ok=True,
)

arvan_input_text_by_hash = {}

for query_record in benchmark_query_records:
    text = query_record["question"]
    text_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
    arvan_input_text_by_hash[text_hash] = text

for chunk_text in all_canonical_chunk_texts:
    text_hash = hashlib.sha256(chunk_text.encode("utf-8")).hexdigest()
    arvan_input_text_by_hash[text_hash] = chunk_text

assert len(arvan_input_text_by_hash) == estimated_remote_inputs, (
    "The Arvan input registry has an unexpected unique-text count."
)


# =========================================================
# LOAD AND VALIDATE EXISTING CHECKPOINTS
# =========================================================

completed_arvan_input_hashes = set()

for batch_path in sorted(
    arvan_embedding_cache_directory.glob("batch_*.npz")
):
    with np.load(batch_path, allow_pickle=False) as batch_data:
        batch_hashes = batch_data["input_hashes"]
        batch_embeddings = batch_data["embeddings"]

        assert batch_embeddings.ndim == 2, (
            f"Invalid embedding matrix in {batch_path.name}."
        )
        assert batch_embeddings.shape[0] == len(batch_hashes), (
            f"Input and embedding counts differ in {batch_path.name}."
        )
        assert batch_embeddings.shape[1] == arvan_test_embedding.size, (
            f"Embedding dimension differs in {batch_path.name}."
        )
        assert not np.isnan(batch_embeddings).any(), (
            f"Cached embedding contains NaN in {batch_path.name}."
        )

        batch_hash_set = {
            str(input_hash)
            for input_hash in batch_hashes
        }

        unknown_hashes = (
            batch_hash_set
            - set(arvan_input_text_by_hash)
        )

        assert not unknown_hashes, (
            f"{batch_path.name} contains hashes that do not belong "
            "to the active Golden v2 benchmark input registry."
        )

        completed_arvan_input_hashes.update(
            batch_hash_set
        )


# =========================================================
# BUILD ONLY THE MISSING WORK
# =========================================================

pending_arvan_input_hashes = sorted(
    set(arvan_input_text_by_hash)
    - completed_arvan_input_hashes
)

max_batch_inputs = 64
max_batch_characters = 100_000

arvan_batches_to_run = []
current_batch_hashes = []
current_batch_character_count = 0

for input_hash in pending_arvan_input_hashes:
    input_text = arvan_input_text_by_hash[input_hash]

    exceeds_input_limit = (
        len(current_batch_hashes)
        >= max_batch_inputs
    )

    exceeds_character_limit = (
        current_batch_hashes
        and current_batch_character_count
        + len(input_text)
        > max_batch_characters
    )

    if (
        exceeds_input_limit
        or exceeds_character_limit
    ):
        arvan_batches_to_run.append(
            current_batch_hashes
        )
        current_batch_hashes = []
        current_batch_character_count = 0

    current_batch_hashes.append(
        input_hash
    )

    current_batch_character_count += len(
        input_text
    )

if current_batch_hashes:
    arvan_batches_to_run.append(
        current_batch_hashes
    )


# =========================================================
# EXECUTION CONFIGURATION
# =========================================================

max_attempts_per_batch = 8

base_retry_delay_seconds = 2.0
max_retry_delay_seconds = 60.0

inter_batch_delay_seconds = 0.1


print(
    "=== ARVAN FULL EMBEDDING EXECUTION ==="
)

print(
    "Unique inputs required:",
    len(arvan_input_text_by_hash),
)

print(
    "Inputs already cached:",
    len(completed_arvan_input_hashes),
)

print(
    "Inputs pending:",
    len(pending_arvan_input_hashes),
)

print(
    "Batches pending:",
    len(arvan_batches_to_run),
)

print(
    "Maximum inputs per batch:",
    max_batch_inputs,
)

print(
    "Maximum characters per batch:",
    max_batch_characters,
)

print(
    "Maximum attempts per batch:",
    max_attempts_per_batch,
)

print(
    "Checkpoint directory:",
    arvan_embedding_cache_directory,
)


# =========================================================
# RUN PENDING BATCHES WITH RETRY + CHECKPOINTING
# =========================================================

for batch_number, batch_hashes in enumerate(
    arvan_batches_to_run,
    start=1,
):
    batch_texts = [
        arvan_input_text_by_hash[input_hash]
        for input_hash in batch_hashes
    ]

    response = None
    last_error = None

    for attempt_number in range(
        1,
        max_attempts_per_batch + 1,
    ):
        try:
            response = (
                arvan_embedding_client
                .embeddings
                .create(
                    model=arvan_embedding_model_id,
                    input=batch_texts,
                )
            )

            last_error = None
            break

        except Exception as error:
            last_error = error

            if (
                attempt_number
                >= max_attempts_per_batch
            ):
                break

            retry_delay_seconds = min(
                base_retry_delay_seconds
                * (2 ** (attempt_number - 1)),
                max_retry_delay_seconds,
            )

            print(
                f"Batch {batch_number}/"
                f"{len(arvan_batches_to_run)} "
                f"attempt {attempt_number}/"
                f"{max_attempts_per_batch} failed: "
                f"{type(error).__name__}: {error}"
            )

            print(
                f"Retrying same batch in "
                f"{retry_delay_seconds:.1f}s..."
            )

            time.sleep(
                retry_delay_seconds
            )

    if response is None:
        raise RuntimeError(
            "Arvan embedding batch failed after retries. "
            "All previously completed checkpoints remain available at "
            f"{arvan_embedding_cache_directory}. "
            f"Failed pending batch: "
            f"{batch_number}/{len(arvan_batches_to_run)}. "
            f"Attempts: {max_attempts_per_batch}. "
            f"Inputs currently cached: "
            f"{len(completed_arvan_input_hashes)}/"
            f"{len(arvan_input_text_by_hash)}. "
            f"Last error: "
            f"{type(last_error).__name__}: {last_error}"
        ) from last_error

    response_items = sorted(
        response.data,
        key=lambda item: item.index,
    )

    assert len(response_items) == len(
        batch_hashes
    ), (
        f"Arvan returned {len(response_items)} embeddings "
        f"for {len(batch_hashes)} inputs "
        f"in pending batch {batch_number}."
    )

    batch_embeddings = np.asarray(
        [
            response_item.embedding
            for response_item
            in response_items
        ],
        dtype=np.float32,
    )

    assert batch_embeddings.shape == (
        len(batch_hashes),
        arvan_test_embedding.size,
    ), (
        f"Unexpected Arvan response shape "
        f"for pending batch {batch_number}: "
        f"{batch_embeddings.shape}"
    )

    assert not np.isnan(
        batch_embeddings
    ).any(), (
        f"Arvan response contains NaN "
        f"in pending batch {batch_number}."
    )


    # -----------------------------------------------------
    # CHECKPOINT ONLY AFTER A COMPLETE VALID RESPONSE
    # -----------------------------------------------------

    batch_output_path = (
        arvan_embedding_cache_directory
        / (
            "batch_"
            f"{len(completed_arvan_input_hashes):06d}"
            ".npz"
        )
    )

    assert not batch_output_path.exists(), (
        "Refusing to overwrite an existing embedding checkpoint: "
        f"{batch_output_path}"
    )

    np.savez_compressed(
        batch_output_path,
        input_hashes=np.asarray(
            batch_hashes
        ),
        embeddings=batch_embeddings,
    )

    completed_arvan_input_hashes.update(
        batch_hashes
    )

    print(
        f"Batch {batch_number}/"
        f"{len(arvan_batches_to_run)} saved | "
        f"inputs cached: "
        f"{len(completed_arvan_input_hashes)}/"
        f"{len(arvan_input_text_by_hash)}"
    )

    time.sleep(
        inter_batch_delay_seconds
    )


# =========================================================
# FINAL CACHE COMPLETENESS CHECK
# =========================================================

expected_arvan_input_hashes = set(
    arvan_input_text_by_hash
)

assert completed_arvan_input_hashes == (
    expected_arvan_input_hashes
), (
    "Arvan embedding cache is incomplete after execution. "
    f"Expected {len(expected_arvan_input_hashes)}, "
    f"cached {len(completed_arvan_input_hashes)}."
)


# =========================================================
# FINAL REPORT
# =========================================================

arvan_embedding_execution_report = {
    "model_id": arvan_embedding_model_id,
    "embedding_dimension": int(
        arvan_test_embedding.size
    ),
    "unique_inputs_required": len(
        arvan_input_text_by_hash
    ),
    "unique_inputs_cached": len(
        completed_arvan_input_hashes
    ),
    "batch_directory": str(
        arvan_embedding_cache_directory
    ),
    "max_attempts_per_batch": (
        max_attempts_per_batch
    ),
    "status": "PASS",
}

arvan_embedding_execution_report_path = (
    active_benchmark_run_directory
    / "arvan_embedding_execution_report.json"
)

arvan_embedding_execution_report_path.write_text(
    json.dumps(
        arvan_embedding_execution_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print(
    "ARVAN FULL EMBEDDING EXECUTION: PASS"
)

print(
    "Total inputs cached:",
    len(completed_arvan_input_hashes),
)

print(
    "Report:",
    arvan_embedding_execution_report_path,
)

=== ARVAN FULL EMBEDDING EXECUTION ===
Unique inputs required: 11433
Inputs already cached: 4544
Inputs pending: 6889
Batches pending: 108
Maximum inputs per batch: 64
Maximum characters per batch: 100000
Maximum attempts per batch: 8
Checkpoint directory: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/arvan_text_embedding_3_large/batches
Batch 1/108 saved | inputs cached: 4608/11433
Batch 2/108 saved | inputs cached: 4672/11433
Batch 3/108 saved | inputs cached: 4736/11433
Batch 4/108 saved | inputs cached: 4800/11433
Batch 5/108 saved | inputs cached: 4864/11433
Batch 6/108 saved | inputs cached: 4928/11433
Batch 7/108 saved | inputs cached: 4992/11433
Batch 8/108 saved | inputs cached: 5056/11433
Batch 9/108 saved | inputs cached: 5120/11433
Batch 10/108 saved | inputs cached: 5184/11433
Batch 11/108 saved | inputs cached: 5248/11433
Batch 12/108 saved | i

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۱. ساخت کامل embeddingهای مدل‌های محلی

Embeddingهای اروان برای همهٔ ورودی‌های Benchmark ساخته و ذخیره شدند.

در این مرحله همان داده‌ها با دو مدل محلی اجرا می‌شوند. برای مدل E5، پیشوندهای استاندارد query و passage به‌صورت جداگانه اعمال می‌شوند.

</div>

In [ ]:
# Code Cell 43

local_embedding_input_registry = {}

query_text_by_hash = {
    hashlib.sha256(
        query_record["question"].encode("utf-8")
    ).hexdigest(): query_record["question"]
    for query_record in benchmark_query_records
}

chunk_text_by_hash = {
    hashlib.sha256(
        chunk_text.encode("utf-8")
    ).hexdigest(): chunk_text
    for chunk_text in all_canonical_chunk_texts
}

local_embedding_execution_reports = {}

for model_name, model in embedding_models.items():
    model_config = embedding_model_registry[model_name]

    if model_config["query_prefix"] or model_config["passage_prefix"]:
        input_text_by_key = {
            **{
                f"query:{text_hash}": (
                    model_config["query_prefix"] + text
                )
                for text_hash, text in query_text_by_hash.items()
            },
            **{
                f"passage:{text_hash}": (
                    model_config["passage_prefix"] + text
                )
                for text_hash, text in chunk_text_by_hash.items()
            },
        }
    else:
        input_text_by_key = {
            f"text:{text_hash}": text
            for text_hash, text in {
                **query_text_by_hash,
                **chunk_text_by_hash,
            }.items()
        }

    local_embedding_input_registry[model_name] = input_text_by_key

    model_cache_directory = (
        active_benchmark_run_directory
        / model_name
        / "batches"
    )

    model_cache_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    completed_input_keys = set()

    for batch_path in sorted(model_cache_directory.glob("batch_*.npz")):
        with np.load(batch_path, allow_pickle=False) as batch_data:
            batch_input_keys = batch_data["input_keys"]
            batch_embeddings = batch_data["embeddings"]

            assert batch_embeddings.ndim == 2, (
                f"Invalid embedding matrix in {batch_path.name}."
            )
            assert batch_embeddings.shape[0] == len(batch_input_keys), (
                f"Input and embedding counts differ in {batch_path.name}."
            )

            completed_input_keys.update(
                str(input_key)
                for input_key in batch_input_keys
            )

    pending_input_keys = sorted(
        set(input_text_by_key) - completed_input_keys
    )

    batch_size = 128
    input_key_batches = [
        pending_input_keys[index:index + batch_size]
        for index in range(0, len(pending_input_keys), batch_size)
    ]

    print(f"\n=== LOCAL FULL EMBEDDING: {model_name} ===")
    print("Inputs required:", len(input_text_by_key))
    print("Inputs already cached:", len(completed_input_keys))
    print("Inputs pending:", len(pending_input_keys))
    print("Batches pending:", len(input_key_batches))

    for batch_number, input_key_batch in enumerate(
        input_key_batches,
        start=1,
    ):
        input_text_batch = [
            input_text_by_key[input_key]
            for input_key in input_key_batch
        ]

        batch_embeddings = model.encode(
            input_text_batch,
            batch_size=64,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)

        assert batch_embeddings.shape[0] == len(input_key_batch), (
            f"Unexpected result count for {model_name}, "
            f"batch {batch_number}."
        )
        assert not np.isnan(batch_embeddings).any(), (
            f"{model_name} returned NaN in batch {batch_number}."
        )

        batch_output_path = (
            model_cache_directory
            / f"batch_{len(completed_input_keys):06d}.npz"
        )

        np.savez_compressed(
            batch_output_path,
            input_keys=np.asarray(input_key_batch),
            embeddings=batch_embeddings,
        )

        completed_input_keys.update(input_key_batch)

        if (
            batch_number == 1
            or batch_number == len(input_key_batches)
            or batch_number % 10 == 0
        ):
            print(
                f"Batch {batch_number}/{len(input_key_batches)} saved | "
                f"inputs cached: {len(completed_input_keys)}"
            )

    assert completed_input_keys == set(input_text_by_key), (
        f"{model_name} embedding cache is incomplete."
    )

    local_embedding_execution_reports[model_name] = {
        "model_id": model_config["model_id"],
        "embedding_dimension": model.get_sentence_embedding_dimension(),
        "unique_inputs_required": len(input_text_by_key),
        "unique_inputs_cached": len(completed_input_keys),
        "batch_directory": str(model_cache_directory),
        "status": "PASS",
    }

    report_path = (
        active_benchmark_run_directory
        / f"{model_name}_embedding_execution_report.json"
    )

    report_path.write_text(
        json.dumps(
            local_embedding_execution_reports[model_name],
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

print("\nLOCAL FULL EMBEDDING EXECUTION: PASS")

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۲. بارگذاری و اعتبارسنجی cacheهای embedding

Embeddingهای اروان و دو مدل محلی با موفقیت ساخته شدند.

در این مرحله cacheهای هر مدل بارگذاری می‌شوند و بررسی می‌شود که برای همهٔ پرسش‌های Golden و همهٔ متن‌های چانک، بردار معتبر وجود دارد.

</div>

In [ ]:
# Code Cell 44

def load_embedding_batches(
    batch_directory,
    key_field_name,
):
    vector_by_key = {}

    for batch_path in sorted(batch_directory.glob("batch_*.npz")):
        with np.load(batch_path, allow_pickle=False) as batch_data:
            batch_keys = batch_data[key_field_name]
            batch_embeddings = batch_data["embeddings"]

            assert batch_embeddings.shape[0] == len(batch_keys), (
                f"Invalid cache batch: {batch_path.name}"
            )

            for key, embedding in zip(batch_keys, batch_embeddings):
                key = str(key)

                assert key not in vector_by_key, (
                    f"Duplicate embedding cache key: {key}"
                )

                vector_by_key[key] = embedding.astype(
                    np.float32,
                    copy=False,
                )

    return vector_by_key


arvan_vector_by_text_hash = load_embedding_batches(
    arvan_embedding_cache_directory,
    key_field_name="input_hashes",
)

expected_arvan_hashes = (
    set(query_text_by_hash)
    | set(chunk_text_by_hash)
)

assert set(arvan_vector_by_text_hash) == expected_arvan_hashes, (
    "Arvan cache does not exactly match the current benchmark input registry."
)

benchmark_embeddings_by_model = {
    "arvan_text_embedding_3_large": {
        "query_vectors_by_text_hash": {
            text_hash: arvan_vector_by_text_hash[text_hash]
            for text_hash in query_text_by_hash
        },
        "passage_vectors_by_text_hash": {
            text_hash: arvan_vector_by_text_hash[text_hash]
            for text_hash in chunk_text_by_hash
        },
        "dimension": int(arvan_test_embedding.size),
    },
}

for model_name, input_text_by_key in local_embedding_input_registry.items():
    model_cache_directory = (
        active_benchmark_run_directory
        / model_name
        / "batches"
    )

    vector_by_input_key = load_embedding_batches(
        model_cache_directory,
        key_field_name="input_keys",
    )

    model_config = embedding_model_registry[model_name]

    if model_config["query_prefix"] or model_config["passage_prefix"]:
        query_vectors_by_text_hash = {
            text_hash: vector_by_input_key[f"query:{text_hash}"]
            for text_hash in query_text_by_hash
        }
        passage_vectors_by_text_hash = {
            text_hash: vector_by_input_key[f"passage:{text_hash}"]
            for text_hash in chunk_text_by_hash
        }
    else:
        query_vectors_by_text_hash = {
            text_hash: vector_by_input_key[f"text:{text_hash}"]
            for text_hash in query_text_by_hash
        }
        passage_vectors_by_text_hash = {
            text_hash: vector_by_input_key[f"text:{text_hash}"]
            for text_hash in chunk_text_by_hash
        }

    benchmark_embeddings_by_model[model_name] = {
        "query_vectors_by_text_hash": query_vectors_by_text_hash,
        "passage_vectors_by_text_hash": passage_vectors_by_text_hash,
        "dimension": int(
            next(iter(query_vectors_by_text_hash.values())).shape[0]
        ),
    }

print("=== EMBEDDING CACHE ASSEMBLY VALIDATION ===")

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_vectors = embedding_lookup["query_vectors_by_text_hash"]
    passage_vectors = embedding_lookup["passage_vectors_by_text_hash"]

    assert set(query_vectors) == set(query_text_by_hash), (
        f"{model_name} query cache is incomplete."
    )
    assert set(passage_vectors) == set(chunk_text_by_hash), (
        f"{model_name} passage cache is incomplete."
    )

    print(f"\n{model_name}")
    print("- Embedding dimension:", embedding_lookup["dimension"])
    print("- Query vectors:", len(query_vectors))
    print("- Passage vectors:", len(passage_vectors))

print("\nEMBEDDING CACHE ASSEMBLY VALIDATION: PASS")

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۳. Benchmark واقعی Retrieval

برای هر مدل و هر روش چانکینگ، similarity میان پرسش‌های Golden و چانک‌ها محاسبه می‌شود.

معیار اصلی مقایسه فقط روی seedهای مشترک و دارای label معتبر اجرا می‌شود.  
بنابراین تفاوت تعداد چانک‌های مثبت یا پوشش اضافی یک روش، مقایسهٔ اصلی را ناعادلانه نمی‌کند.

</div>

In [ ]:
# Code Cell 45

evaluation_cutoffs = (1, 3, 5, 10)

common_eligible_seed_ids = sorted(
    set.intersection(
        *(
            {
                source_seed_id
                for source_seed_id, label in seed_labels.items()
                if label["eligible"]
            }
            for seed_labels in retrieval_evaluation_registry.values()
        )
    )
)

assert common_eligible_seed_ids, (
    "No Golden v2 evaluation identities are eligible across all chunk candidates."
)

assert set(common_eligible_seed_ids).issubset(set(active_seed_ids)), (
    "Common eligible identities must be a subset of active Golden v2 identities."
)

evaluation_query_records = [
    query_record
    for query_record in benchmark_query_records
    if query_record["source_seed_id"] in common_eligible_seed_ids
]

assert evaluation_query_records, "No eligible Golden queries were found."

def l2_normalize_rows(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    assert np.all(norms > 0), "Cannot normalize zero-length embeddings."
    return matrix / norms


candidate_chunk_registry = {}

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])

    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    chunk_ids = []
    chunk_text_hashes = []

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)
            chunk_ids.append(chunk["chunk_id"])
            chunk_text_hashes.append(
                hashlib.sha256(
                    canonical_exact_text(chunk["content"]).encode("utf-8")
                ).hexdigest()
            )

    chunk_index_by_id = {
        chunk_id: index
        for index, chunk_id in enumerate(chunk_ids)
    }

    positive_chunk_indexes_by_seed = {}

    for source_seed_id in common_eligible_seed_ids:
        positive_chunk_ids = retrieval_evaluation_registry[
            candidate_name
        ][source_seed_id]["positive_chunk_ids"]

        positive_indexes = sorted(
            chunk_index_by_id[chunk_id]
            for chunk_id in positive_chunk_ids
            if chunk_id in chunk_index_by_id
        )

        assert positive_indexes, (
            f"{candidate_name} has no indexed positive chunk for "
            f"seed {source_seed_id}."
        )

        positive_chunk_indexes_by_seed[source_seed_id] = set(
            positive_indexes
        )

    candidate_chunk_registry[candidate_name] = {
        "chunk_ids": chunk_ids,
        "chunk_text_hashes": chunk_text_hashes,
        "positive_chunk_indexes_by_seed": positive_chunk_indexes_by_seed,
    }

print("=== FULL RETRIEVAL BENCHMARK ===")
print("Common eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(evaluation_query_records))
print("Evaluation cutoffs:", evaluation_cutoffs)

embedding_benchmark_results = {}

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_matrix = np.vstack(
        [
            embedding_lookup["query_vectors_by_text_hash"][
                hashlib.sha256(
                    query_record["question"].encode("utf-8")
                ).hexdigest()
            ]
            for query_record in evaluation_query_records
        ]
    ).astype(np.float32)

    query_matrix = l2_normalize_rows(query_matrix)

    embedding_benchmark_results[model_name] = {}

    for candidate_name, candidate_data in candidate_chunk_registry.items():
        passage_matrix = np.vstack(
            [
                embedding_lookup["passage_vectors_by_text_hash"][
                    text_hash
                ]
                for text_hash in candidate_data["chunk_text_hashes"]
            ]
        ).astype(np.float32)

        passage_matrix = l2_normalize_rows(passage_matrix)

        first_positive_ranks = []
        retrieval_batch_size = 16

        started_at = time.perf_counter()

        for batch_start in range(
            0,
            len(evaluation_query_records),
            retrieval_batch_size,
        ):
            batch_end = min(
                batch_start + retrieval_batch_size,
                len(evaluation_query_records),
            )

            similarity_matrix = (
                query_matrix[batch_start:batch_end]
                @ passage_matrix.T
            )

            for local_index, query_record in enumerate(
                evaluation_query_records[batch_start:batch_end]
            ):
                ranked_chunk_indexes = np.argsort(
                    similarity_matrix[local_index]
                )[::-1]

                positive_chunk_indexes = candidate_data[
                    "positive_chunk_indexes_by_seed"
                ][query_record["source_seed_id"]]

                first_positive_rank = next(
                    rank
                    for rank, chunk_index in enumerate(
                        ranked_chunk_indexes,
                        start=1,
                    )
                    if chunk_index in positive_chunk_indexes
                )

                first_positive_ranks.append(first_positive_rank)

        elapsed_seconds = time.perf_counter() - started_at

        first_positive_ranks = np.asarray(
            first_positive_ranks,
            dtype=np.int32,
        )

        metrics = {
            f"recall_at_{cutoff}": float(
                np.mean(first_positive_ranks <= cutoff)
            )
            for cutoff in evaluation_cutoffs
        }

        metrics.update(
            {
                "mrr": float(np.mean(1.0 / first_positive_ranks)),
                "mean_first_positive_rank": float(
                    np.mean(first_positive_ranks)
                ),
                "median_first_positive_rank": float(
                    np.median(first_positive_ranks)
                ),
                "query_count": int(len(first_positive_ranks)),
                "candidate_chunk_count": int(
                    len(candidate_data["chunk_ids"])
                ),
                "elapsed_seconds": round(elapsed_seconds, 3),
            }
        )

        embedding_benchmark_results[model_name][candidate_name] = metrics

        print(f"\n{model_name} × {candidate_name}")
        print("- Recall@1:", f"{metrics['recall_at_1']:.4f}")
        print("- Recall@3:", f"{metrics['recall_at_3']:.4f}")
        print("- Recall@5:", f"{metrics['recall_at_5']:.4f}")
        print("- Recall@10:", f"{metrics['recall_at_10']:.4f}")
        print("- MRR:", f"{metrics['mrr']:.4f}")
        print(
            "- Mean first-positive rank:",
            f"{metrics['mean_first_positive_rank']:.2f}",
        )
        print("- Seconds:", metrics["elapsed_seconds"])

print("\nFULL RETRIEVAL BENCHMARK: PASS")

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۴. آزمون تشخیصی با متن شاهد Golden

نتیجهٔ retrieval روی سؤال‌های Golden برای انتخاب نهایی مدل کافی نیست.

در این آزمون، به‌جای سؤال تولیدشده، خود متن شاهد Golden به‌عنوان query استفاده می‌شود.  
هدف تشخیص این است که افت معیارها از نوع سؤال‌ها ناشی می‌شود یا از مدل embedding، چانکینگ و labelهای retrieval.

</div>

In [ ]:
# Code Cell 46

gold_evidence_text_by_seed = {}

for source_seed_id in common_eligible_seed_ids:
    observed_evidence_texts = {
        canonical_exact_text(record["source_evidence"])
        for record in gold_records_by_seed[source_seed_id]
    }

    assert len(observed_evidence_texts) == 1, (
        f"Seed {source_seed_id} has inconsistent Golden evidence text."
    )

    gold_evidence_text_by_seed[source_seed_id] = next(
        iter(observed_evidence_texts)
    )

evidence_probe_arvan_path = (
    active_benchmark_run_directory
    / "evidence_probe_arvan.npz"
)

if evidence_probe_arvan_path.is_file():
    with np.load(evidence_probe_arvan_path, allow_pickle=False) as probe_data:
        evidence_probe_seed_ids = probe_data["source_seed_ids"].astype(int)
        arvan_evidence_probe_matrix = probe_data["embeddings"].astype(
            np.float32
        )
else:
    evidence_probe_seed_ids = np.asarray(
        common_eligible_seed_ids,
        dtype=np.int32,
    )

    arvan_evidence_probe_response = arvan_embedding_client.embeddings.create(
        model=arvan_embedding_model_id,
        input=[
            gold_evidence_text_by_seed[source_seed_id]
            for source_seed_id in evidence_probe_seed_ids
        ],
    )

    arvan_evidence_probe_matrix = np.asarray(
        [
            item.embedding
            for item in sorted(
                arvan_evidence_probe_response.data,
                key=lambda item: item.index,
            )
        ],
        dtype=np.float32,
    )

    np.savez_compressed(
        evidence_probe_arvan_path,
        source_seed_ids=evidence_probe_seed_ids,
        embeddings=arvan_evidence_probe_matrix,
    )

assert evidence_probe_seed_ids.tolist() == common_eligible_seed_ids, (
    "Evidence-probe seed ordering is inconsistent."
)

evidence_probe_query_matrices = {
    "arvan_text_embedding_3_large": l2_normalize_rows(
        arvan_evidence_probe_matrix
    ),
}

for model_name, model in embedding_models.items():
    model_config = embedding_model_registry[model_name]

    local_probe_matrix = model.encode(
        [
            model_config["query_prefix"]
            + gold_evidence_text_by_seed[source_seed_id]
            for source_seed_id in common_eligible_seed_ids
        ],
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)

    evidence_probe_query_matrices[model_name] = l2_normalize_rows(
        local_probe_matrix
    )

evidence_probe_results = {}

print("=== GOLD EVIDENCE RETRIEVAL DIAGNOSTIC ===")
print("Diagnostic seeds:", len(common_eligible_seed_ids))

for model_name, probe_query_matrix in evidence_probe_query_matrices.items():
    evidence_probe_results[model_name] = {}

    for candidate_name, candidate_data in candidate_chunk_registry.items():
        embedding_lookup = benchmark_embeddings_by_model[model_name]

        passage_matrix = np.vstack(
            [
                embedding_lookup["passage_vectors_by_text_hash"][text_hash]
                for text_hash in candidate_data["chunk_text_hashes"]
            ]
        ).astype(np.float32)

        passage_matrix = l2_normalize_rows(passage_matrix)

        similarity_matrix = probe_query_matrix @ passage_matrix.T
        first_positive_ranks = []

        for row_index, source_seed_id in enumerate(
            common_eligible_seed_ids
        ):
            ranked_chunk_indexes = np.argsort(
                similarity_matrix[row_index]
            )[::-1]

            positive_chunk_indexes = candidate_data[
                "positive_chunk_indexes_by_seed"
            ][source_seed_id]

            first_positive_ranks.append(
                next(
                    rank
                    for rank, chunk_index in enumerate(
                        ranked_chunk_indexes,
                        start=1,
                    )
                    if chunk_index in positive_chunk_indexes
                )
            )

        first_positive_ranks = np.asarray(
            first_positive_ranks,
            dtype=np.int32,
        )

        metrics = {
            f"recall_at_{cutoff}": float(
                np.mean(first_positive_ranks <= cutoff)
            )
            for cutoff in evaluation_cutoffs
        }

        metrics["mrr"] = float(
            np.mean(1.0 / first_positive_ranks)
        )

        evidence_probe_results[model_name][candidate_name] = metrics

        print(f"\n{model_name} × {candidate_name}")
        print("- Recall@1:", f"{metrics['recall_at_1']:.4f}")
        print("- Recall@3:", f"{metrics['recall_at_3']:.4f}")
        print("- Recall@5:", f"{metrics['recall_at_5']:.4f}")
        print("- Recall@10:", f"{metrics['recall_at_10']:.4f}")
        print("- MRR:", f"{metrics['mrr']:.4f}")

print("\nGOLD EVIDENCE RETRIEVAL DIAGNOSTIC: PASS")

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۵. خواندن معیار فریز‌شدهٔ انتخاب نهایی

Benchmark نشان داد که نتیجهٔ سؤال‌های Golden و نتیجهٔ متن شاهد یکسان نیست.

پیش از هر انتخاب نهایی، معیارهای فریز‌شدهٔ پروژه برای انتخاب مدل و روش چانکینگ خوانده می‌شوند تا تصمیم بر اساس قرارداد پروژه گرفته شود، نه یک آستانهٔ دلخواه.

</div>

In [ ]:
# Code Cell 47

selection_requirements_artifact = chunking_freeze_manifest["artifacts"][
    "final_selection_requirements"
]

selection_requirements_path = Path(
    selection_requirements_artifact["path"]
)

if not selection_requirements_path.is_absolute():
    selection_requirements_path = project_root / selection_requirements_path

assert selection_requirements_path.is_file(), (
    "Frozen final-selection requirements artifact is missing: "
    f"{selection_requirements_path}"
)

selection_requirements = json.loads(
    selection_requirements_path.read_text(
        encoding="utf-8"
    )
)

print("=== FROZEN FINAL-SELECTION REQUIREMENTS ===")
print("Path:", selection_requirements_path.relative_to(project_root))
print(
    json.dumps(
        selection_requirements,
        ensure_ascii=False,
        indent=2,
    )
)

print("\nFROZEN FINAL-SELECTION REQUIREMENTS: PASS")

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۶. ارزیابی Citation و Attribution در خروجی Retrieval

در این مرحله خروجی‌های Top-10 هر مدل و روش چانکینگ بررسی می‌شوند.

دو موضوع سنجیده می‌شود:

- کامل‌بودن قرارداد citation در چانک‌های بازیابی‌شده؛
- انطباق provenance چانک‌های بازیابی‌شده با سندهای معتبر همان پرسش Golden.

این ارزیابی ساختاری و خودکار است؛ ارزیابی انسانی کیفیت پاسخ و context در مرحلهٔ بعد انجام می‌شود.

</div>

In [ ]:
# Code Cell 48

citation_evaluation_cutoff = 10

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])

    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    chunk_metadata_by_id = {}

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)

            chunk_metadata_by_id[chunk["chunk_id"]] = {
                "source_id": chunk.get("source_id"),
                "source_type": chunk.get("source_type"),
                "locator_start": chunk.get("locator_start"),
                "locator_end": chunk.get("locator_end"),
                "attribution_context": chunk.get(
                    "attribution_context"
                ),
            }

    candidate_chunk_registry[candidate_name][
        "chunk_metadata_by_id"
    ] = chunk_metadata_by_id

print("=== CITATION AND ATTRIBUTION EVALUATION ===")
print("Queries evaluated:", len(evaluation_query_records))
print("Top-K cutoff:", citation_evaluation_cutoff)

citation_attribution_evaluation_results = {}

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_matrix = np.vstack(
        [
            embedding_lookup["query_vectors_by_text_hash"][
                hashlib.sha256(
                    query_record["question"].encode("utf-8")
                ).hexdigest()
            ]
            for query_record in evaluation_query_records
        ]
    ).astype(np.float32)

    query_matrix = l2_normalize_rows(query_matrix)

    citation_attribution_evaluation_results[model_name] = {}

    for candidate_name, candidate_data in candidate_chunk_registry.items():
        passage_matrix = np.vstack(
            [
                embedding_lookup["passage_vectors_by_text_hash"][text_hash]
                for text_hash in candidate_data["chunk_text_hashes"]
            ]
        ).astype(np.float32)

        passage_matrix = l2_normalize_rows(passage_matrix)

        similarity_matrix = query_matrix @ passage_matrix.T

        top_k_contract_passes = []
        top_k_positive_precisions = []
        top_k_has_positive = []
        top_1_is_positive = []

        for row_index, query_record in enumerate(
            evaluation_query_records
        ):
            top_k_indexes_unsorted = np.argpartition(
                similarity_matrix[row_index],
                -citation_evaluation_cutoff,
            )[-citation_evaluation_cutoff:]

            top_k_indexes = top_k_indexes_unsorted[
                np.argsort(
                    similarity_matrix[row_index][top_k_indexes_unsorted]
                )[::-1]
            ]

            top_k_chunk_ids = [
                candidate_data["chunk_ids"][chunk_index]
                for chunk_index in top_k_indexes
            ]

            positive_chunk_ids = retrieval_evaluation_registry[
                candidate_name
            ][query_record["source_seed_id"]][
                "positive_chunk_ids"
            ]

            top_k_positive_flags = [
                chunk_id in positive_chunk_ids
                for chunk_id in top_k_chunk_ids
            ]

            top_k_metadata = [
                candidate_data["chunk_metadata_by_id"][chunk_id]
                for chunk_id in top_k_chunk_ids
            ]

            top_k_contract_passes.append(
                all(
                    isinstance(metadata["source_id"], str)
                    and metadata["source_id"]
                    and isinstance(metadata["source_type"], str)
                    and metadata["source_type"]
                    and isinstance(metadata["locator_start"], dict)
                    and isinstance(metadata["locator_end"], dict)
                    and isinstance(
                        metadata["attribution_context"],
                        dict,
                    )
                    for metadata in top_k_metadata
                )
            )

            top_k_positive_precisions.append(
                np.mean(top_k_positive_flags)
            )
            top_k_has_positive.append(
                any(top_k_positive_flags)
            )
            top_1_is_positive.append(
                top_k_positive_flags[0]
            )

        metrics = {
            "citation_contract_pass_rate_at_10": float(
                np.mean(top_k_contract_passes)
            ),
            "attribution_correct_at_1": float(
                np.mean(top_1_is_positive)
            ),
            "attribution_correct_at_10": float(
                np.mean(top_k_has_positive)
            ),
            "mean_provenance_precision_at_10": float(
                np.mean(top_k_positive_precisions)
            ),
            "query_count": len(evaluation_query_records),
        }

        citation_attribution_evaluation_results[model_name][
            candidate_name
        ] = metrics

        print(f"\n{model_name} × {candidate_name}")
        print(
            "- Citation contract pass@10:",
            f"{metrics['citation_contract_pass_rate_at_10']:.4f}",
        )
        print(
            "- Attribution correct@1:",
            f"{metrics['attribution_correct_at_1']:.4f}",
        )
        print(
            "- Attribution correct@10:",
            f"{metrics['attribution_correct_at_10']:.4f}",
        )
        print(
            "- Mean provenance precision@10:",
            f"{metrics['mean_provenance_precision_at_10']:.4f}",
        )

print("\nCITATION AND ATTRIBUTION EVALUATION: PASS")

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۷. پروفایل خودکار کیفیت Context

قرارداد citation در همهٔ خروجی‌ها سالم است.

در این مرحله contextهای Top-10 از نظر ساختاری بررسی می‌شوند: حجم متن، تعداد token، تنوع سندهای منبع و میزان تکرار چانک‌ها.  
این مرحله کیفیت ساختاری context را می‌سنجد؛ داوری انسانیِ مفیدبودن پاسخ در مراحل بعد انجام می‌شود.

</div>

In [ ]:
# Code Cell 49

context_quality_results = {}

print("=== AUTOMATED CONTEXT QUALITY PROFILE ===")
print("Queries evaluated:", len(evaluation_query_records))
print("Top-K cutoff:", citation_evaluation_cutoff)

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_matrix = np.vstack(
        [
            embedding_lookup["query_vectors_by_text_hash"][
                hashlib.sha256(
                    query_record["question"].encode("utf-8")
                ).hexdigest()
            ]
            for query_record in evaluation_query_records
        ]
    ).astype(np.float32)

    query_matrix = l2_normalize_rows(query_matrix)

    context_quality_results[model_name] = {}

    for candidate_name, candidate_data in candidate_chunk_registry.items():
        candidate_path = Path(
            retrieval_chunking_candidates[candidate_name]["path"]
        )

        if not candidate_path.is_absolute():
            candidate_path = project_root / candidate_path

        chunk_context_metadata_by_id = {}

        with candidate_path.open(encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue

                chunk = json.loads(line)

                chunk_context_metadata_by_id[chunk["chunk_id"]] = {
                    "source_id": chunk.get("source_id"),
                    "token_count": chunk.get("token_count"),
                    "char_count": chunk.get("char_count"),
                    "content_hash": hashlib.sha256(
                        canonical_exact_text(
                            chunk["content"]
                        ).encode("utf-8")
                    ).hexdigest(),
                }

        passage_matrix = np.vstack(
            [
                embedding_lookup["passage_vectors_by_text_hash"][text_hash]
                for text_hash in candidate_data["chunk_text_hashes"]
            ]
        ).astype(np.float32)

        passage_matrix = l2_normalize_rows(passage_matrix)
        similarity_matrix = query_matrix @ passage_matrix.T

        context_token_totals = []
        context_char_totals = []
        unique_source_counts = []
        unique_content_counts = []

        for row_index in range(len(evaluation_query_records)):
            top_k_indexes_unsorted = np.argpartition(
                similarity_matrix[row_index],
                -citation_evaluation_cutoff,
            )[-citation_evaluation_cutoff:]

            top_k_indexes = top_k_indexes_unsorted[
                np.argsort(
                    similarity_matrix[row_index][top_k_indexes_unsorted]
                )[::-1]
            ]

            top_k_chunk_ids = [
                candidate_data["chunk_ids"][chunk_index]
                for chunk_index in top_k_indexes
            ]

            top_k_metadata = [
                chunk_context_metadata_by_id[chunk_id]
                for chunk_id in top_k_chunk_ids
            ]

            context_token_totals.append(
                sum(
                    metadata["token_count"]
                    for metadata in top_k_metadata
                    if isinstance(metadata["token_count"], int)
                )
            )
            context_char_totals.append(
                sum(
                    metadata["char_count"]
                    for metadata in top_k_metadata
                    if isinstance(metadata["char_count"], int)
                )
            )
            unique_source_counts.append(
                len(
                    {
                        metadata["source_id"]
                        for metadata in top_k_metadata
                        if isinstance(metadata["source_id"], str)
                        and metadata["source_id"]
                    }
                )
            )
            unique_content_counts.append(
                len(
                    {
                        metadata["content_hash"]
                        for metadata in top_k_metadata
                    }
                )
            )

        metrics = {
            "mean_context_tokens_at_10": float(
                np.mean(context_token_totals)
            ),
            "p95_context_tokens_at_10": float(
                np.percentile(context_token_totals, 95)
            ),
            "mean_context_characters_at_10": float(
                np.mean(context_char_totals)
            ),
            "mean_unique_sources_at_10": float(
                np.mean(unique_source_counts)
            ),
            "mean_unique_chunks_at_10": float(
                np.mean(unique_content_counts)
            ),
        }

        context_quality_results[model_name][candidate_name] = metrics

        print(f"\n{model_name} × {candidate_name}")
        print(
            "- Mean context tokens@10:",
            f"{metrics['mean_context_tokens_at_10']:.1f}",
        )
        print(
            "- P95 context tokens@10:",
            f"{metrics['p95_context_tokens_at_10']:.1f}",
        )
        print(
            "- Mean context characters@10:",
            f"{metrics['mean_context_characters_at_10']:.1f}",
        )
        print(
            "- Mean unique sources@10:",
            f"{metrics['mean_unique_sources_at_10']:.2f}",
        )
        print(
            "- Mean unique chunks@10:",
            f"{metrics['mean_unique_chunks_at_10']:.2f}",
        )

print("\nAUTOMATED CONTEXT QUALITY PROFILE: PASS")

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۸. اعتبارسنجی اتصال Parent–Child

در روش Parent–Child، جست‌وجو روی child انجام می‌شود اما context نهایی باید از parent متناظر ساخته شود.

در این مرحله ارتباط هر child با parent آن بررسی می‌شود تا ارزیابی context این روش بر اساس ساختار واقعی آن انجام شود.

</div>

In [ ]:
# Code Cell 50

parent_candidate = chunking_candidates["parent_child_parents"]
parent_candidate_path = Path(parent_candidate["path"])

if not parent_candidate_path.is_absolute():
    parent_candidate_path = project_root / parent_candidate_path

child_candidate = retrieval_chunking_candidates[
    "parent_child_children"
]
child_candidate_path = Path(child_candidate["path"])

if not child_candidate_path.is_absolute():
    child_candidate_path = project_root / child_candidate_path

parent_chunks_by_id = {}

with parent_candidate_path.open(encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        parent_chunk = json.loads(line)
        parent_chunks_by_id[parent_chunk["chunk_id"]] = parent_chunk

child_to_parent_ids = {}
children_without_parent_id = []
children_with_missing_parent = []

with child_candidate_path.open(encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        child_chunk = json.loads(line)
        child_chunk_id = child_chunk["chunk_id"]
        parent_chunk_id = child_chunk.get("parent_chunk_id")

        if not isinstance(parent_chunk_id, str) or not parent_chunk_id:
            children_without_parent_id.append(child_chunk_id)
            continue

        child_to_parent_ids[child_chunk_id] = parent_chunk_id

        if parent_chunk_id not in parent_chunks_by_id:
            children_with_missing_parent.append(
                {
                    "child_chunk_id": child_chunk_id,
                    "parent_chunk_id": parent_chunk_id,
                }
            )

print("=== PARENT–CHILD CONTEXT LINKAGE VALIDATION ===")
print("Parent chunks:", len(parent_chunks_by_id))
print("Child chunks:", len(child_to_parent_ids))
print("Children without parent_chunk_id:", len(children_without_parent_id))
print("Children with missing parent record:", len(children_with_missing_parent))

assert not children_without_parent_id, (
    "Some parent-child retrieval chunks do not declare a parent_chunk_id."
)
assert not children_with_missing_parent, (
    "Some parent-child retrieval chunks point to missing parent records."
)

sample_child_chunk_id = next(iter(child_to_parent_ids))
sample_parent_chunk_id = child_to_parent_ids[sample_child_chunk_id]
sample_parent_chunk = parent_chunks_by_id[sample_parent_chunk_id]

print("\nSample linkage:")
print("- Child chunk ID:", sample_child_chunk_id)
print("- Parent chunk ID:", sample_parent_chunk_id)
print("- Parent token count:", sample_parent_chunk["token_count"])
print("- Parent source ID:", sample_parent_chunk["source_id"])
print(
    "- Parent locator start:",
    sample_parent_chunk["locator_start"],
)
print(
    "- Parent locator end:",
    sample_parent_chunk["locator_end"],
)

print("\nPARENT–CHILD CONTEXT LINKAGE VALIDATION: PASS")

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۹. ارزیابی Context واقعی Parent–Child

در روش Parent–Child، childها برای retrieval استفاده می‌شوند و parentهای متناظر context نهایی را می‌سازند.

در این مرحله حجم، تنوع و قرارداد citation context والدها برای هر مدل اندازه‌گیری می‌شود.

</div>

In [ ]:
# Code Cell 51

parent_child_context_quality_results = {}

parent_child_candidate_data = candidate_chunk_registry[
    "parent_child_children"
]

print("=== PARENT–CHILD REAL CONTEXT QUALITY ===")
print("Queries evaluated:", len(evaluation_query_records))
print("Retrieved children per query:", citation_evaluation_cutoff)

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_matrix = np.vstack(
        [
            embedding_lookup["query_vectors_by_text_hash"][
                hashlib.sha256(
                    query_record["question"].encode("utf-8")
                ).hexdigest()
            ]
            for query_record in evaluation_query_records
        ]
    ).astype(np.float32)

    query_matrix = l2_normalize_rows(query_matrix)

    child_passage_matrix = np.vstack(
        [
            embedding_lookup["passage_vectors_by_text_hash"][text_hash]
            for text_hash in parent_child_candidate_data[
                "chunk_text_hashes"
            ]
        ]
    ).astype(np.float32)

    child_passage_matrix = l2_normalize_rows(
        child_passage_matrix
    )

    similarity_matrix = query_matrix @ child_passage_matrix.T

    parent_context_token_totals = []
    parent_context_char_totals = []
    parent_context_unique_source_counts = []
    parent_context_parent_counts = []
    parent_context_contract_passes = []

    for row_index in range(len(evaluation_query_records)):
        top_child_indexes_unsorted = np.argpartition(
            similarity_matrix[row_index],
            -citation_evaluation_cutoff,
        )[-citation_evaluation_cutoff:]

        top_child_indexes = top_child_indexes_unsorted[
            np.argsort(
                similarity_matrix[row_index][top_child_indexes_unsorted]
            )[::-1]
        ]

        top_child_chunk_ids = [
            parent_child_candidate_data["chunk_ids"][child_index]
            for child_index in top_child_indexes
        ]

        parent_chunk_ids = list(
            dict.fromkeys(
                child_to_parent_ids[child_chunk_id]
                for child_chunk_id in top_child_chunk_ids
            )
        )

        parent_context_chunks = [
            parent_chunks_by_id[parent_chunk_id]
            for parent_chunk_id in parent_chunk_ids
        ]

        parent_context_token_totals.append(
            sum(
                parent_chunk["token_count"]
                for parent_chunk in parent_context_chunks
            )
        )
        parent_context_char_totals.append(
            sum(
                parent_chunk["char_count"]
                for parent_chunk in parent_context_chunks
            )
        )
        parent_context_unique_source_counts.append(
            len(
                {
                    parent_chunk["source_id"]
                    for parent_chunk in parent_context_chunks
                }
            )
        )
        parent_context_parent_counts.append(
            len(parent_context_chunks)
        )
        parent_context_contract_passes.append(
            all(
                isinstance(parent_chunk["source_id"], str)
                and parent_chunk["source_id"]
                and isinstance(parent_chunk["source_type"], str)
                and parent_chunk["source_type"]
                and isinstance(parent_chunk["locator_start"], dict)
                and isinstance(parent_chunk["locator_end"], dict)
                and isinstance(
                    parent_chunk["attribution_context"],
                    dict,
                )
                for parent_chunk in parent_context_chunks
            )
        )

    metrics = {
        "mean_parent_context_tokens": float(
            np.mean(parent_context_token_totals)
        ),
        "p95_parent_context_tokens": float(
            np.percentile(parent_context_token_totals, 95)
        ),
        "mean_parent_context_characters": float(
            np.mean(parent_context_char_totals)
        ),
        "mean_unique_parent_sources": float(
            np.mean(parent_context_unique_source_counts)
        ),
        "mean_unique_parent_chunks": float(
            np.mean(parent_context_parent_counts)
        ),
        "parent_context_citation_contract_pass_rate": float(
            np.mean(parent_context_contract_passes)
        ),
    }

    parent_child_context_quality_results[model_name] = metrics

    print(f"\n{model_name}")
    print(
        "- Mean parent-context tokens:",
        f"{metrics['mean_parent_context_tokens']:.1f}",
    )
    print(
        "- P95 parent-context tokens:",
        f"{metrics['p95_parent_context_tokens']:.1f}",
    )
    print(
        "- Mean parent-context characters:",
        f"{metrics['mean_parent_context_characters']:.1f}",
    )
    print(
        "- Mean unique parent sources:",
        f"{metrics['mean_unique_parent_sources']:.2f}",
    )
    print(
        "- Mean unique parent chunks:",
        f"{metrics['mean_unique_parent_chunks']:.2f}",
    )
    print(
        "- Citation contract pass:",
        f"{metrics['parent_context_citation_contract_pass_rate']:.4f}",
    )

print("\nPARENT–CHILD REAL CONTEXT QUALITY: PASS")

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۴۰. ثبت تصمیم Trade-off و وضعیت آمادگی انتخاب

همهٔ ارزیابی‌های خودکار retrieval، citation، attribution و context اجرا شده‌اند.

در این مرحله نتیجهٔ واقعی ثبت می‌شود.  
اگر یک ترکیب در همهٔ ابعاد برنده نباشد، انتخاب نهایی انجام نمی‌شود و علت آن به‌صورت شفاف ثبت خواهد شد.

</div>

In [ ]:
# Code Cell 52

question_retrieval_ranking = sorted(
    [
        {
            "model": model_name,
            "candidate": candidate_name,
            **metrics,
        }
        for model_name, candidate_results
        in embedding_benchmark_results.items()
        for candidate_name, metrics
        in candidate_results.items()
    ],
    key=lambda item: (
        -item["recall_at_10"],
        -item["mrr"],
    ),
)

evidence_retrieval_ranking = sorted(
    [
        {
            "model": model_name,
            "candidate": candidate_name,
            **metrics,
        }
        for model_name, candidate_results
        in evidence_probe_results.items()
        for candidate_name, metrics
        in candidate_results.items()
    ],
    key=lambda item: (
        -item["recall_at_10"],
        -item["mrr"],
    ),
)

best_question_retrieval = question_retrieval_ranking[0]
best_evidence_retrieval = evidence_retrieval_ranking[0]

embedding_benchmark_tradeoff = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "selection_status": "NO_FINAL_SELECTION",
    "decision_basis": {
        "best_question_retrieval": best_question_retrieval,
        "best_evidence_retrieval": best_evidence_retrieval,
        "citation_contract_status": "PASS_FOR_ALL_COMBINATIONS",
        "parent_child_context_contract_status": "PASS_FOR_ALL_MODELS",
    },
    "blocking_findings": [
        (
            "Question-based retrieval is materially weaker than "
            "source-evidence retrieval across the evaluated combinations."
        ),
        (
            "No single model-and-chunking combination leads both "
            "question retrieval and source-evidence retrieval."
        ),
        (
            "Parent-child context requires a context-budget policy "
            "before production use."
        ),
    ],
    "required_next_work": [
        "query_side_retrieval_remediation",
        "context_budget_and_assembly_policy",
        "post_remediation_retrieval_benchmark",
        "final_tradeoff_decision",
    ],
}

tradeoff_report_path = (
    active_benchmark_run_directory
    / "embedding_benchmark_tradeoff.json"
)

tradeoff_report_path.write_text(
    json.dumps(
        embedding_benchmark_tradeoff,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== EMBEDDING BENCHMARK TRADE-OFF DECISION ===")
print("Selection status:", embedding_benchmark_tradeoff["selection_status"])

print("\nBest question-retrieval combination:")
print(
    f"- model={best_question_retrieval['model']}, "
    f"candidate={best_question_retrieval['candidate']}, "
    f"Recall@10={best_question_retrieval['recall_at_10']:.4f}, "
    f"MRR={best_question_retrieval['mrr']:.4f}"
)

print("\nBest source-evidence-retrieval combination:")
print(
    f"- model={best_evidence_retrieval['model']}, "
    f"candidate={best_evidence_retrieval['candidate']}, "
    f"Recall@10={best_evidence_retrieval['recall_at_10']:.4f}, "
    f"MRR={best_evidence_retrieval['mrr']:.4f}"
)

print("\nBlocking findings:")
for finding in embedding_benchmark_tradeoff["blocking_findings"]:
    print("-", finding)

print("\nRequired next work:")
for next_step in embedding_benchmark_tradeoff["required_next_work"]:
    print("-", next_step)

print("\nReport:", tradeoff_report_path)
print("\nEMBEDDING BENCHMARK TRADE-OFF DECISION: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## اصلاح قرارداد ورودی مدل E5

مدل `multilingual-e5-small` برای بازیابی معنایی با دو پیشوند استاندارد طراحی شده است:

- برای پرسش: `query:`
- برای متن بازیابی‌شونده: `passage:`

بنابراین پیش از تغییر Chunking یا انتخاب مدل نهایی، همین مدل را یک‌بار با قرارداد صحیح خودش اجرا می‌کنیم و نتیجه را جدا از اجرای قبلی نگه می‌داریم.

این آزمایش:

- Golden Dataset فعال را از متغیر اصلی می‌خواند؛
- هیچ فایل یا نتیجهٔ قبلی را جایگزین نمی‌کند؛
- خروجی را با هش Golden Dataset فعال ذخیره می‌کند؛
- در اجرای آینده با Golden Dataset جدید، خودکار در پوشهٔ جداگانهٔ همان نسخه اجرا می‌شود.

</div>

In [ ]:
# Code Cell 53
# NB13 — E5 instructional embedding cache
# Reuses the existing cache when the active Golden Dataset is unchanged.

from pathlib import Path
import hashlib
import json
import numpy as np

assert "golden_dataset_path" in globals(), (
    "Run the Active Golden Dataset loader cell first."
)
assert "query_text_by_hash" in globals(), (
    "Run the embedding workload-discovery cell first."
)
assert "chunk_text_by_hash" in globals(), (
    "Run the embedding workload-discovery cell first."
)
assert "embedding_models" in globals(), (
    "Run the local embedding-model validation cell first."
)
assert "multilingual_e5_small" in embedding_models, (
    "The multilingual_e5_small model is not loaded."
)

e5_model = embedding_models["multilingual_e5_small"]

project_root = golden_dataset_path.parents[3]
e5_instructional_run_directory = (
    project_root
    / "artifacts"
    / "phase2"
    / "embedding_benchmark"
    / golden_dataset_context["sha256"]
)
e5_instructional_run_directory.mkdir(parents=True, exist_ok=True)

e5_instructional_cache_path = (
    e5_instructional_run_directory
    / "multilingual_e5_small_instructional_embeddings.npz"
)
e5_instructional_manifest_path = (
    e5_instructional_run_directory
    / "multilingual_e5_small_instructional_manifest.json"
)

query_hashes = list(query_text_by_hash.keys())
passage_hashes = list(chunk_text_by_hash.keys())

def hash_sequence(values):
    return hashlib.sha256(
        "\n".join(sorted(values)).encode("utf-8")
    ).hexdigest()

expected_query_hash_sequence = hash_sequence(query_hashes)
expected_passage_hash_sequence = hash_sequence(passage_hashes)

def encode_e5_instructional(text_by_hash, prefix, batch_size=128):
    hashes = list(text_by_hash.keys())
    prefixed_texts = [
        f"{prefix}: {text_by_hash[text_hash]}"
        for text_hash in hashes
    ]

    vectors = e5_model.encode(
        prefixed_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    return hashes, vectors

cache_is_reusable = False

if e5_instructional_cache_path.is_file():
    cached_arrays = np.load(
        e5_instructional_cache_path,
        allow_pickle=True,
    )

    cached_query_hashes = [
        str(text_hash)
        for text_hash in cached_arrays["query_hashes"].tolist()
    ]
    cached_passage_hashes = [
        str(text_hash)
        for text_hash in cached_arrays["passage_hashes"].tolist()
    ]

    cache_is_reusable = (
        set(cached_query_hashes) == set(query_hashes)
        and set(cached_passage_hashes) == set(passage_hashes)
        and len(cached_query_hashes) == len(set(cached_query_hashes))
        and len(cached_passage_hashes) == len(set(cached_passage_hashes))
        and cached_arrays["query_vectors"].shape
        == (len(query_hashes), 384)
        and cached_arrays["passage_vectors"].shape
        == (len(passage_hashes), 384)
    )

if cache_is_reusable:
    e5_instructional_query_vectors = {
        text_hash: vector
        for text_hash, vector in zip(
            cached_query_hashes,
            cached_arrays["query_vectors"],
        )
    }
    e5_instructional_passage_vectors = {
        text_hash: vector
        for text_hash, vector in zip(
            cached_passage_hashes,
            cached_arrays["passage_vectors"],
        )
    }

    print("Existing E5 instructional cache loaded.")
else:
    print("Encoding E5 queries with prefix: query:")
    encoded_query_hashes, encoded_query_vectors = encode_e5_instructional(
        query_text_by_hash,
        prefix="query",
    )

    print("\nEncoding E5 passages with prefix: passage:")
    encoded_passage_hashes, encoded_passage_vectors = encode_e5_instructional(
        chunk_text_by_hash,
        prefix="passage",
    )

    np.savez_compressed(
        e5_instructional_cache_path,
        query_hashes=np.array(encoded_query_hashes, dtype=object),
        query_vectors=encoded_query_vectors,
        passage_hashes=np.array(encoded_passage_hashes, dtype=object),
        passage_vectors=encoded_passage_vectors,
    )

    e5_instructional_query_vectors = {
        text_hash: vector
        for text_hash, vector in zip(
            encoded_query_hashes,
            encoded_query_vectors,
        )
    }
    e5_instructional_passage_vectors = {
        text_hash: vector
        for text_hash, vector in zip(
            encoded_passage_hashes,
            encoded_passage_vectors,
        )
    }

e5_instructional_manifest = {
    "artifact": e5_instructional_cache_path.name,
    "golden_dataset_path": str(golden_dataset_path),
    "golden_dataset_sha256": golden_dataset_context["sha256"],
    "model_name": "intfloat/multilingual-e5-small",
    "query_prefix": "query",
    "passage_prefix": "passage",
    "normalization": "L2",
    "query_count": len(query_hashes),
    "passage_count": len(passage_hashes),
    "query_hash_set_sha256": expected_query_hash_sequence,
    "passage_hash_set_sha256": expected_passage_hash_sequence,
    "embedding_dimension": 384,
}

e5_instructional_manifest_path.write_text(
    json.dumps(
        e5_instructional_manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

e5_instructional_embeddings = {
    "model_name": "multilingual_e5_small_instructional",
    "model_id": "intfloat/multilingual-e5-small",
    "query_prefix": "query:",
    "passage_prefix": "passage:",
    "dimension": 384,
    "query_vectors_by_text_hash": e5_instructional_query_vectors,
    "passage_vectors_by_text_hash": e5_instructional_passage_vectors,
    "cache_path": e5_instructional_cache_path,
}

print("\n=== E5 INSTRUCTIONAL EMBEDDING CACHE: PASS ===")
print("Cache reused:", cache_is_reusable)
print("Active Golden Dataset hash:", golden_dataset_context["sha256"])
print("Queries:", len(e5_instructional_query_vectors))
print("Passages:", len(e5_instructional_passage_vectors))
print("Dimension:", e5_instructional_embeddings["dimension"])
print("Cache:", e5_instructional_cache_path)

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## کنترل آماده‌بودن Benchmark اصلاح‌شدهٔ E5

در این مرحله بررسی می‌کنیم که embeddingهای جدید E5 کامل، نرمال‌سازی‌شده و آمادهٔ ورود به همان Benchmark قبلی هستند.

هیچ فایل، نتیجه یا Benchmark قبلی تغییر نمی‌کند.

</div>

In [ ]:
# Code Cell 54
# NB13 — E5 instructional embedding validation and benchmark preflight

expected_query_hashes = set(query_text_by_hash)
expected_passage_hashes = set(chunk_text_by_hash)

actual_query_hashes = set(
    e5_instructional_embeddings["query_vectors_by_text_hash"]
)
actual_passage_hashes = set(
    e5_instructional_embeddings["passage_vectors_by_text_hash"]
)

assert actual_query_hashes == expected_query_hashes, (
    "Instructional E5 query cache is incomplete or has unexpected keys."
)
assert actual_passage_hashes == expected_passage_hashes, (
    "Instructional E5 passage cache is incomplete or has unexpected keys."
)

query_matrix = np.vstack(
    [
        e5_instructional_embeddings["query_vectors_by_text_hash"][text_hash]
        for text_hash in query_text_by_hash
    ]
)
passage_matrix = np.vstack(
    [
        e5_instructional_embeddings["passage_vectors_by_text_hash"][text_hash]
        for text_hash in chunk_text_by_hash
    ]
)

assert query_matrix.shape == (len(query_text_by_hash), 384)
assert passage_matrix.shape == (len(chunk_text_by_hash), 384)
assert np.isfinite(query_matrix).all()
assert np.isfinite(passage_matrix).all()

assert np.allclose(
    np.linalg.norm(query_matrix, axis=1),
    1.0,
    atol=1e-4,
), "Instructional E5 query vectors are not L2-normalized."

assert np.allclose(
    np.linalg.norm(passage_matrix, axis=1),
    1.0,
    atol=1e-4,
), "Instructional E5 passage vectors are not L2-normalized."

print("=== E5 INSTRUCTIONAL BENCHMARK PREFLIGHT ===")
print("Query vectors:", query_matrix.shape)
print("Passage vectors:", passage_matrix.shape)
print("Query prefix:", e5_instructional_embeddings["query_prefix"])
print("Passage prefix:", e5_instructional_embeddings["passage_prefix"])
print("\nE5 INSTRUCTIONAL BENCHMARK PREFLIGHT: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## کشف قرارداد Benchmark بازیابی

پیش از اجرای Benchmark جدید، ساختار دقیق برچسب‌های مثبت و پرسش‌های واجد شرایط را از همان اجرای قبلی بررسی می‌کنیم.

هدف این است که E5 اصلاح‌شده دقیقاً با همان داده، همان چهار Chunking و همان معیارهای قبلی ارزیابی شود؛ بنابراین مقایسه کاملاً منصفانه خواهد بود.

</div>

In [ ]:
# Code Cell 55
# NB13 — Retrieval benchmark contract discovery

def summarize_benchmark_object(name, value, max_keys=15):
    print(f"\n{name}")
    print("Type:", type(value).__name__)

    if isinstance(value, dict):
        print("Key count:", len(value))
        print("Key preview:", list(value.keys())[:max_keys])

        if value:
            first_key = next(iter(value))
            first_value = value[first_key]
            print("First key:", repr(first_key))
            print("First value type:", type(first_value).__name__)

            if isinstance(first_value, dict):
                print(
                    "First nested-key preview:",
                    list(first_value.keys())[:max_keys],
                )

    elif isinstance(value, (list, tuple, set)):
        print("Item count:", len(value))

        if value:
            first_value = next(iter(value))
            print("First item type:", type(first_value).__name__)

            if isinstance(first_value, dict):
                print(
                    "First-item key preview:",
                    list(first_value.keys())[:max_keys],
                )

    else:
        print("Value preview:", repr(value)[:500])


benchmark_contract_candidates = {
    name: value
    for name, value in globals().items()
    if any(
        token in name.lower()
        for token in (
            "evaluation_registry",
            "retrieval_registry",
            "positive",
            "eligible",
            "benchmark_query",
            "golden_query",
        )
    )
}

print("=== RETRIEVAL BENCHMARK CONTRACT DISCOVERY ===")

print("\nRetrieval candidates:")
for candidate_name, candidate in retrieval_chunking_candidates.items():
    print(
        f"- {candidate_name}: "
        f"records={candidate['records']}, "
        f"path={candidate['path']}"
    )

print("\nRelevant evaluation objects discovered:")
assert benchmark_contract_candidates, (
    "No benchmark-contract object was found. "
    "Run the Final Retrieval Evaluation Registry cell again."
)

for object_name, object_value in benchmark_contract_candidates.items():
    summarize_benchmark_object(object_name, object_value)

print("\nRETRIEVAL BENCHMARK CONTRACT DISCOVERY: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## Benchmark بازیابی پرسش با E5 اصلاح‌شده

در این آزمایش، E5 با پیشوندهای استاندارد `query:` و `passage:` روی همان پرسش‌ها، همان برچسب‌های مثبت و همان چهار روش Chunking قبلی ارزیابی می‌شود.

نتیجه مستقیماً با اجرای قبلی `multilingual_e5_small` مقایسه خواهد شد.

</div>

In [ ]:
# Code Cell 56
# NB13 — Instructional E5 cache-to-chunk key reconciliation

query_hash_by_text = {
    text: text_hash
    for text_hash, text in query_text_by_hash.items()
}

passage_hash_by_text = {
    text: text_hash
    for text_hash, text in chunk_text_by_hash.items()
}

assert len(query_hash_by_text) == len(query_text_by_hash)
assert len(passage_hash_by_text) == len(chunk_text_by_hash)

candidate_content_reconciliation = {}

print("=== INSTRUCTIONAL E5 CACHE-TO-CHUNK RECONCILIATION ===")
print("Cached passage texts:", len(passage_hash_by_text))

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = project_root / candidate["path"]

    assert candidate_path.is_file(), (
        f"Missing frozen chunk candidate: {candidate_path}"
    )

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    matched_records = [
        record
        for record in candidate_records
        if record["content"] in passage_hash_by_text
    ]
    unmatched_records = [
        record
        for record in candidate_records
        if record["content"] not in passage_hash_by_text
    ]

    candidate_content_reconciliation[candidate_name] = {
        "records": len(candidate_records),
        "matched_records": len(matched_records),
        "unmatched_records": len(unmatched_records),
        "first_unmatched_record": (
            unmatched_records[0]
            if unmatched_records
            else None
        ),
    }

    print(f"\n{candidate_name}")
    print("Records:", len(candidate_records))
    print("Exact cache-text matches:", len(matched_records))
    print("Unmatched records:", len(unmatched_records))

    if unmatched_records:
        sample = unmatched_records[0]["content"]
        sample_prefix = sample[:180]

        prefix_matches = [
            cached_text
            for cached_text in chunk_text_by_hash.values()
            if cached_text.startswith(sample_prefix)
            or sample.startswith(cached_text[:180])
        ]

        print("\nFirst unmatched-content diagnostic:")
        print("Raw content length:", len(sample))
        print("Raw content preview:", repr(sample[:300]))
        print("Possible cached-text prefix matches:", len(prefix_matches))

        if prefix_matches:
            matched_text = prefix_matches[0]
            print("Cached content length:", len(matched_text))
            print("Cached content preview:", repr(matched_text[:300]))
            print("Texts exactly equal:", sample == matched_text)

print("\nAvailable text-normalization functions:")
normalization_function_names = sorted(
    name
    for name, value in globals().items()
    if callable(value)
    and any(
        token in name.lower()
        for token in (
            "normalize",
            "canonical",
            "clean_text",
            "text_hash",
            "hash_text",
        )
    )
)

for function_name in normalization_function_names:
    print("-", function_name)

print("\nINSTRUCTIONAL E5 CACHE-TO-CHUNK RECONCILIATION: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## تعیین کلید متنی استاندارد برای Chunkها

فایل‌های Chunk شامل متن خام هستند، اما cache embedding با نسخهٔ استانداردشدهٔ متن ساخته شده است.

در این مرحله تابع استانداردسازی درست را تعیین می‌کنیم تا Benchmark اصلاح‌شده دقیقاً با همان cache معتبر اجرا شود.

</div>

In [ ]:
# Code Cell 57
# NB13 — Canonical chunk-text key validation and selection

canonical_key_functions = {
    "canonical_exact_text": canonical_exact_text,
    "canonical_comparison_text": canonical_comparison_text,
}

canonical_key_validation = {}
all_candidate_raw_contents = []

print("=== CANONICAL CHUNK-TEXT KEY VALIDATION ===")

for function_name, canonicalize_text in canonical_key_functions.items():
    print(f"\nFunction: {function_name}")

    function_results = {}

    for candidate_name, candidate in retrieval_chunking_candidates.items():
        candidate_path = project_root / candidate["path"]

        candidate_records = [
            json.loads(line)
            for line in candidate_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]

        if function_name == "canonical_exact_text":
            all_candidate_raw_contents.extend(
                record["content"]
                for record in candidate_records
            )

        matched_records = [
            record
            for record in candidate_records
            if canonicalize_text(record["content"]) in passage_hash_by_text
        ]

        function_results[candidate_name] = {
            "records": len(candidate_records),
            "canonical_matches": len(matched_records),
            "unmatched_records": len(candidate_records) - len(matched_records),
        }

        print(
            f"- {candidate_name}: "
            f"{len(matched_records)} / {len(candidate_records)} "
            f"canonical matches"
        )

    canonical_key_validation[function_name] = function_results

fully_matching_functions = [
    function_name
    for function_name, candidate_results in canonical_key_validation.items()
    if all(
        result["canonical_matches"] == result["records"]
        for result in candidate_results.values()
    )
]

assert "canonical_exact_text" in fully_matching_functions, (
    "canonical_exact_text does not fully match the existing embedding cache."
)

canonical_functions_are_equivalent = all(
    canonical_exact_text(content)
    == canonical_comparison_text(content)
    for content in all_candidate_raw_contents
)

active_chunk_cache_key_function_name = "canonical_exact_text"
active_chunk_cache_key_function = canonical_exact_text

print(
    "\nFunctions equivalent for all frozen candidate contents:",
    canonical_functions_are_equivalent,
)
print("Selected cache-key function:", active_chunk_cache_key_function_name)
print("\nCANONICAL CHUNK-TEXT KEY VALIDATION: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## Benchmark بازیابی پرسش با E5 اصلاح‌شده

E5 با پیشوندهای استاندارد `query:` و `passage:` اکنون با همان ۵۵۳ پرسش، همان ۲۷ seed واجد شرایط، همان برچسب‌های مثبت و همان چهار روش Chunking قبلی ارزیابی می‌شود.

نتیجه مستقیماً با E5 قبلی مقایسه خواهد شد. هیچ نتیجهٔ قبلی جایگزین نمی‌شود.

</div>

In [ ]:
# Code Cell 58
# NB13 — Full question-retrieval benchmark for instructional E5

import time

instructional_benchmark_queries = [
    record
    for record in benchmark_query_records
    if record["source_seed_id"] in common_eligible_seed_ids
]

assert len(instructional_benchmark_queries) == 553, (
    "Expected the same 553 eligible Golden questions as the prior benchmark."
)

assert all(
    record["question"] in query_hash_by_text
    for record in instructional_benchmark_queries
), "One or more Golden questions are missing from the instructional E5 cache."

instructional_query_matrix = np.vstack(
    [
        e5_instructional_embeddings["query_vectors_by_text_hash"][
            query_hash_by_text[record["question"]]
        ]
        for record in instructional_benchmark_queries
    ]
)

e5_instructional_benchmark_results = {}

print("=== E5 INSTRUCTIONAL FULL RETRIEVAL BENCHMARK ===")
print("Eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(instructional_benchmark_queries))
print("Evaluation cutoffs: (1, 3, 5, 10)")

for candidate_name, seed_registry in retrieval_evaluation_registry.items():
    candidate_path = project_root / retrieval_chunking_candidates[candidate_name]["path"]

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    candidate_chunk_ids = [
        record["chunk_id"]
        for record in candidate_records
    ]

    canonical_candidate_texts = [
        active_chunk_cache_key_function(record["content"])
        for record in candidate_records
    ]

    assert all(
        canonical_text in passage_hash_by_text
        for canonical_text in canonical_candidate_texts
    ), (
        f"{candidate_name} contains a Chunk not represented in "
        "the instructional E5 cache."
    )

    candidate_passage_matrix = np.vstack(
        [
            e5_instructional_embeddings["passage_vectors_by_text_hash"][
                passage_hash_by_text[canonical_text]
            ]
            for canonical_text in canonical_candidate_texts
        ]
    )

    positive_chunk_indexes_by_seed = {}

    for seed_id in common_eligible_seed_ids:
        seed_entry = seed_registry[seed_id]

        assert seed_entry["eligible"] is True, (
            f"{candidate_name}, seed {seed_id} is unexpectedly ineligible."
        )

        positive_ids = set(seed_entry["positive_chunk_ids"])
        positive_indexes = {
            index
            for index, chunk_id in enumerate(candidate_chunk_ids)
            if chunk_id in positive_ids
        }

        assert positive_indexes, (
            f"{candidate_name}, seed {seed_id} has no indexed positive Chunk."
        )

        positive_chunk_indexes_by_seed[seed_id] = positive_indexes

    started_at = time.perf_counter()
    similarity_matrix = instructional_query_matrix @ candidate_passage_matrix.T
    ranked_indexes = np.argsort(-similarity_matrix, axis=1)
    elapsed_seconds = time.perf_counter() - started_at

    first_positive_ranks = []
    hit_lists = {cutoff: [] for cutoff in (1, 3, 5, 10)}

    for query_index, query_record in enumerate(instructional_benchmark_queries):
        seed_id = query_record["source_seed_id"]
        positive_indexes = positive_chunk_indexes_by_seed[seed_id]

        ranked_candidate_indexes = ranked_indexes[query_index]
        positive_rank_positions = np.flatnonzero(
            np.isin(ranked_candidate_indexes, list(positive_indexes))
        )

        first_rank = int(positive_rank_positions[0]) + 1
        first_positive_ranks.append(first_rank)

        for cutoff in hit_lists:
            hit_lists[cutoff].append(first_rank <= cutoff)

    metrics = {
        "recall_at_1": float(np.mean(hit_lists[1])),
        "recall_at_3": float(np.mean(hit_lists[3])),
        "recall_at_5": float(np.mean(hit_lists[5])),
        "recall_at_10": float(np.mean(hit_lists[10])),
        "mrr": float(np.mean([1 / rank for rank in first_positive_ranks])),
        "mean_first_positive_rank": float(np.mean(first_positive_ranks)),
        "seconds": round(elapsed_seconds, 3),
        "query_count": len(instructional_benchmark_queries),
        "candidate_chunk_count": len(candidate_records),
    }

    e5_instructional_benchmark_results[candidate_name] = metrics
    prior_metrics = embedding_benchmark_results["multilingual_e5_small"][candidate_name]

    print(f"\nmultilingual_e5_small_instructional × {candidate_name}")
    print(f"- Recall@1: {metrics['recall_at_1']:.4f}")
    print(f"- Recall@3: {metrics['recall_at_3']:.4f}")
    print(f"- Recall@5: {metrics['recall_at_5']:.4f}")
    print(f"- Recall@10: {metrics['recall_at_10']:.4f}")
    print(f"- MRR: {metrics['mrr']:.4f}")
    print(
        f"- Δ Recall@10 vs prior E5: "
        f"{metrics['recall_at_10'] - prior_metrics['recall_at_10']:+.4f}"
    )
    print(
        f"- Δ MRR vs prior E5: "
        f"{metrics['mrr'] - prior_metrics['mrr']:+.4f}"
    )
    print(f"- Seconds: {metrics['seconds']}")

e5_instructional_benchmark_report_path = (
    active_benchmark_run_directory
    / "multilingual_e5_small_instructional_retrieval_benchmark.json"
)

e5_instructional_benchmark_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "model": e5_instructional_embeddings["model_id"],
            "query_prefix": e5_instructional_embeddings["query_prefix"],
            "passage_prefix": e5_instructional_embeddings["passage_prefix"],
            "comparison_baseline": "multilingual_e5_small_without_prefixes",
            "common_eligible_seed_ids": common_eligible_seed_ids,
            "query_count": len(instructional_benchmark_queries),
            "results": e5_instructional_benchmark_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nReport:", e5_instructional_benchmark_report_path)
print("\nE5 INSTRUCTIONAL FULL RETRIEVAL BENCHMARK: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی بازیابی واژگانی فارسی

نتایج قبلی نشان دادند که بازیابی embedding برای پرسش‌های Golden ضعیف‌تر از انتظار است.

در این مرحله یک بازیابی واژگانی مستقل بر پایهٔ TF-IDF فارسی اجرا می‌کنیم. این روش از شباهت واژه‌ها و عبارت‌های پرسش با متن Chunk استفاده می‌کند و نتیجهٔ آن با همان برچسب‌های مثبت و همان چهار روش Chunking ارزیابی می‌شود.

این فقط یک Baseline تشخیصی است و جایگزین embedding یا انتخاب نهایی سیستم نیست.

</div>

In [ ]:
# Code Cell 59
# NB13 — Persian lexical retrieval baseline: TF-IDF word and phrase matching

import re
import time
from sklearn.feature_extraction.text import TfidfVectorizer

def persian_lexical_tokens(text):
    canonical_text = canonical_exact_text(text).lower()
    return re.findall(r"[\u0600-\u06FFa-z0-9]+", canonical_text)

lexical_benchmark_queries = [
    record
    for record in benchmark_query_records
    if record["source_seed_id"] in common_eligible_seed_ids
]

assert len(lexical_benchmark_queries) == 553

lexical_query_texts = [
    record["question"]
    for record in lexical_benchmark_queries
]

lexical_retrieval_results = {}

print("=== PERSIAN LEXICAL RETRIEVAL BASELINE ===")
print("Eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(lexical_benchmark_queries))
print("Tokenizer: Persian word tokens")
print("Features: word unigrams and bigrams")
print("Evaluation cutoffs: (1, 3, 5, 10)")

for candidate_name, seed_registry in retrieval_evaluation_registry.items():
    candidate_path = project_root / retrieval_chunking_candidates[candidate_name]["path"]

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    candidate_chunk_ids = [
        record["chunk_id"]
        for record in candidate_records
    ]
    candidate_texts = [
        canonical_exact_text(record["content"])
        for record in candidate_records
    ]

    positive_chunk_indexes_by_seed = {}

    for seed_id in common_eligible_seed_ids:
        seed_entry = seed_registry[seed_id]
        assert seed_entry["eligible"] is True

        positive_ids = set(seed_entry["positive_chunk_ids"])
        positive_indexes = {
            index
            for index, chunk_id in enumerate(candidate_chunk_ids)
            if chunk_id in positive_ids
        }

        assert positive_indexes, (
            f"{candidate_name}, seed {seed_id} has no indexed positive Chunk."
        )

        positive_chunk_indexes_by_seed[seed_id] = positive_indexes

    vectorizer = TfidfVectorizer(
        tokenizer=persian_lexical_tokens,
        token_pattern=None,
        ngram_range=(1, 2),
        lowercase=False,
        norm="l2",
        sublinear_tf=True,
        dtype=np.float32,
    )

    started_at = time.perf_counter()

    candidate_tfidf = vectorizer.fit_transform(candidate_texts)
    query_tfidf = vectorizer.transform(lexical_query_texts)

    similarity_matrix = (query_tfidf @ candidate_tfidf.T).toarray()
    ranked_indexes = np.argsort(-similarity_matrix, axis=1)

    elapsed_seconds = time.perf_counter() - started_at

    first_positive_ranks = []
    hit_lists = {cutoff: [] for cutoff in (1, 3, 5, 10)}

    for query_index, query_record in enumerate(lexical_benchmark_queries):
        seed_id = query_record["source_seed_id"]
        positive_indexes = positive_chunk_indexes_by_seed[seed_id]

        ranked_candidate_indexes = ranked_indexes[query_index]
        positive_rank_positions = np.flatnonzero(
            np.isin(ranked_candidate_indexes, list(positive_indexes))
        )

        first_rank = int(positive_rank_positions[0]) + 1
        first_positive_ranks.append(first_rank)

        for cutoff in hit_lists:
            hit_lists[cutoff].append(first_rank <= cutoff)

    metrics = {
        "recall_at_1": float(np.mean(hit_lists[1])),
        "recall_at_3": float(np.mean(hit_lists[3])),
        "recall_at_5": float(np.mean(hit_lists[5])),
        "recall_at_10": float(np.mean(hit_lists[10])),
        "mrr": float(np.mean([1 / rank for rank in first_positive_ranks])),
        "mean_first_positive_rank": float(np.mean(first_positive_ranks)),
        "seconds": round(elapsed_seconds, 3),
        "query_count": len(lexical_benchmark_queries),
        "candidate_chunk_count": len(candidate_records),
        "vocabulary_size": int(len(vectorizer.vocabulary_)),
    }

    lexical_retrieval_results[candidate_name] = metrics

    print(f"\npersian_tfidf_lexical × {candidate_name}")
    print(f"- Recall@1: {metrics['recall_at_1']:.4f}")
    print(f"- Recall@3: {metrics['recall_at_3']:.4f}")
    print(f"- Recall@5: {metrics['recall_at_5']:.4f}")
    print(f"- Recall@10: {metrics['recall_at_10']:.4f}")
    print(f"- MRR: {metrics['mrr']:.4f}")
    print(f"- Vocabulary size: {metrics['vocabulary_size']}")
    print(f"- Seconds: {metrics['seconds']}")

print("\nPERSIAN LEXICAL RETRIEVAL BASELINE: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی Hybrid Retrieval با ترکیب E5 و بازیابی واژگانی

Embedding و بازیابی واژگانی هر کدام ممکن است بخشی از پرسش‌ها را بهتر بازیابی کنند.

در این مرحله رتبه‌های E5 و TF-IDF فارسی با روش Reciprocal Rank Fusion یا RRF ترکیب می‌شوند. هدف، بررسی این است که آیا ترکیب دو روش از هر روش به‌تنهایی بهتر است یا خیر.

این یک آزمایش Benchmark است و هنوز انتخاب نهایی سیستم نیست.

</div>

In [ ]:
# Code Cell 60
# NB13 — Hybrid E5 + Persian lexical retrieval benchmark with RRF

from sklearn.feature_extraction.text import TfidfVectorizer
import time

def calculate_retrieval_metrics(
    ranked_indexes,
    query_records,
    positive_chunk_indexes_by_seed,
):
    first_positive_ranks = []
    hit_lists = {cutoff: [] for cutoff in (1, 3, 5, 10)}

    for query_index, query_record in enumerate(query_records):
        seed_id = query_record["source_seed_id"]
        positive_indexes = positive_chunk_indexes_by_seed[seed_id]

        positive_rank_positions = np.flatnonzero(
            np.isin(
                ranked_indexes[query_index],
                list(positive_indexes),
            )
        )

        first_rank = int(positive_rank_positions[0]) + 1
        first_positive_ranks.append(first_rank)

        for cutoff in hit_lists:
            hit_lists[cutoff].append(first_rank <= cutoff)

    return {
        "recall_at_1": float(np.mean(hit_lists[1])),
        "recall_at_3": float(np.mean(hit_lists[3])),
        "recall_at_5": float(np.mean(hit_lists[5])),
        "recall_at_10": float(np.mean(hit_lists[10])),
        "mrr": float(np.mean([1 / rank for rank in first_positive_ranks])),
        "mean_first_positive_rank": float(np.mean(first_positive_ranks)),
    }


hybrid_query_records = [
    record
    for record in benchmark_query_records
    if record["source_seed_id"] in common_eligible_seed_ids
]

assert len(hybrid_query_records) == 553

hybrid_e5_query_matrix = np.vstack(
    [
        e5_instructional_embeddings["query_vectors_by_text_hash"][
            query_hash_by_text[record["question"]]
        ]
        for record in hybrid_query_records
    ]
)

hybrid_query_texts = [
    record["question"]
    for record in hybrid_query_records
]

rrf_constant = 60
hybrid_retrieval_results = {}

print("=== HYBRID E5 + PERSIAN LEXICAL RETRIEVAL BENCHMARK ===")
print("Eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(hybrid_query_records))
print("Fusion method: Reciprocal Rank Fusion")
print("RRF constant:", rrf_constant)

for candidate_name, seed_registry in retrieval_evaluation_registry.items():
    candidate_path = project_root / retrieval_chunking_candidates[candidate_name]["path"]

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    candidate_chunk_ids = [
        record["chunk_id"]
        for record in candidate_records
    ]
    candidate_texts = [
        canonical_exact_text(record["content"])
        for record in candidate_records
    ]

    positive_chunk_indexes_by_seed = {}

    for seed_id in common_eligible_seed_ids:
        seed_entry = seed_registry[seed_id]
        positive_ids = set(seed_entry["positive_chunk_ids"])

        positive_indexes = {
            index
            for index, chunk_id in enumerate(candidate_chunk_ids)
            if chunk_id in positive_ids
        }

        assert positive_indexes, (
            f"{candidate_name}, seed {seed_id} has no indexed positive Chunk."
        )

        positive_chunk_indexes_by_seed[seed_id] = positive_indexes

    e5_candidate_matrix = np.vstack(
        [
            e5_instructional_embeddings["passage_vectors_by_text_hash"][
                passage_hash_by_text[text]
            ]
            for text in candidate_texts
        ]
    )

    lexical_vectorizer = TfidfVectorizer(
        tokenizer=persian_lexical_tokens,
        token_pattern=None,
        ngram_range=(1, 2),
        lowercase=False,
        norm="l2",
        sublinear_tf=True,
        dtype=np.float32,
    )

    started_at = time.perf_counter()

    lexical_candidate_matrix = lexical_vectorizer.fit_transform(candidate_texts)
    lexical_query_matrix = lexical_vectorizer.transform(hybrid_query_texts)

    e5_scores = hybrid_e5_query_matrix @ e5_candidate_matrix.T
    lexical_scores = (
        lexical_query_matrix @ lexical_candidate_matrix.T
    ).toarray()

    e5_ranked_indexes = np.argsort(-e5_scores, axis=1)
    lexical_ranked_indexes = np.argsort(-lexical_scores, axis=1)

    e5_rank_positions = np.empty_like(e5_ranked_indexes)
    lexical_rank_positions = np.empty_like(lexical_ranked_indexes)

    row_indexes = np.arange(len(hybrid_query_records))[:, None]

    e5_rank_positions[row_indexes, e5_ranked_indexes] = (
        np.arange(len(candidate_records))[None, :] + 1
    )
    lexical_rank_positions[row_indexes, lexical_ranked_indexes] = (
        np.arange(len(candidate_records))[None, :] + 1
    )

    rrf_scores = (
        1.0 / (rrf_constant + e5_rank_positions)
        + 1.0 / (rrf_constant + lexical_rank_positions)
    )
    rrf_ranked_indexes = np.argsort(-rrf_scores, axis=1)

    elapsed_seconds = time.perf_counter() - started_at

    e5_metrics = calculate_retrieval_metrics(
        e5_ranked_indexes,
        hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )
    lexical_metrics = calculate_retrieval_metrics(
        lexical_ranked_indexes,
        hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )
    rrf_metrics = calculate_retrieval_metrics(
        rrf_ranked_indexes,
        hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )

    rrf_metrics["seconds"] = round(elapsed_seconds, 3)
    rrf_metrics["candidate_chunk_count"] = len(candidate_records)

    hybrid_retrieval_results[candidate_name] = {
        "e5_instructional": e5_metrics,
        "persian_tfidf_lexical": lexical_metrics,
        "rrf_e5_plus_lexical": rrf_metrics,
    }

    print(f"\nE5 + lexical RRF × {candidate_name}")
    print(f"- E5 Recall@10: {e5_metrics['recall_at_10']:.4f}")
    print(f"- Lexical Recall@10: {lexical_metrics['recall_at_10']:.4f}")
    print(f"- Hybrid Recall@10: {rrf_metrics['recall_at_10']:.4f}")
    print(f"- Hybrid MRR: {rrf_metrics['mrr']:.4f}")
    print(f"- Seconds: {rrf_metrics['seconds']}")

print("\nHYBRID E5 + PERSIAN LEXICAL RETRIEVAL BENCHMARK: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## آماده‌سازی Hybrid Retrieval با Arvan و بازیابی واژگانی

مدل Arvan در بازیابی متن شواهد عملکرد خوبی نشان داد، اما در بازیابی مستقیم پرسش ضعیف‌تر بود.

در این مرحله ابتدا محل دقیق embeddingهای Arvan موجود در حافظهٔ Notebook را پیدا می‌کنیم تا در مرحلهٔ بعد، بدون اجرای دوبارهٔ API و بدون هزینهٔ جدید، آن را با بازیابی واژگانی ترکیب کنیم.

</div>

In [ ]:
# Code Cell 61
# NB13 — Arvan vector-cache discovery for hybrid retrieval

arvan_vector_lookup_candidates = []

for global_name, global_value in list(globals().items()):
    if not isinstance(global_value, dict):
        continue

    if {
        "query_vectors_by_text_hash",
        "passage_vectors_by_text_hash",
    }.issubset(global_value.keys()):
        arvan_vector_lookup_candidates.append(
            {
                "container_name": global_name,
                "lookup_name": None,
                "lookup": global_value,
            }
        )

    for nested_name, nested_value in global_value.items():
        if not isinstance(nested_value, dict):
            continue

        if {
            "query_vectors_by_text_hash",
            "passage_vectors_by_text_hash",
        }.issubset(nested_value.keys()):
            arvan_vector_lookup_candidates.append(
                {
                    "container_name": global_name,
                    "lookup_name": nested_name,
                    "lookup": nested_value,
                }
            )

print("=== ARVAN VECTOR-CACHE DISCOVERY ===")
print("Vector-cache candidates found:", len(arvan_vector_lookup_candidates))

for candidate in arvan_vector_lookup_candidates:
    lookup = candidate["lookup"]
    model_identifier = (
        lookup.get("model_id")
        or lookup.get("provider")
        or candidate["lookup_name"]
        or candidate["container_name"]
    )

    print("\nContainer:", candidate["container_name"])
    print("Nested lookup:", candidate["lookup_name"])
    print("Model identifier:", model_identifier)
    print(
        "Query-vector count:",
        len(lookup["query_vectors_by_text_hash"]),
    )
    print(
        "Passage-vector count:",
        len(lookup["passage_vectors_by_text_hash"]),
    )
    print(
        "Dimension:",
        lookup.get("dimension", lookup.get("embedding_dimension")),
    )

arvan_matches = [
    candidate
    for candidate in arvan_vector_lookup_candidates
    if (
        "arvan" in str(candidate["lookup_name"]).lower()
        or "arvan" in str(candidate["container_name"]).lower()
        or "text-embedding-3-large"
        in str(candidate["lookup"].get("model_id", "")).lower()
        or candidate["lookup"].get("embedding_dimension") == 3072
        or candidate["lookup"].get("dimension") == 3072
    )
]

assert len(arvan_matches) == 1, (
    "Expected exactly one Arvan vector cache. Found: "
    f"{len(arvan_matches)}."
)

arvan_embedding_lookup = arvan_matches[0]["lookup"]

assert len(arvan_embedding_lookup["query_vectors_by_text_hash"]) == 573
assert len(arvan_embedding_lookup["passage_vectors_by_text_hash"]) == 7698

print("\nResolved Arvan model:", arvan_embedding_lookup.get("model_id"))
print("\nARVAN VECTOR-CACHE DISCOVERY: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی Hybrid Retrieval با Arvan و بازیابی واژگانی

در این آزمایش، رتبه‌های مدل Arvan و بازیابی واژگانی فارسی با روش RRF ترکیب می‌شوند.

هدف این است که بررسی کنیم آیا مدل معنایی Arvan و تطبیق واژگانی، پرسش‌های Golden را مکمل یکدیگر بازیابی می‌کنند یا خیر.

</div>

In [ ]:
# Code Cell 62
# NB13 — Hybrid Arvan + Persian lexical retrieval benchmark with RRF

arvan_hybrid_query_records = [
    record
    for record in benchmark_query_records
    if record["source_seed_id"] in common_eligible_seed_ids
]

assert len(arvan_hybrid_query_records) == 553

assert all(
    record["question"] in query_hash_by_text
    for record in arvan_hybrid_query_records
)

arvan_query_matrix = np.vstack(
    [
        arvan_embedding_lookup["query_vectors_by_text_hash"][
            query_hash_by_text[record["question"]]
        ]
        for record in arvan_hybrid_query_records
    ]
)

arvan_hybrid_query_texts = [
    record["question"]
    for record in arvan_hybrid_query_records
]

arvan_lexical_hybrid_results = {}
rrf_constant = 60

print("=== HYBRID ARVAN + PERSIAN LEXICAL RETRIEVAL BENCHMARK ===")
print("Eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(arvan_hybrid_query_records))
print("Fusion method: Reciprocal Rank Fusion")
print("RRF constant:", rrf_constant)

for candidate_name, seed_registry in retrieval_evaluation_registry.items():
    candidate_path = project_root / retrieval_chunking_candidates[candidate_name]["path"]

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    candidate_chunk_ids = [
        record["chunk_id"]
        for record in candidate_records
    ]
    candidate_texts = [
        canonical_exact_text(record["content"])
        for record in candidate_records
    ]

    positive_chunk_indexes_by_seed = {}

    for seed_id in common_eligible_seed_ids:
        positive_ids = set(seed_registry[seed_id]["positive_chunk_ids"])

        positive_indexes = {
            index
            for index, chunk_id in enumerate(candidate_chunk_ids)
            if chunk_id in positive_ids
        }

        assert positive_indexes, (
            f"{candidate_name}, seed {seed_id} has no indexed positive Chunk."
        )

        positive_chunk_indexes_by_seed[seed_id] = positive_indexes

    arvan_candidate_matrix = np.vstack(
        [
            arvan_embedding_lookup["passage_vectors_by_text_hash"][
                passage_hash_by_text[text]
            ]
            for text in candidate_texts
        ]
    )

    lexical_vectorizer = TfidfVectorizer(
        tokenizer=persian_lexical_tokens,
        token_pattern=None,
        ngram_range=(1, 2),
        lowercase=False,
        norm="l2",
        sublinear_tf=True,
        dtype=np.float32,
    )

    started_at = time.perf_counter()

    lexical_candidate_matrix = lexical_vectorizer.fit_transform(candidate_texts)
    lexical_query_matrix = lexical_vectorizer.transform(
        arvan_hybrid_query_texts
    )

    arvan_scores = arvan_query_matrix @ arvan_candidate_matrix.T
    lexical_scores = (
        lexical_query_matrix @ lexical_candidate_matrix.T
    ).toarray()

    arvan_ranked_indexes = np.argsort(-arvan_scores, axis=1)
    lexical_ranked_indexes = np.argsort(-lexical_scores, axis=1)

    arvan_rank_positions = np.empty_like(arvan_ranked_indexes)
    lexical_rank_positions = np.empty_like(lexical_ranked_indexes)

    row_indexes = np.arange(len(arvan_hybrid_query_records))[:, None]
    candidate_rank_positions = (
        np.arange(len(candidate_records))[None, :] + 1
    )

    arvan_rank_positions[row_indexes, arvan_ranked_indexes] = (
        candidate_rank_positions
    )
    lexical_rank_positions[row_indexes, lexical_ranked_indexes] = (
        candidate_rank_positions
    )

    rrf_scores = (
        1.0 / (rrf_constant + arvan_rank_positions)
        + 1.0 / (rrf_constant + lexical_rank_positions)
    )
    rrf_ranked_indexes = np.argsort(-rrf_scores, axis=1)

    elapsed_seconds = time.perf_counter() - started_at

    arvan_metrics = calculate_retrieval_metrics(
        arvan_ranked_indexes,
        arvan_hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )
    lexical_metrics = calculate_retrieval_metrics(
        lexical_ranked_indexes,
        arvan_hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )
    rrf_metrics = calculate_retrieval_metrics(
        rrf_ranked_indexes,
        arvan_hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )

    rrf_metrics["seconds"] = round(elapsed_seconds, 3)
    rrf_metrics["candidate_chunk_count"] = len(candidate_records)

    arvan_lexical_hybrid_results[candidate_name] = {
        "arvan": arvan_metrics,
        "persian_tfidf_lexical": lexical_metrics,
        "rrf_arvan_plus_lexical": rrf_metrics,
    }

    print(f"\nArvan + lexical RRF × {candidate_name}")
    print(f"- Arvan Recall@10: {arvan_metrics['recall_at_10']:.4f}")
    print(f"- Lexical Recall@10: {lexical_metrics['recall_at_10']:.4f}")
    print(f"- Hybrid Recall@10: {rrf_metrics['recall_at_10']:.4f}")
    print(f"- Hybrid MRR: {rrf_metrics['mrr']:.4f}")
    print(f"- Seconds: {rrf_metrics['seconds']}")

print("\nHYBRID ARVAN + PERSIAN LEXICAL RETRIEVAL BENCHMARK: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## تحلیل شکست بازیابی در بهترین ترکیب فعلی

بهترین ترکیب فعلی، Arvan به‌همراه بازیابی واژگانی و Chunking والد–فرزند است.

در این مرحله عملکرد این ترکیب برای هر source seed جداگانه بررسی می‌شود تا مشخص شود کدام گروه از پرسش‌ها بازیابی می‌شوند، کدام گروه شکست می‌خورند و اصلاح بعدی باید روی چه بخش‌هایی متمرکز باشد.

</div>

In [ ]:
# Code Cell 63
# NB13 — Per-seed failure profile for best current hybrid:
# Arvan + Persian lexical RRF × parent_child_children

best_candidate_name = "parent_child_children"
best_seed_registry = retrieval_evaluation_registry[best_candidate_name]
best_candidate_path = (
    project_root
    / retrieval_chunking_candidates[best_candidate_name]["path"]
)

best_candidate_records = [
    json.loads(line)
    for line in best_candidate_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

best_candidate_chunk_ids = [
    record["chunk_id"]
    for record in best_candidate_records
]
best_candidate_texts = [
    canonical_exact_text(record["content"])
    for record in best_candidate_records
]

best_positive_indexes_by_seed = {}

for seed_id in common_eligible_seed_ids:
    positive_ids = set(best_seed_registry[seed_id]["positive_chunk_ids"])

    positive_indexes = {
        index
        for index, chunk_id in enumerate(best_candidate_chunk_ids)
        if chunk_id in positive_ids
    }

    assert positive_indexes, (
        f"Seed {seed_id} has no indexed positive Chunk "
        f"for {best_candidate_name}."
    )

    best_positive_indexes_by_seed[seed_id] = positive_indexes

best_arvan_query_matrix = np.vstack(
    [
        arvan_embedding_lookup["query_vectors_by_text_hash"][
            query_hash_by_text[record["question"]]
        ]
        for record in arvan_hybrid_query_records
    ]
)

best_arvan_candidate_matrix = np.vstack(
    [
        arvan_embedding_lookup["passage_vectors_by_text_hash"][
            passage_hash_by_text[text]
        ]
        for text in best_candidate_texts
    ]
)

best_lexical_vectorizer = TfidfVectorizer(
    tokenizer=persian_lexical_tokens,
    token_pattern=None,
    ngram_range=(1, 2),
    lowercase=False,
    norm="l2",
    sublinear_tf=True,
    dtype=np.float32,
)

best_lexical_candidate_matrix = best_lexical_vectorizer.fit_transform(
    best_candidate_texts
)
best_lexical_query_matrix = best_lexical_vectorizer.transform(
    [
        record["question"]
        for record in arvan_hybrid_query_records
    ]
)

best_arvan_scores = (
    best_arvan_query_matrix @ best_arvan_candidate_matrix.T
)
best_lexical_scores = (
    best_lexical_query_matrix @ best_lexical_candidate_matrix.T
).toarray()

best_arvan_ranked_indexes = np.argsort(-best_arvan_scores, axis=1)
best_lexical_ranked_indexes = np.argsort(-best_lexical_scores, axis=1)

best_arvan_rank_positions = np.empty_like(best_arvan_ranked_indexes)
best_lexical_rank_positions = np.empty_like(best_lexical_ranked_indexes)

best_row_indexes = np.arange(
    len(arvan_hybrid_query_records)
)[:, None]
best_candidate_rank_positions = (
    np.arange(len(best_candidate_records))[None, :] + 1
)

best_arvan_rank_positions[
    best_row_indexes,
    best_arvan_ranked_indexes,
] = best_candidate_rank_positions

best_lexical_rank_positions[
    best_row_indexes,
    best_lexical_ranked_indexes,
] = best_candidate_rank_positions

best_rrf_scores = (
    1.0 / (60 + best_arvan_rank_positions)
    + 1.0 / (60 + best_lexical_rank_positions)
)
best_rrf_ranked_indexes = np.argsort(-best_rrf_scores, axis=1)

seed_failure_profile = {}

for query_index, query_record in enumerate(arvan_hybrid_query_records):
    seed_id = query_record["source_seed_id"]
    positive_indexes = best_positive_indexes_by_seed[seed_id]

    positive_rank_positions = np.flatnonzero(
        np.isin(
            best_rrf_ranked_indexes[query_index],
            list(positive_indexes),
        )
    )
    first_rank = int(positive_rank_positions[0]) + 1

    seed_failure_profile.setdefault(
        seed_id,
        {
            "seed_id": seed_id,
            "query_count": 0,
            "first_positive_ranks": [],
            "example_hit_questions": [],
            "example_miss_questions": [],
        },
    )

    seed_profile = seed_failure_profile[seed_id]
    seed_profile["query_count"] += 1
    seed_profile["first_positive_ranks"].append(first_rank)

    if first_rank <= 10 and len(seed_profile["example_hit_questions"]) < 2:
        seed_profile["example_hit_questions"].append(
            {
                "candidate_id": query_record["candidate_id"],
                "question": query_record["question"],
                "first_positive_rank": first_rank,
            }
        )

    if first_rank > 10 and len(seed_profile["example_miss_questions"]) < 2:
        seed_profile["example_miss_questions"].append(
            {
                "candidate_id": query_record["candidate_id"],
                "question": query_record["question"],
                "first_positive_rank": first_rank,
            }
        )

for seed_profile in seed_failure_profile.values():
    ranks = seed_profile["first_positive_ranks"]

    seed_profile["recall_at_1"] = float(
        np.mean([rank <= 1 for rank in ranks])
    )
    seed_profile["recall_at_3"] = float(
        np.mean([rank <= 3 for rank in ranks])
    )
    seed_profile["recall_at_5"] = float(
        np.mean([rank <= 5 for rank in ranks])
    )
    seed_profile["recall_at_10"] = float(
        np.mean([rank <= 10 for rank in ranks])
    )
    seed_profile["mrr"] = float(np.mean([1 / rank for rank in ranks]))
    seed_profile["mean_first_positive_rank"] = float(np.mean(ranks))
    seed_profile["miss_count_at_10"] = int(
        np.sum([rank > 10 for rank in ranks])
    )

seed_failure_profile_rows = sorted(
    seed_failure_profile.values(),
    key=lambda item: (
        item["recall_at_10"],
        item["mrr"],
        item["seed_id"],
    ),
)

seed_failure_profile_report_path = (
    active_benchmark_run_directory
    / "arvan_lexical_parent_child_seed_failure_profile.json"
)

seed_failure_profile_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "model": "arvan_text_embedding_3_large_plus_persian_tfidf",
            "fusion_method": "RRF",
            "rrf_constant": 60,
            "candidate": best_candidate_name,
            "query_count": len(arvan_hybrid_query_records),
            "seed_profiles": seed_failure_profile_rows,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== BEST HYBRID PER-SEED FAILURE PROFILE ===")
print("Configuration: Arvan + lexical RRF × parent_child_children")
print("Seeds evaluated:", len(seed_failure_profile_rows))
print("Queries evaluated:", len(arvan_hybrid_query_records))

print("\nPer-seed results, weakest first:")
for profile in seed_failure_profile_rows:
    print(
        f"- seed {profile['seed_id']}: "
        f"Recall@10={profile['recall_at_10']:.4f}, "
        f"MRR={profile['mrr']:.4f}, "
        f"misses@10={profile['miss_count_at_10']}/{profile['query_count']}"
    )

print("\nSeeds with zero Recall@10:")
zero_recall_seed_ids = [
    profile["seed_id"]
    for profile in seed_failure_profile_rows
    if profile["recall_at_10"] == 0.0
]
print(zero_recall_seed_ids)

print("\nReport:", seed_failure_profile_report_path)
print("\nBEST HYBRID PER-SEED FAILURE PROFILE: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## بررسی نمونه‌های واقعی شکست بازیابی

در این مرحله، برای seedهایی که Recall@10 آن‌ها صفر است، یک پرسش ناموفق، شاهد طلایی، پاسخ مورد انتظار و اولین Chunk بازیابی‌شده کنار هم نمایش داده می‌شوند.

این بررسی فقط برای تشخیص علت شکست است. پاسخ مورد انتظار و شاهد طلایی در ساخت query یا رتبه‌بندی استفاده نمی‌شوند.

</div>

In [ ]:
# Code Cell 64
# NB13 — Human-readable inspection of zero-recall retrieval failures

active_golden_records = [
    json.loads(line)
    for line in golden_dataset_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

golden_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in active_golden_records
}

assert len(golden_record_by_candidate_id) == len(active_golden_records) == 573

zero_recall_seed_ids = [
    profile["seed_id"]
    for profile in seed_failure_profile_rows
    if profile["recall_at_10"] == 0.0
]

failure_examples = []

for seed_id in zero_recall_seed_ids:
    seed_query_indexes = [
        query_index
        for query_index, query_record in enumerate(arvan_hybrid_query_records)
        if query_record["source_seed_id"] == seed_id
    ]

    seed_query_rank_pairs = []

    for query_index in seed_query_indexes:
        positive_rank_positions = np.flatnonzero(
            np.isin(
                best_rrf_ranked_indexes[query_index],
                list(best_positive_indexes_by_seed[seed_id]),
            )
        )

        seed_query_rank_pairs.append(
            (
                int(positive_rank_positions[0]) + 1,
                query_index,
            )
        )

    best_failed_rank, selected_query_index = min(seed_query_rank_pairs)
    selected_query = arvan_hybrid_query_records[selected_query_index]
    selected_golden_record = golden_record_by_candidate_id[
        selected_query["candidate_id"]
    ]

    top_chunk_index = int(best_rrf_ranked_indexes[selected_query_index][0])
    top_chunk_record = best_candidate_records[top_chunk_index]

    failure_examples.append(
        {
            "seed_id": seed_id,
            "candidate_id": selected_query["candidate_id"],
            "first_positive_rank": best_failed_rank,
            "question": selected_golden_record["question"],
            "expected_answer": selected_golden_record["expected_answer"],
            "source_evidence": selected_golden_record["source_evidence"],
            "top_retrieved_chunk_id": top_chunk_record["chunk_id"],
            "top_retrieved_source_id": top_chunk_record["source_id"],
            "top_retrieved_content": top_chunk_record["content"],
        }
    )

failure_example_report_path = (
    active_benchmark_run_directory
    / "arvan_lexical_parent_child_zero_recall_examples.json"
)

failure_example_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "candidate": best_candidate_name,
            "zero_recall_seed_ids": zero_recall_seed_ids,
            "examples": failure_examples,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== ZERO-RECALL FAILURE EXAMPLES ===")
print("Configuration: Arvan + lexical RRF × parent_child_children")
print("Seeds inspected:", zero_recall_seed_ids)

for example in failure_examples:
    print("\n" + "=" * 80)
    print(
        f"SEED {example['seed_id']} | "
        f"best positive rank: {example['first_positive_rank']}"
    )
    print("\nQuestion:")
    print(example["question"])

    print("\nGolden evidence preview:")
    print(example["source_evidence"][:600])

    print("\nExpected-answer preview (diagnostic only):")
    print(example["expected_answer"][:500])

    print("\nTop retrieved Chunk:")
    print("Chunk ID:", example["top_retrieved_chunk_id"])
    print("Source ID:", example["top_retrieved_source_id"])
    print("Content preview:", example["top_retrieved_content"][:600])

print("\nReport:", failure_example_report_path)
print("\nZERO-RECALL FAILURE EXAMPLES: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## تحلیل اطلاعات متمایزکننده در پرسش‌های Golden Dataset

در این مرحله بررسی می‌کنیم هر پرسش چند واژه یا عبارت متمایزکننده دارد؛ یعنی واژه‌هایی که در کل Chunkهای کاندیدا کم‌تکرارترند و می‌توانند به تشخیص منبع درست کمک کنند.

این تحلیل تشخیصی است، Golden Dataset را تغییر نمی‌دهد و از پاسخ مورد انتظار برای بازیابی استفاده نمی‌کند.

</div>

In [ ]:
# Code Cell 65
# NB13 — Query information-profile diagnostic using corpus-derived IDF

active_golden_records = [
    json.loads(line)
    for line in golden_dataset_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

golden_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in active_golden_records
}

assert len(active_golden_records) == 573

# Rebuild ranks locally from Code Cell 63 outputs.
question_rank_by_candidate_id = {}

for query_index, query_record in enumerate(arvan_hybrid_query_records):
    seed_id = query_record["source_seed_id"]
    positive_indexes = best_positive_indexes_by_seed[seed_id]
    ranked_indexes = best_rrf_ranked_indexes[query_index]

    positive_positions = np.flatnonzero(
        np.isin(ranked_indexes, list(positive_indexes))
    )

    assert len(positive_positions) > 0, (
        "No positive Chunk rank was found for "
        f"candidate_id={query_record['candidate_id']}."
    )

    question_rank_by_candidate_id[query_record["candidate_id"]] = (
        int(positive_positions[0]) + 1
    )

assert len(question_rank_by_candidate_id) == len(arvan_hybrid_query_records) == 553

idf_values = best_lexical_vectorizer.idf_
high_idf_threshold = float(np.quantile(idf_values, 0.75))

information_profile_query_matrix = best_lexical_vectorizer.transform(
    [
        golden_record_by_candidate_id[candidate_id]["question"]
        for candidate_id in question_rank_by_candidate_id
    ]
)

information_profile_rows = []

for row_index, candidate_id in enumerate(question_rank_by_candidate_id):
    golden_record = golden_record_by_candidate_id[candidate_id]
    query_vector = information_profile_query_matrix[row_index]
    feature_indexes = query_vector.indices
    query_idf_values = idf_values[feature_indexes]

    high_idf_feature_count = int(
        np.sum(query_idf_values >= high_idf_threshold)
    )
    first_positive_rank = question_rank_by_candidate_id[candidate_id]

    information_profile_rows.append(
        {
            "candidate_id": candidate_id,
            "source_seed_id": golden_record["source_seed_id"],
            "question": golden_record["question"],
            "query_feature_count": int(len(feature_indexes)),
            "high_idf_feature_count": high_idf_feature_count,
            "maximum_query_idf": (
                float(np.max(query_idf_values))
                if len(query_idf_values)
                else 0.0
            ),
            "mean_query_idf": (
                float(np.mean(query_idf_values))
                if len(query_idf_values)
                else 0.0
            ),
            "information_status": (
                "NO_HIGH_IDF_SIGNAL"
                if high_idf_feature_count == 0
                else "HAS_HIGH_IDF_SIGNAL"
            ),
            "first_positive_rank": first_positive_rank,
            "hit_at_10": first_positive_rank <= 10,
        }
    )

assert len(information_profile_rows) == 553

information_summary = {}

for status in ("NO_HIGH_IDF_SIGNAL", "HAS_HIGH_IDF_SIGNAL"):
    rows = [
        row
        for row in information_profile_rows
        if row["information_status"] == status
    ]

    information_summary[status] = {
        "query_count": len(rows),
        "share_of_queries": len(rows) / len(information_profile_rows),
        "recall_at_10": (
            float(np.mean([row["hit_at_10"] for row in rows]))
            if rows
            else None
        ),
        "mean_first_positive_rank": (
            float(np.mean([row["first_positive_rank"] for row in rows]))
            if rows
            else None
        ),
    }

information_profile_report_path = (
    active_benchmark_run_directory
    / "active_golden_query_information_profile.json"
)

information_profile_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "candidate": best_candidate_name,
            "configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "idf_policy": {
                "source": "parent_child_children_tfidf_vocabulary",
                "high_idf_percentile": 0.75,
                "high_idf_threshold": high_idf_threshold,
            },
            "summary": information_summary,
            "rows": information_profile_rows,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== ACTIVE GOLDEN QUERY INFORMATION PROFILE ===")
print("Queries analysed:", len(information_profile_rows))
print(f"High-IDF threshold: {high_idf_threshold:.4f}")

for status, summary in information_summary.items():
    print(f"\n{status}")
    print("- Query count:", summary["query_count"])
    print(f"- Share: {summary['share_of_queries']:.4f}")

    if summary["recall_at_10"] is None:
        print("- Recall@10: N/A (no queries in this group)")
        print("- Mean first-positive rank: N/A (no queries in this group)")
    else:
        print(f"- Recall@10: {summary['recall_at_10']:.4f}")
        print(
            "- Mean first-positive rank: "
            f"{summary['mean_first_positive_rank']:.2f}"
        )

weak_information_examples = sorted(
    information_profile_rows,
    key=lambda row: (
        row["high_idf_feature_count"],
        row["maximum_query_idf"],
        row["first_positive_rank"],
    ),
)

print("\nLowest-information example questions, first 12:")
for example in weak_information_examples[:12]:
    print(
        f"- seed {example['source_seed_id']} | "
        f"rank={example['first_positive_rank']} | "
        f"high_idf_features={example['high_idf_feature_count']} | "
        f"question={example['question']}"
    )

print("\nReport:", information_profile_report_path)
print("\nACTIVE GOLDEN QUERY INFORMATION PROFILE: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی سیاست بودجهٔ Context در روش Parent–Child

در Parent–Child، ابتدا Chunkهای کوچک بازیابی می‌شوند و سپس متن والد آن‌ها برای تولید پاسخ استفاده می‌شود.

در این مرحله چند سیاست متفاوت برای تعداد Childهای بازیابی‌شده، تعداد والدها و سقف tokenهای Context بررسی می‌شود تا هزینه و حجم Context قابل‌کنترل باشد، بدون آنکه اتصال Child به Parent از بین برود.

</div>

In [ ]:
# Code Cell 66
# NB13 — Parent-child context-budget policy evaluation

assert "parent_child_parents" in chunking_candidates, (
    "The parent_child_parents candidate is unavailable."
)

parent_context_path = (
    project_root
    / chunking_candidates["parent_child_parents"]["path"]
)

assert parent_context_path.is_file(), (
    f"Missing parent-context file: {parent_context_path}"
)

parent_context_records = [
    json.loads(line)
    for line in parent_context_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

parent_record_by_chunk_id = {
    record["chunk_id"]: record
    for record in parent_context_records
}

assert len(parent_record_by_chunk_id) == len(parent_context_records)

context_budget_policies = [
    {
        "policy_id": "conservative",
        "retrieved_child_k": 5,
        "max_parent_chunks": 3,
        "max_context_tokens": 3000,
    },
    {
        "policy_id": "balanced",
        "retrieved_child_k": 10,
        "max_parent_chunks": 4,
        "max_context_tokens": 5000,
    },
    {
        "policy_id": "expansive",
        "retrieved_child_k": 10,
        "max_parent_chunks": 6,
        "max_context_tokens": 8000,
    },
]

context_budget_results = {}

print("=== PARENT–CHILD CONTEXT-BUDGET POLICY EVALUATION ===")
print("Queries evaluated:", len(arvan_hybrid_query_records))
print("Parent chunks available:", len(parent_context_records))

for policy in context_budget_policies:
    assembled_contexts = []
    missing_parent_references = 0

    for query_index, _ in enumerate(arvan_hybrid_query_records):
        ranked_child_indexes = best_rrf_ranked_indexes[query_index]
        selected_child_indexes = ranked_child_indexes[
            :policy["retrieved_child_k"]
        ]

        selected_parent_ids = []
        selected_parent_records = []
        current_token_total = 0

        for child_index in selected_child_indexes:
            child_record = best_candidate_records[int(child_index)]
            parent_chunk_id = child_record["parent_chunk_id"]

            if parent_chunk_id not in parent_record_by_chunk_id:
                missing_parent_references += 1
                continue

            if parent_chunk_id in selected_parent_ids:
                continue

            parent_record = parent_record_by_chunk_id[parent_chunk_id]
            parent_token_count = parent_record["token_count"]

            if len(selected_parent_records) >= policy["max_parent_chunks"]:
                continue

            if (
                current_token_total + parent_token_count
                > policy["max_context_tokens"]
            ):
                continue

            selected_parent_ids.append(parent_chunk_id)
            selected_parent_records.append(parent_record)
            current_token_total += parent_token_count

        assembled_contexts.append(
            {
                "parent_chunk_count": len(selected_parent_records),
                "token_count": current_token_total,
                "source_count": len(
                    {
                        parent_record["source_id"]
                        for parent_record in selected_parent_records
                    }
                ),
                "selected_parent_chunk_ids": selected_parent_ids,
            }
        )

    token_counts = np.array(
        [context["token_count"] for context in assembled_contexts]
    )
    parent_chunk_counts = np.array(
        [context["parent_chunk_count"] for context in assembled_contexts]
    )
    source_counts = np.array(
        [context["source_count"] for context in assembled_contexts]
    )

    result = {
        **policy,
        "query_count": len(assembled_contexts),
        "missing_parent_references": missing_parent_references,
        "empty_context_count": int(np.sum(token_counts == 0)),
        "mean_context_tokens": float(np.mean(token_counts)),
        "p95_context_tokens": float(np.percentile(token_counts, 95)),
        "maximum_context_tokens": int(np.max(token_counts)),
        "mean_parent_chunk_count": float(np.mean(parent_chunk_counts)),
        "mean_unique_source_count": float(np.mean(source_counts)),
    }

    context_budget_results[policy["policy_id"]] = result

    print(f"\nPolicy: {policy['policy_id']}")
    print("- Retrieved children:", policy["retrieved_child_k"])
    print("- Maximum parent chunks:", policy["max_parent_chunks"])
    print("- Maximum context tokens:", policy["max_context_tokens"])
    print(f"- Mean context tokens: {result['mean_context_tokens']:.1f}")
    print(f"- P95 context tokens: {result['p95_context_tokens']:.1f}")
    print(f"- Maximum actual tokens: {result['maximum_context_tokens']}")
    print(
        "- Mean parent chunks: "
        f"{result['mean_parent_chunk_count']:.2f}"
    )
    print(
        "- Mean unique sources: "
        f"{result['mean_unique_source_count']:.2f}"
    )
    print("- Empty contexts:", result["empty_context_count"])
    print(
        "- Missing parent references: "
        f"{result['missing_parent_references']}"
    )

assert all(
    result["missing_parent_references"] == 0
    for result in context_budget_results.values()
), "A selected Child referenced a missing Parent."

context_budget_report_path = (
    active_benchmark_run_directory
    / "parent_child_context_budget_policy_evaluation.json"
)

context_budget_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "retrieval_configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "child_candidate": "parent_child_children",
            "parent_candidate": "parent_child_parents",
            "results": context_budget_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nReport:", context_budget_report_path)
print("\nPARENT–CHILD CONTEXT-BUDGET POLICY EVALUATION: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## سنجش حفظ شواهد مثبت در بودجهٔ Context

برای هر سیاست Parent–Child بررسی می‌کنیم:

- آیا Child مثبت در نتایج بازیابی‌شده وجود داشته است؟
- اگر وجود داشته، آیا Parent متناظر آن واقعاً در Context نهایی باقی مانده است؟

این سنجش نشان می‌دهد محدودیت token و تعداد والدها چه مقدار از شواهد بازیابی‌شده را حفظ یا حذف می‌کند.

</div>

In [ ]:
# Code Cell 67
# NB13 — Positive-evidence retention under parent-child context budgets

context_budget_evidence_retention = {}

print("=== PARENT–CHILD POSITIVE-EVIDENCE RETENTION ===")
print("Queries evaluated:", len(arvan_hybrid_query_records))

for policy in context_budget_policies:
    retrieved_positive_query_count = 0
    retained_positive_parent_query_count = 0
    missing_parent_references = 0

    for query_index, query_record in enumerate(arvan_hybrid_query_records):
        seed_id = query_record["source_seed_id"]
        positive_child_indexes = best_positive_indexes_by_seed[seed_id]

        selected_child_indexes = best_rrf_ranked_indexes[query_index][
            :policy["retrieved_child_k"]
        ]

        positive_child_was_retrieved = any(
            int(child_index) in positive_child_indexes
            for child_index in selected_child_indexes
        )

        if positive_child_was_retrieved:
            retrieved_positive_query_count += 1

        positive_parent_ids = {
            best_candidate_records[child_index]["parent_chunk_id"]
            for child_index in positive_child_indexes
        }

        selected_parent_ids = []
        selected_parent_token_total = 0

        for child_index in selected_child_indexes:
            child_record = best_candidate_records[int(child_index)]
            parent_chunk_id = child_record["parent_chunk_id"]

            if parent_chunk_id not in parent_record_by_chunk_id:
                missing_parent_references += 1
                continue

            if parent_chunk_id in selected_parent_ids:
                continue

            parent_record = parent_record_by_chunk_id[parent_chunk_id]
            parent_token_count = parent_record["token_count"]

            if len(selected_parent_ids) >= policy["max_parent_chunks"]:
                continue

            if (
                selected_parent_token_total + parent_token_count
                > policy["max_context_tokens"]
            ):
                continue

            selected_parent_ids.append(parent_chunk_id)
            selected_parent_token_total += parent_token_count

        if positive_parent_ids.intersection(selected_parent_ids):
            retained_positive_parent_query_count += 1

    retrieval_rate = (
        retrieved_positive_query_count
        / len(arvan_hybrid_query_records)
    )
    parent_retention_rate = (
        retained_positive_parent_query_count
        / len(arvan_hybrid_query_records)
    )
    conditional_retention_rate = (
        retained_positive_parent_query_count
        / retrieved_positive_query_count
        if retrieved_positive_query_count
        else None
    )

    result = {
        **policy,
        "query_count": len(arvan_hybrid_query_records),
        "retrieved_positive_query_count": retrieved_positive_query_count,
        "retrieval_recall_at_child_k": retrieval_rate,
        "retained_positive_parent_query_count": (
            retained_positive_parent_query_count
        ),
        "parent_evidence_recall_after_budget": parent_retention_rate,
        "conditional_parent_retention_after_retrieval": (
            conditional_retention_rate
        ),
        "missing_parent_references": missing_parent_references,
    }

    context_budget_evidence_retention[policy["policy_id"]] = result

    print(f"\nPolicy: {policy['policy_id']}")
    print(
        "- Positive Child retrieved: "
        f"{retrieved_positive_query_count}/{len(arvan_hybrid_query_records)} "
        f"({retrieval_rate:.4f})"
    )
    print(
        "- Positive Parent retained in final Context: "
        f"{retained_positive_parent_query_count}/"
        f"{len(arvan_hybrid_query_records)} "
        f"({parent_retention_rate:.4f})"
    )
    print(
        "- Conditional retention after positive retrieval: "
        f"{conditional_retention_rate:.4f}"
    )
    print(
        "- Missing parent references: "
        f"{missing_parent_references}"
    )

assert all(
    result["missing_parent_references"] == 0
    for result in context_budget_evidence_retention.values()
), "A Child referenced a missing Parent."

context_budget_evidence_retention_report_path = (
    active_benchmark_run_directory
    / "parent_child_context_budget_evidence_retention.json"
)

context_budget_evidence_retention_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "retrieval_configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "candidate": best_candidate_name,
            "results": context_budget_evidence_retention,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nReport:", context_budget_evidence_retention_report_path)
print("\nPARENT–CHILD POSITIVE-EVIDENCE RETENTION: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ثبت پیکربندی موقت برای ادامهٔ ارزیابی

بهترین پیکربندی فعلی برای ادامهٔ آزمایش‌ها ثبت می‌شود، اما به دلیل پایین‌بودن Recall بازیابی پرسش و نبود انتخاب نهایی، برای استفادهٔ Production تأیید نمی‌شود.

این ثبت نسخه‌محور است و با تغییر Golden Dataset، در اجرای مجدد به‌صورت مستقل ارزیابی خواهد شد.

</div>

In [ ]:
# Code Cell 68
# NB13 — Provisional retrieval configuration registration

best_hybrid_metrics = arvan_lexical_hybrid_results[
    "parent_child_children"
]["rrf_arvan_plus_lexical"]

expansive_context_metrics = context_budget_evidence_retention["expansive"]
expansive_context_profile = context_budget_results["expansive"]

provisional_retrieval_configuration = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "status": "PROVISIONAL_FOR_FURTHER_EVALUATION_NOT_PRODUCTION",
    "retrieval": {
        "dense_model": "arvan_text_embedding_3_large",
        "lexical_model": "persian_tfidf_word_unigram_bigram",
        "fusion_method": "reciprocal_rank_fusion",
        "rrf_constant": 60,
        "retrieval_candidate": "parent_child_children",
        "query_recall_at_10": best_hybrid_metrics["recall_at_10"],
        "query_mrr": best_hybrid_metrics["mrr"],
    },
    "context_assembly": {
        "parent_candidate": "parent_child_parents",
        "policy_id": "expansive",
        "retrieved_child_k": expansive_context_profile[
            "retrieved_child_k"
        ],
        "max_parent_chunks": expansive_context_profile[
            "max_parent_chunks"
        ],
        "max_context_tokens": expansive_context_profile[
            "max_context_tokens"
        ],
        "p95_context_tokens": expansive_context_profile[
            "p95_context_tokens"
        ],
        "conditional_parent_retention_after_retrieval": (
            expansive_context_metrics[
                "conditional_parent_retention_after_retrieval"
            ]
        ),
    },
    "not_approved_for_production_because": [
        (
            "Question retrieval remains too weak: only "
            f"{best_hybrid_metrics['recall_at_10']:.4f} Recall@10."
        ),
        (
            "Context assembly cannot recover evidence that was not "
            "retrieved among the top Child results."
        ),
        (
            "Final model-and-chunking selection requires a completed "
            "query-side remediation evaluation."
        ),
    ],
    "required_next_work": [
        "query_reformulation_or_reranking_experiment",
        "post_remediation_retrieval_benchmark",
        "rerun_with_active_golden_dataset_after_feedback_integration",
        "final_tradeoff_decision",
    ],
}

provisional_retrieval_configuration_path = (
    active_benchmark_run_directory
    / "provisional_retrieval_configuration.json"
)

provisional_retrieval_configuration_path.write_text(
    json.dumps(
        provisional_retrieval_configuration,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== PROVISIONAL RETRIEVAL CONFIGURATION ===")
print(
    "Status:",
    provisional_retrieval_configuration["status"],
)
print(
    "Retrieval Recall@10:",
    f"{best_hybrid_metrics['recall_at_10']:.4f}",
)
print(
    "Retrieval MRR:",
    f"{best_hybrid_metrics['mrr']:.4f}",
)
print(
    "Context policy:",
    provisional_retrieval_configuration["context_assembly"]["policy_id"],
)
print(
    "Conditional parent retention:",
    f"{expansive_context_metrics['conditional_parent_retention_after_retrieval']:.4f}",
)
print("\nReport:", provisional_retrieval_configuration_path)
print("\nPROVISIONAL RETRIEVAL CONFIGURATION: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## کشف قابلیت Reranking

Reranker پس از بازیابی اولیه، پرسش و تعداد محدودی Chunk برتر را با هم بررسی می‌کند و می‌تواند ترتیب نتایج را دقیق‌تر کند.

در این مرحله فقط مدل‌های محلی و تنظیمات موجود بررسی می‌شوند. هیچ دانلود، نصب، فراخوانی API یا هزینه‌ای انجام نمی‌شود.

</div>

In [ ]:
# Code Cell 69
# NB13 — Read-only reranking capability discovery

import importlib.util
import os
from pathlib import Path

huggingface_cache_roots = [
    Path.home() / ".cache" / "huggingface" / "hub",
    Path.home() / "Library" / "Caches" / "huggingface" / "hub",
]

local_model_directories = []

for cache_root in huggingface_cache_roots:
    if not cache_root.is_dir():
        continue

    local_model_directories.extend(
        sorted(
            path
            for path in cache_root.glob("models--*")
            if path.is_dir()
        )
    )

reranker_name_indicators = (
    "reranker",
    "cross-encoder",
    "cross_encoder",
    "bge",
    "ms-marco",
)

local_reranker_directories = [
    model_directory
    for model_directory in local_model_directories
    if any(
        indicator in model_directory.name.lower()
        for indicator in reranker_name_indicators
    )
]

arvan_configuration_keys = sorted(
    environment_key
    for environment_key in os.environ
    if environment_key.startswith("ARVAN")
)

print("=== RERANKING CAPABILITY DISCOVERY ===")
print(
    "sentence_transformers available:",
    importlib.util.find_spec("sentence_transformers") is not None,
)
print(
    "CrossEncoder class available:",
    end=" ",
)

try:
    from sentence_transformers import CrossEncoder
    print(True)
except Exception:
    print(False)

print("\nLocal Hugging Face model directories:", len(local_model_directories))

print("\nLocal reranker-like model directories:")
if local_reranker_directories:
    for model_directory in local_reranker_directories:
        print("-", model_directory.name)
else:
    print("NONE FOUND")

print("\nConfigured Arvan environment-variable names:")
for environment_key in arvan_configuration_keys:
    print("-", environment_key)

known_reranking_environment_keys = [
    key
    for key in arvan_configuration_keys
    if any(
        indicator in key.lower()
        for indicator in (
            "rerank",
            "chat",
            "completion",
            "generation",
        )
    )
]

print("\nConfigured reranking-or-generation endpoint names:")
if known_reranking_environment_keys:
    for environment_key in known_reranking_environment_keys:
        print("-", environment_key)
else:
    print("NONE FOUND")

print("\nRERANKING CAPABILITY DISCOVERY: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## بررسی آماده‌بودن سرویس گفت‌وگویی Arvan برای Reranking

مدل گفت‌وگویی می‌تواند در مرحلهٔ بعد برای بازنویسی کنترل‌شدهٔ پرسش یا رتبه‌بندی دوبارهٔ تعداد محدودی Chunk برتر آزمایش شود.

در این مرحله فقط وجود تنظیمات و سازگاری اتصال بررسی می‌شود. هیچ درخواست API و هیچ هزینه‌ای ایجاد نمی‌شود.

</div>

In [ ]:
# Code Cell 70
# NB13 — Arvan chat configuration preflight for reranking experiments

from urllib.parse import urlparse
from dotenv import load_dotenv
import importlib.util
import os

arvan_env_path = project_root / ".env"

assert arvan_env_path.is_file(), (
    f"Arvan environment file is missing: {arvan_env_path}"
)

load_dotenv(arvan_env_path, override=False)

arvan_chat_api_key = os.getenv("ARVAN_AI_API_KEY")
arvan_chat_url = os.getenv("ARVAN_CHAT_URL")
arvan_chat_model = os.getenv("ARVAN_CHAT_MODEL")

assert arvan_chat_api_key, "ARVAN_AI_API_KEY is not configured."
assert arvan_chat_url, "ARVAN_CHAT_URL is not configured."
assert arvan_chat_model, "ARVAN_CHAT_MODEL is not configured."

parsed_chat_url = urlparse(arvan_chat_url)

assert parsed_chat_url.scheme == "https", (
    "Arvan chat endpoint must use HTTPS."
)
assert parsed_chat_url.netloc, (
    "Arvan chat endpoint has no valid host."
)
assert importlib.util.find_spec("openai") is not None, (
    "The OpenAI-compatible client package is unavailable."
)

arvan_chat_configuration = {
    "model": arvan_chat_model,
    "endpoint_scheme": parsed_chat_url.scheme,
    "endpoint_host": parsed_chat_url.netloc,
    "endpoint_path_configured": bool(parsed_chat_url.path),
    "api_key_available": True,
}

print("=== ARVAN CHAT CONFIGURATION PREFLIGHT ===")
print("OpenAI-compatible client available: True")
print("Arvan API key available: True")
print("Chat model:", arvan_chat_configuration["model"])
print("Gateway scheme:", arvan_chat_configuration["endpoint_scheme"])
print("Gateway host:", arvan_chat_configuration["endpoint_host"])
print(
    "Gateway path configured:",
    arvan_chat_configuration["endpoint_path_configured"],
)
print("\nARVAN CHAT CONFIGURATION PREFLIGHT: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## اعتبارسنجی اتصال واقعی مدل گفت‌وگویی Arvan

در این مرحله با یک درخواست کوتاه و بدون دادهٔ پروژه، اتصال واقعی به مدل گفت‌وگویی Arvan بررسی می‌شود.

این آزمایش فقط برای اطمینان از آماده‌بودن زیرساخت Query Reformulation و Reranking است.

</div>

In [ ]:
# Code Cell 71
# NB13 — Arvan chat live connection validation (corrected)

from openai import OpenAI
from datetime import datetime, timezone

arvan_chat_client = OpenAI(
    api_key=arvan_chat_api_key,
    base_url=arvan_chat_url,
)

live_chat_started_at = datetime.now(timezone.utc)

live_chat_response = arvan_chat_client.chat.completions.create(
    model=arvan_chat_model,
    messages=[
        {
            "role": "user",
            "content": (
                "Return exactly one word: READY. "
                "Do not explain your answer."
            ),
        }
    ],
    temperature=0,
    max_tokens=128,
)

assert live_chat_response.choices, (
    "Arvan chat returned no completion choices."
)

live_chat_choice = live_chat_response.choices[0]
live_chat_message = (
    live_chat_choice.message.content or ""
).strip()

arvan_chat_connection_report = {
    "checked_at_utc": live_chat_started_at.isoformat(),
    "model": arvan_chat_model,
    "endpoint_host": parsed_chat_url.netloc,
    "response_received": True,
    "finish_reason": live_chat_choice.finish_reason,
    "visible_text_received": bool(live_chat_message),
    "visible_text_character_count": len(live_chat_message),
}

arvan_chat_connection_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_connection_report.json"
)

arvan_chat_connection_report_path.write_text(
    json.dumps(
        arvan_chat_connection_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== ARVAN CHAT LIVE CONNECTION VALIDATION ===")
print("Model:", arvan_chat_model)
print("Response received:", True)
print("Finish reason:", live_chat_choice.finish_reason)
print("Visible text received:", bool(live_chat_message))
print("Response:", repr(live_chat_message))
print("Report:", arvan_chat_connection_report_path)
print("\nARVAN CHAT LIVE CONNECTION VALIDATION: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## بررسی ساختار پاسخ مدل گفت‌وگویی Arvan

در درخواست قبلی پاسخ از سرویس دریافت شد، اما متن در فیلد معمول `content` قرار نگرفت.

در این مرحله بدون ارسال درخواست جدید، ساختار همان پاسخ بررسی می‌شود تا روش درست خواندن خروجی مدل مشخص شود.

</div>

In [ ]:
# Code Cell 72
# NB13 — Read-only inspection of the existing Arvan chat response

assert "live_chat_response" in globals(), (
    "The previous live-chat response is unavailable. "
    "Run Code Cell 71 once before this cell."
)

assert live_chat_response.choices, (
    "Arvan chat returned no choices."
)

live_chat_choice = live_chat_response.choices[0]
live_chat_response_message = live_chat_choice.message

message_fields = [
    field_name
    for field_name in (
        "content",
        "reasoning_content",
        "refusal",
        "tool_calls",
        "function_call",
    )
    if hasattr(live_chat_response_message, field_name)
]

print("=== ARVAN CHAT RESPONSE STRUCTURE DISCOVERY ===")
print("Response model:", getattr(live_chat_response, "model", None))
print("Finish reason:", getattr(live_chat_choice, "finish_reason", None))
print("Message fields available:", message_fields)

for field_name in message_fields:
    field_value = getattr(live_chat_response_message, field_name)
    print(f"\n{field_name}:")
    print(repr(field_value))

print("\nUsage:")
print(getattr(live_chat_response, "usage", None))

print("\nARVAN CHAT RESPONSE STRUCTURE DISCOVERY: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## پایلوت Reranking با مدل گفت‌وگویی Arvan

در این آزمایش محدود، مدل گفت‌وگویی فقط پرسش و ده Chunk برتر بازیابی‌شده را می‌بیند و آن‌ها را دوباره رتبه‌بندی می‌کند.

پاسخ مورد انتظار، شواهد طلایی و اطلاعات label به مدل داده نمی‌شود. از آن‌ها فقط پس از پاسخ مدل، برای اندازه‌گیری کیفیت استفاده می‌شود.

هدف این است که قبل از هر اجرای بزرگ، بررسی کنیم آیا Reranking می‌تواند شواهدی را که در ده نتیجهٔ اول هستند به رتبه‌های بالاتر منتقل کند یا خیر.

</div>

In [ ]:
# Code Cell 73
# NB13 — Controlled Arvan-chat reranking pilot:
# 20 queries with at least one positive Child already in the top 10

import re
import time

reranker_top_k = 10
reranker_pilot_size = 20
reranker_excerpt_char_limit = 1000

eligible_reranker_query_indexes = []

for query_index, query_record in enumerate(arvan_hybrid_query_records):
    seed_id = query_record["source_seed_id"]
    positive_indexes = best_positive_indexes_by_seed[seed_id]
    top_k_indexes = best_rrf_ranked_indexes[query_index][:reranker_top_k]

    if any(int(index) in positive_indexes for index in top_k_indexes):
        eligible_reranker_query_indexes.append(query_index)

assert eligible_reranker_query_indexes, (
    "No query has a positive Child inside the current top-k retrieval set."
)

# One query per seed first, then fill remaining pilot positions deterministically.
selected_reranker_query_indexes = []
selected_seed_ids = set()

for query_index in eligible_reranker_query_indexes:
    seed_id = arvan_hybrid_query_records[query_index]["source_seed_id"]

    if seed_id not in selected_seed_ids:
        selected_reranker_query_indexes.append(query_index)
        selected_seed_ids.add(seed_id)

    if len(selected_reranker_query_indexes) == reranker_pilot_size:
        break

if len(selected_reranker_query_indexes) < reranker_pilot_size:
    for query_index in eligible_reranker_query_indexes:
        if query_index not in selected_reranker_query_indexes:
            selected_reranker_query_indexes.append(query_index)

        if len(selected_reranker_query_indexes) == reranker_pilot_size:
            break

assert len(selected_reranker_query_indexes) == min(
    reranker_pilot_size,
    len(eligible_reranker_query_indexes),
)

def extract_reranked_labels(response_text, allowed_labels):
    match = re.search(r"\{.*\}", response_text, flags=re.DOTALL)

    if not match:
        return None

    try:
        parsed_response = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

    proposed_labels = parsed_response.get("ranking")

    if not isinstance(proposed_labels, list):
        return None

    normalized_labels = [
        str(label).strip()
        for label in proposed_labels
        if str(label).strip() in allowed_labels
    ]

    deduplicated_labels = list(dict.fromkeys(normalized_labels))

    if set(deduplicated_labels) != set(allowed_labels):
        return None

    return deduplicated_labels

reranker_pilot_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_reranking_pilot.json"
)

if reranker_pilot_report_path.is_file():
    cached_reranker_pilot_report = json.loads(
        reranker_pilot_report_path.read_text(encoding="utf-8")
    )

    if (
        cached_reranker_pilot_report.get("active_golden_dataset_sha256")
        == golden_dataset_context["sha256"]
        and cached_reranker_pilot_report.get("model")
        == arvan_chat_model
    ):
        reranker_pilot_results = cached_reranker_pilot_report.get(
            "results",
            [],
        )
    else:
        reranker_pilot_results = []
else:
    reranker_pilot_results = []

cached_reranker_pilot_candidate_ids = {
    result["candidate_id"]
    for result in reranker_pilot_results
}

print("=== ARVAN CHAT RERANKING PILOT ===")
print("Top-k candidate passages:", reranker_top_k)
print("Eligible queries with a positive Child in top-k:", len(eligible_reranker_query_indexes))
print("Pilot queries:", len(selected_reranker_query_indexes))

for pilot_position, query_index in enumerate(
    selected_reranker_query_indexes,
    start=1,
):
    query_record = arvan_hybrid_query_records[query_index]

    if query_record["candidate_id"] in cached_reranker_pilot_candidate_ids:
        print(
            f"Pilot {pilot_position}/{len(selected_reranker_query_indexes)} | "
            f"cached | candidate_id={query_record['candidate_id']}"
        )
        continue

    seed_id = query_record["source_seed_id"]
    positive_indexes = best_positive_indexes_by_seed[seed_id]
    top_k_indexes = best_rrf_ranked_indexes[query_index][:reranker_top_k]

    passage_labels = [
        f"P{rank:02d}"
        for rank in range(1, reranker_top_k + 1)
    ]

    passage_by_label = {
        label: best_candidate_records[int(candidate_index)]
        for label, candidate_index in zip(passage_labels, top_k_indexes)
    }

    prompt_passages = "\n\n".join(
        (
            f"[{label}]\n"
            f"{passage_by_label[label]['content'][:reranker_excerpt_char_limit]}"
        )
        for label in passage_labels
    )

    reranker_prompt = f"""
وظیفه: فقط Passageهای زیر را برای پاسخ‌گویی به پرسش رتبه‌بندی کن.

قواعد:
- فقط بر اساس متن Passageها تصمیم بگیر.
- از دانش بیرونی استفاده نکن.
- همهٔ برچسب‌ها را دقیقاً یک‌بار در خروجی بیاور.
- خروجی فقط JSON معتبر باشد و دقیقاً این ساختار را داشته باشد:
{{"ranking":["P01","P02","P03","P04","P05","P06","P07","P08","P09","P10"]}}

پرسش:
{query_record["question"]}

Passageها:
{prompt_passages}
""".strip()

    started_at = time.perf_counter()

    reranker_response = arvan_chat_client.chat.completions.create(
        model=arvan_chat_model,
        messages=[
            {
                "role": "user",
                "content": reranker_prompt,
            }
        ],
        temperature=0,
        max_tokens=256,
    )

    elapsed_seconds = time.perf_counter() - started_at

    raw_response_text = (
        reranker_response.choices[0].message.content or ""
    ).strip()

    reranked_labels = extract_reranked_labels(
        raw_response_text,
        allowed_labels=set(passage_labels),
    )

    original_positive_rank = next(
        rank
        for rank, candidate_index in enumerate(top_k_indexes, start=1)
        if int(candidate_index) in positive_indexes
    )

    result = {
        "pilot_position": pilot_position,
        "candidate_id": query_record["candidate_id"],
        "source_seed_id": seed_id,
        "question": query_record["question"],
        "original_positive_rank": original_positive_rank,
        "response_parse_status": (
            "PASS"
            if reranked_labels is not None
            else "FAIL"
        ),
        "seconds": round(elapsed_seconds, 3),
        "raw_response": raw_response_text,
    }

    if reranked_labels is not None:
        reranked_candidate_indexes = [
            int(top_k_indexes[passage_labels.index(label)])
            for label in reranked_labels
        ]

        reranked_positive_rank = next(
            rank
            for rank, candidate_index in enumerate(
                reranked_candidate_indexes,
                start=1,
            )
            if candidate_index in positive_indexes
        )

        result["reranked_positive_rank"] = reranked_positive_rank
        result["rank_change"] = (
            original_positive_rank - reranked_positive_rank
        )

    reranker_pilot_results.append(result)

    print(
        f"Pilot {pilot_position}/{len(selected_reranker_query_indexes)} | "
        f"seed={seed_id} | "
        f"original_rank={original_positive_rank} | "
        f"parse={result['response_parse_status']}"
    )

parsed_results = [
    result
    for result in reranker_pilot_results
    if result["response_parse_status"] == "PASS"
]

reranker_pilot_summary = {
    "pilot_query_count": len(reranker_pilot_results),
    "parse_success_count": len(parsed_results),
    "parse_success_rate": (
        len(parsed_results) / len(reranker_pilot_results)
    ),
    "mean_original_positive_rank": (
        float(
            np.mean(
                [
                    result["original_positive_rank"]
                    for result in parsed_results
                ]
            )
        )
        if parsed_results
        else None
    ),
    "mean_reranked_positive_rank": (
        float(
            np.mean(
                [
                    result["reranked_positive_rank"]
                    for result in parsed_results
                ]
            )
        )
        if parsed_results
        else None
    ),
    "improved_query_count": (
        int(
            np.sum(
                [
                    result["rank_change"] > 0
                    for result in parsed_results
                ]
            )
        )
        if parsed_results
        else 0
    ),
    "worsened_query_count": (
        int(
            np.sum(
                [
                    result["rank_change"] < 0
                    for result in parsed_results
                ]
            )
        )
        if parsed_results
        else 0
    ),
}

reranker_pilot_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_reranking_pilot.json"
)

reranker_pilot_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "model": arvan_chat_model,
            "retrieval_configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "candidate": best_candidate_name,
            "top_k": reranker_top_k,
            "excerpt_char_limit": reranker_excerpt_char_limit,
            "summary": reranker_pilot_summary,
            "results": reranker_pilot_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nPilot parse success:", f"{reranker_pilot_summary['parse_success_rate']:.4f}")
print("Improved queries:", reranker_pilot_summary["improved_query_count"])
print("Worsened queries:", reranker_pilot_summary["worsened_query_count"])
print(
    "Mean original positive rank:",
    reranker_pilot_summary["mean_original_positive_rank"],
)
print(
    "Mean reranked positive rank:",
    reranker_pilot_summary["mean_reranked_positive_rank"],
)
print("Report:", reranker_pilot_report_path)
print("\nARVAN CHAT RERANKING PILOT: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## تحلیل پاسخ‌های نامعتبر پایلوت Reranking

در این مرحله پاسخ‌های دریافت‌شده از مدل که قابل‌خواندن نبودند بررسی می‌شوند.

هدف، تشخیص تفاوت میان دو حالت است:

- مدل رتبه‌بندی درست ساخته اما قالب JSON را رعایت نکرده است؛
- مدل دستور رتبه‌بندی را به‌درستی انجام نداده است.

هیچ درخواست جدیدی به سرویس ارسال نمی‌شود.

</div>

In [ ]:
# Code Cell 74
# NB13 — Read-only analysis of malformed reranking-pilot responses

failed_reranker_pilot_results = [
    result
    for result in reranker_pilot_results
    if result["response_parse_status"] == "FAIL"
]

print("=== RERANKING PILOT RESPONSE-FORMAT ANALYSIS ===")
print("Total pilot responses:", len(reranker_pilot_results))
print("Malformed or incomplete responses:", len(failed_reranker_pilot_results))

for result in failed_reranker_pilot_results:
    raw_response = result["raw_response"]

    contains_json_object = bool(
        re.search(r"\{.*\}", raw_response, flags=re.DOTALL)
    )
    contains_json_array = bool(
        re.search(r"\[.*\]", raw_response, flags=re.DOTALL)
    )
    contains_passage_label = bool(
        re.search(r"\bP\d{2}\b", raw_response)
    )

    print("\n" + "=" * 80)
    print(
        f"Pilot {result['pilot_position']} | "
        f"seed={result['source_seed_id']} | "
        f"original_positive_rank={result['original_positive_rank']}"
    )
    print("Contains JSON object:", contains_json_object)
    print("Contains JSON array:", contains_json_array)
    print("Contains passage label:", contains_passage_label)
    print("Raw response:")
    print(repr(raw_response[:2000]))

reranker_format_analysis_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_reranking_pilot_format_analysis.json"
)

reranker_format_analysis_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "pilot_query_count": len(reranker_pilot_results),
            "failed_response_count": len(failed_reranker_pilot_results),
            "failed_responses": failed_reranker_pilot_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nReport:", reranker_format_analysis_report_path)
print("\nRERANKING PILOT RESPONSE-FORMAT ANALYSIS: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## آزمون کنترل‌شدهٔ سقف پاسخ Reranker

برخی پاسخ‌های پایلوت کاملاً خالی بودند. در این مرحله فقط یک نمونه از آن‌ها با سقف پاسخ بالاتر تکرار می‌شود تا مشخص شود آیا مدل پس از پایان reasoning داخلی، خروجی رتبه‌بندی قابل‌خواندن تولید می‌کند یا خیر.

</div>

In [ ]:
# Code Cell 75
# NB13 — One controlled reranking retry with a larger completion budget

assert failed_reranker_pilot_results, (
    "No failed pilot response is available for the controlled retry."
)

retry_source_result = failed_reranker_pilot_results[0]
retry_query_index = next(
    query_index
    for query_index in selected_reranker_query_indexes
    if arvan_hybrid_query_records[query_index]["candidate_id"]
    == retry_source_result["candidate_id"]
)

retry_query_record = arvan_hybrid_query_records[retry_query_index]
retry_top_k_indexes = best_rrf_ranked_indexes[retry_query_index][:reranker_top_k]
retry_passage_labels = [
    f"P{rank:02d}"
    for rank in range(1, reranker_top_k + 1)
]

retry_passage_by_label = {
    label: best_candidate_records[int(candidate_index)]
    for label, candidate_index in zip(
        retry_passage_labels,
        retry_top_k_indexes,
    )
}

retry_prompt_passages = "\n\n".join(
    (
        f"[{label}]\n"
        f"{retry_passage_by_label[label]['content'][:reranker_excerpt_char_limit]}"
    )
    for label in retry_passage_labels
)

retry_reranker_prompt = f"""
فقط Passageهای زیر را برای پاسخ‌گویی به پرسش رتبه‌بندی کن.

قواعد:
- فقط بر اساس متن Passageها تصمیم بگیر.
- از دانش بیرونی استفاده نکن.
- همهٔ برچسب‌ها را دقیقاً یک‌بار بیاور.
- پاسخ را فوراً و فقط به شکل JSON معتبر بده.
- هیچ توضیح، تحلیل یا متن دیگری ننویس.

ساختار دقیق خروجی:
{{"ranking":["P01","P02","P03","P04","P05","P06","P07","P08","P09","P10"]}}

پرسش:
{retry_query_record["question"]}

Passageها:
{retry_prompt_passages}
""".strip()

retry_started_at = time.perf_counter()

retry_response = arvan_chat_client.chat.completions.create(
    model=arvan_chat_model,
    messages=[
        {
            "role": "user",
            "content": retry_reranker_prompt,
        }
    ],
    temperature=0,
    max_tokens=1024,
)

retry_elapsed_seconds = time.perf_counter() - retry_started_at
retry_choice = retry_response.choices[0]
retry_raw_response_text = (
    retry_choice.message.content or ""
).strip()

retry_reranked_labels = extract_reranked_labels(
    retry_raw_response_text,
    allowed_labels=set(retry_passage_labels),
)

retry_usage = getattr(retry_response, "usage", None)

print("=== CONTROLLED RERANKING RETRY ===")
print("Source pilot position:", retry_source_result["pilot_position"])
print("Seed:", retry_query_record["source_seed_id"])
print("Finish reason:", retry_choice.finish_reason)
print("Visible text received:", bool(retry_raw_response_text))
print("Valid ranking parsed:", retry_reranked_labels is not None)
print("Seconds:", round(retry_elapsed_seconds, 3))
print("Usage:", retry_usage)
print("Raw response:", repr(retry_raw_response_text[:2000]))
print("\nCONTROLLED RERANKING RETRY: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## پایلوت کالیبره‌شدهٔ Reranking

پس از تأیید سقف مناسب پاسخ، Reranking روی ده پرسش از seedهای متفاوت اجرا می‌شود.

هدف این مرحله سنجش هم‌زمان سه معیار است:

- پایداری قالب خروجی؛
- توان مدل در حفظ Passage مثبت؛
- بهبود یا افت رتبهٔ Passage مثبت نسبت به رتبه‌بندی اولیه.

</div>

In [ ]:
# Code Cell 76
# NB13 — Calibrated 10-query Arvan-chat reranking pilot

calibrated_reranker_pilot_size = 10
calibrated_query_indexes = []
calibrated_seed_ids = set()

for query_index in eligible_reranker_query_indexes:
    seed_id = arvan_hybrid_query_records[query_index]["source_seed_id"]

    if seed_id not in calibrated_seed_ids:
        calibrated_query_indexes.append(query_index)
        calibrated_seed_ids.add(seed_id)

    if len(calibrated_query_indexes) == calibrated_reranker_pilot_size:
        break

assert len(calibrated_query_indexes) == calibrated_reranker_pilot_size

calibrated_reranker_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_calibrated_reranking_pilot.json"
)

if calibrated_reranker_report_path.is_file():
    cached_calibrated_reranker_report = json.loads(
        calibrated_reranker_report_path.read_text(encoding="utf-8")
    )

    if (
        cached_calibrated_reranker_report.get("active_golden_dataset_sha256")
        == golden_dataset_context["sha256"]
        and cached_calibrated_reranker_report.get("model")
        == arvan_chat_model
    ):
        calibrated_reranker_results = cached_calibrated_reranker_report.get(
            "results",
            [],
        )
    else:
        calibrated_reranker_results = []
else:
    calibrated_reranker_results = []

cached_calibrated_reranker_candidate_ids = {
    result["candidate_id"]
    for result in calibrated_reranker_results
}

print("=== CALIBRATED ARVAN CHAT RERANKING PILOT ===")
print("Pilot queries:", len(calibrated_query_indexes))
print("Top-k passages:", reranker_top_k)
print("Maximum completion tokens: 1024")

for pilot_position, query_index in enumerate(
    calibrated_query_indexes,
    start=1,
):
    query_record = arvan_hybrid_query_records[query_index]

    if query_record["candidate_id"] in cached_calibrated_reranker_candidate_ids:
        print(
            f"Pilot {pilot_position}/{len(calibrated_query_indexes)} | "
            f"cached | candidate_id={query_record['candidate_id']}"
        )
        continue

    seed_id = query_record["source_seed_id"]
    positive_indexes = best_positive_indexes_by_seed[seed_id]
    top_k_indexes = best_rrf_ranked_indexes[query_index][:reranker_top_k]

    passage_labels = [
        f"P{rank:02d}"
        for rank in range(1, reranker_top_k + 1)
    ]

    passage_by_label = {
        label: best_candidate_records[int(candidate_index)]
        for label, candidate_index in zip(
            passage_labels,
            top_k_indexes,
        )
    }

    prompt_passages = "\n\n".join(
        (
            f"[{label}]\n"
            f"{passage_by_label[label]['content'][:reranker_excerpt_char_limit]}"
        )
        for label in passage_labels
    )

    reranker_prompt = f"""
فقط Passageهای زیر را برای پاسخ‌گویی به پرسش رتبه‌بندی کن.

قواعد:
- فقط بر اساس متن Passageها تصمیم بگیر.
- از دانش بیرونی استفاده نکن.
- همهٔ برچسب‌ها را دقیقاً یک‌بار بیاور.
- پاسخ را فوراً و فقط به شکل JSON معتبر بده.
- هیچ توضیح، تحلیل یا متن دیگری ننویس.

ساختار دقیق خروجی:
{{"ranking":["P01","P02","P03","P04","P05","P06","P07","P08","P09","P10"]}}

پرسش:
{query_record["question"]}

Passageها:
{prompt_passages}
""".strip()

    started_at = time.perf_counter()

    response = arvan_chat_client.chat.completions.create(
        model=arvan_chat_model,
        messages=[
            {
                "role": "user",
                "content": reranker_prompt,
            }
        ],
        temperature=0,
        max_tokens=1024,
    )

    elapsed_seconds = time.perf_counter() - started_at
    choice = response.choices[0]
    raw_response_text = (choice.message.content or "").strip()

    reranked_labels = extract_reranked_labels(
        raw_response_text,
        allowed_labels=set(passage_labels),
    )

    original_positive_rank = next(
        rank
        for rank, candidate_index in enumerate(top_k_indexes, start=1)
        if int(candidate_index) in positive_indexes
    )

    result = {
        "pilot_position": pilot_position,
        "candidate_id": query_record["candidate_id"],
        "source_seed_id": seed_id,
        "original_positive_rank": original_positive_rank,
        "finish_reason": choice.finish_reason,
        "response_parse_status": (
            "PASS"
            if reranked_labels is not None
            else "FAIL"
        ),
        "seconds": round(elapsed_seconds, 3),
        "cost": getattr(getattr(response, "usage", None), "cost", None),
        "raw_response": raw_response_text,
    }

    if reranked_labels is not None:
        reranked_candidate_indexes = [
            int(top_k_indexes[passage_labels.index(label)])
            for label in reranked_labels
        ]

        reranked_positive_rank = next(
            rank
            for rank, candidate_index in enumerate(
                reranked_candidate_indexes,
                start=1,
            )
            if candidate_index in positive_indexes
        )

        result["reranked_positive_rank"] = reranked_positive_rank
        result["rank_change"] = (
            original_positive_rank - reranked_positive_rank
        )

    calibrated_reranker_results.append(result)

    print(
        f"Pilot {pilot_position}/{len(calibrated_query_indexes)} | "
        f"seed={seed_id} | "
        f"original_rank={original_positive_rank} | "
        f"parse={result['response_parse_status']}"
    )

calibrated_parsed_results = [
    result
    for result in calibrated_reranker_results
    if result["response_parse_status"] == "PASS"
]

calibrated_reranker_summary = {
    "pilot_query_count": len(calibrated_reranker_results),
    "parse_success_rate": (
        len(calibrated_parsed_results)
        / len(calibrated_reranker_results)
    ),
    "mean_original_positive_rank": (
        float(
            np.mean(
                [
                    result["original_positive_rank"]
                    for result in calibrated_parsed_results
                ]
            )
        )
        if calibrated_parsed_results
        else None
    ),
    "mean_reranked_positive_rank": (
        float(
            np.mean(
                [
                    result["reranked_positive_rank"]
                    for result in calibrated_parsed_results
                ]
            )
        )
        if calibrated_parsed_results
        else None
    ),
    "improved_query_count": int(
        np.sum(
            [
                result["rank_change"] > 0
                for result in calibrated_parsed_results
            ]
        )
    ),
    "unchanged_query_count": int(
        np.sum(
            [
                result["rank_change"] == 0
                for result in calibrated_parsed_results
            ]
        )
    ),
    "worsened_query_count": int(
        np.sum(
            [
                result["rank_change"] < 0
                for result in calibrated_parsed_results
            ]
        )
    ),
    "total_observed_cost": float(
        np.sum(
            [
                result["cost"] or 0.0
                for result in calibrated_reranker_results
            ]
        )
    ),
}

calibrated_reranker_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_calibrated_reranking_pilot.json"
)

calibrated_reranker_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "model": arvan_chat_model,
            "candidate": best_candidate_name,
            "top_k": reranker_top_k,
            "max_tokens": 1024,
            "summary": calibrated_reranker_summary,
            "results": calibrated_reranker_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "\nParse success:",
    f"{calibrated_reranker_summary['parse_success_rate']:.4f}",
)
print(
    "Mean original positive rank:",
    calibrated_reranker_summary["mean_original_positive_rank"],
)
print(
    "Mean reranked positive rank:",
    calibrated_reranker_summary["mean_reranked_positive_rank"],
)
print(
    "Improved / unchanged / worsened:",
    calibrated_reranker_summary["improved_query_count"],
    "/",
    calibrated_reranker_summary["unchanged_query_count"],
    "/",
    calibrated_reranker_summary["worsened_query_count"],
)
print(
    "Total observed cost:",
    f"{calibrated_reranker_summary['total_observed_cost']:.6f}",
)
print("Report:", calibrated_reranker_report_path)
print("\nCALIBRATED ARVAN CHAT RERANKING PILOT: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی کامل مشروط Reranking

Reranker نمی‌تواند Chunkی را که در ده نتیجهٔ اولیه وجود ندارد بازیابی کند. بنابراین در این ارزیابی فقط پرسش‌هایی بررسی می‌شوند که حداقل یک شاهد مثبت از قبل در Top-10 آن‌ها وجود دارد.

هدف، سنجش دقیق اثر Reranking بر رتبهٔ شاهدهای موجود است. شواهد طلایی فقط برای محاسبهٔ معیار پس از پاسخ مدل استفاده می‌شوند و به مدل داده نمی‌شوند.

خروجی هر درخواست به‌صورت checkpoint ذخیره می‌شود تا اجرای مجدد از نقطهٔ قبلی ادامه پیدا کند.

</div>

In [ ]:
# Code Cell 77
# NB13 — Resumable complete conditional reranking evaluation:
# all queries with a positive Child already present in top 10

conditional_reranker_top_k = 10
conditional_reranker_max_tokens = 1536
conditional_reranker_excerpt_char_limit = 1000

conditional_reranker_cache_path = (
    active_benchmark_run_directory
    / "arvan_chat_conditional_reranking_cache.json"
)

conditional_reranker_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_conditional_reranking_evaluation.json"
)

if conditional_reranker_cache_path.is_file():
    conditional_reranker_cache = json.loads(
        conditional_reranker_cache_path.read_text(encoding="utf-8")
    )
else:
    conditional_reranker_cache = {
        "active_golden_dataset_sha256": golden_dataset_context["sha256"],
        "model": arvan_chat_model,
        "candidate": best_candidate_name,
        "top_k": conditional_reranker_top_k,
        "max_tokens": conditional_reranker_max_tokens,
        "results_by_candidate_id": {},
    }

assert (
    conditional_reranker_cache["active_golden_dataset_sha256"]
    == golden_dataset_context["sha256"]
), "The existing reranker cache belongs to a different Golden Dataset."

assert conditional_reranker_cache["model"] == arvan_chat_model, (
    "The existing reranker cache belongs to a different chat model."
)

conditional_query_indexes = eligible_reranker_query_indexes

print("=== RESUMABLE CONDITIONAL RERANKING EVALUATION ===")
print("Eligible queries:", len(conditional_query_indexes))
print(
    "Previously cached:",
    len(conditional_reranker_cache["results_by_candidate_id"]),
)
print("Top-k passages:", conditional_reranker_top_k)
print("Maximum completion tokens:", conditional_reranker_max_tokens)

for execution_position, query_index in enumerate(
    conditional_query_indexes,
    start=1,
):
    query_record = arvan_hybrid_query_records[query_index]
    candidate_id = query_record["candidate_id"]

    if candidate_id in conditional_reranker_cache["results_by_candidate_id"]:
        print(
            f"Query {execution_position}/{len(conditional_query_indexes)} "
            f"| cached | candidate_id={candidate_id}"
        )
        continue

    seed_id = query_record["source_seed_id"]
    positive_indexes = best_positive_indexes_by_seed[seed_id]
    top_k_indexes = best_rrf_ranked_indexes[query_index][
        :conditional_reranker_top_k
    ]

    passage_labels = [
        f"P{rank:02d}"
        for rank in range(1, conditional_reranker_top_k + 1)
    ]

    passage_by_label = {
        label: best_candidate_records[int(candidate_index)]
        for label, candidate_index in zip(
            passage_labels,
            top_k_indexes,
        )
    }

    prompt_passages = "\n\n".join(
        (
            f"[{label}]\n"
            f"{passage_by_label[label]['content'][:conditional_reranker_excerpt_char_limit]}"
        )
        for label in passage_labels
    )

    reranker_prompt = f"""
فقط Passageهای زیر را برای پاسخ‌گویی به پرسش رتبه‌بندی کن.

قواعد:
- فقط بر اساس متن Passageها تصمیم بگیر.
- از دانش بیرونی استفاده نکن.
- همهٔ برچسب‌ها را دقیقاً یک‌بار بیاور.
- پاسخ را فوراً و فقط به شکل JSON معتبر بده.
- هیچ توضیح، تحلیل یا متن دیگری ننویس.

ساختار دقیق خروجی:
{{"ranking":["P01","P02","P03","P04","P05","P06","P07","P08","P09","P10"]}}

پرسش:
{query_record["question"]}

Passageها:
{prompt_passages}
""".strip()

    original_positive_rank = next(
        rank
        for rank, candidate_index in enumerate(top_k_indexes, start=1)
        if int(candidate_index) in positive_indexes
    )

    attempts = []

    for attempt_number, max_tokens in enumerate(
        (conditional_reranker_max_tokens, 2048),
        start=1,
    ):
        started_at = time.perf_counter()

        response = arvan_chat_client.chat.completions.create(
            model=arvan_chat_model,
            messages=[
                {
                    "role": "user",
                    "content": reranker_prompt,
                }
            ],
            temperature=0,
            max_tokens=max_tokens,
        )

        elapsed_seconds = time.perf_counter() - started_at
        choice = response.choices[0]
        raw_response_text = (choice.message.content or "").strip()

        reranked_labels = extract_reranked_labels(
            raw_response_text,
            allowed_labels=set(passage_labels),
        )

        attempts.append(
            {
                "attempt_number": attempt_number,
                "max_tokens": max_tokens,
                "finish_reason": choice.finish_reason,
                "seconds": round(elapsed_seconds, 3),
                "cost": getattr(
                    getattr(response, "usage", None),
                    "cost",
                    None,
                ),
                "raw_response": raw_response_text,
                "parsed": reranked_labels is not None,
            }
        )

        if reranked_labels is not None:
            break

    result = {
        "candidate_id": candidate_id,
        "source_seed_id": seed_id,
        "question": query_record["question"],
        "original_positive_rank": original_positive_rank,
        "attempts": attempts,
        "response_parse_status": (
            "PASS"
            if reranked_labels is not None
            else "FAIL"
        ),
    }

    if reranked_labels is not None:
        reranked_candidate_indexes = [
            int(top_k_indexes[passage_labels.index(label)])
            for label in reranked_labels
        ]

        reranked_positive_rank = next(
            rank
            for rank, candidate_index in enumerate(
                reranked_candidate_indexes,
                start=1,
            )
            if candidate_index in positive_indexes
        )

        result["reranked_positive_rank"] = reranked_positive_rank
        result["rank_change"] = (
            original_positive_rank - reranked_positive_rank
        )

    conditional_reranker_cache["results_by_candidate_id"][
        candidate_id
    ] = result

    conditional_reranker_cache_path.write_text(
        json.dumps(
            conditional_reranker_cache,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        f"Query {execution_position}/{len(conditional_query_indexes)} | "
        f"seed={seed_id} | "
        f"original_rank={original_positive_rank} | "
        f"parse={result['response_parse_status']}"
    )

conditional_reranker_results = list(
    conditional_reranker_cache["results_by_candidate_id"].values()
)

parsed_conditional_results = [
    result
    for result in conditional_reranker_results
    if result["response_parse_status"] == "PASS"
]

conditional_reranker_summary = {
    "eligible_query_count": len(conditional_query_indexes),
    "completed_query_count": len(conditional_reranker_results),
    "parse_success_rate": (
        len(parsed_conditional_results)
        / len(conditional_reranker_results)
        if conditional_reranker_results
        else 0.0
    ),
    "mean_original_positive_rank": (
        float(
            np.mean(
                [
                    result["original_positive_rank"]
                    for result in parsed_conditional_results
                ]
            )
        )
        if parsed_conditional_results
        else None
    ),
    "mean_reranked_positive_rank": (
        float(
            np.mean(
                [
                    result["reranked_positive_rank"]
                    for result in parsed_conditional_results
                ]
            )
        )
        if parsed_conditional_results
        else None
    ),
    "improved_query_count": int(
        np.sum(
            [
                result["rank_change"] > 0
                for result in parsed_conditional_results
            ]
        )
    ),
    "unchanged_query_count": int(
        np.sum(
            [
                result["rank_change"] == 0
                for result in parsed_conditional_results
            ]
        )
    ),
    "worsened_query_count": int(
        np.sum(
            [
                result["rank_change"] < 0
                for result in parsed_conditional_results
            ]
        )
    ),
    "total_observed_cost": float(
        np.sum(
            [
                attempt["cost"] or 0.0
                for result in conditional_reranker_results
                for attempt in result["attempts"]
            ]
        )
    ),
}

conditional_reranker_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "model": arvan_chat_model,
            "retrieval_configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "candidate": best_candidate_name,
            "top_k": conditional_reranker_top_k,
            "summary": conditional_reranker_summary,
            "results": conditional_reranker_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nCompleted:", conditional_reranker_summary["completed_query_count"])
print(
    "Parse success:",
    f"{conditional_reranker_summary['parse_success_rate']:.4f}",
)
print(
    "Mean original positive rank:",
    conditional_reranker_summary["mean_original_positive_rank"],
)
print(
    "Mean reranked positive rank:",
    conditional_reranker_summary["mean_reranked_positive_rank"],
)
print(
    "Improved / unchanged / worsened:",
    conditional_reranker_summary["improved_query_count"],
    "/",
    conditional_reranker_summary["unchanged_query_count"],
    "/",
    conditional_reranker_summary["worsened_query_count"],
)
print(
    "Total observed cost:",
    f"{conditional_reranker_summary['total_observed_cost']:.6f}",
)
print("Report:", conditional_reranker_report_path)
print("\nRESUMABLE CONDITIONAL RERANKING EVALUATION: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## محاسبهٔ اثر کلی Reranking

Reranking فقط می‌تواند ترتیب Chunkهایی را تغییر دهد که از قبل در Top-10 بازیابی اولیه حضور دارند.

در این مرحله معیارهای نهایی بازیابی اولیه و بازیابی پس از Reranking روی تمام ۵۵۳ پرسش مقایسه می‌شوند. هیچ درخواست جدیدی به سرویس ارسال نمی‌شود.

</div>

In [ ]:
# Code Cell 78
# NB13 — Full-query metric impact of conditional chat reranking

reranked_positive_rank_by_candidate_id = {
    result["candidate_id"]: result["reranked_positive_rank"]
    for result in parsed_conditional_results
}

assert len(reranked_positive_rank_by_candidate_id) == 38
assert len(parsed_conditional_results) == 38

baseline_first_positive_rank_by_candidate_id = {}
final_first_positive_rank_by_candidate_id = {}

for query_index, query_record in enumerate(arvan_hybrid_query_records):
    candidate_id = query_record["candidate_id"]
    seed_id = query_record["source_seed_id"]

    positive_rank_positions = np.flatnonzero(
        np.isin(
            best_rrf_ranked_indexes[query_index],
            list(best_positive_indexes_by_seed[seed_id]),
        )
    )

    baseline_first_rank = int(positive_rank_positions[0]) + 1
    baseline_first_positive_rank_by_candidate_id[candidate_id] = (
        baseline_first_rank
    )

    final_first_positive_rank_by_candidate_id[candidate_id] = (
        reranked_positive_rank_by_candidate_id.get(
            candidate_id,
            baseline_first_rank,
        )
    )

assert len(final_first_positive_rank_by_candidate_id) == 553

def metrics_from_first_positive_ranks(rank_by_candidate_id):
    ranks = list(rank_by_candidate_id.values())

    return {
        "recall_at_1": float(np.mean([rank <= 1 for rank in ranks])),
        "recall_at_3": float(np.mean([rank <= 3 for rank in ranks])),
        "recall_at_5": float(np.mean([rank <= 5 for rank in ranks])),
        "recall_at_10": float(np.mean([rank <= 10 for rank in ranks])),
        "mrr": float(np.mean([1 / rank for rank in ranks])),
        "mean_first_positive_rank": float(np.mean(ranks)),
    }

baseline_hybrid_metrics = metrics_from_first_positive_ranks(
    baseline_first_positive_rank_by_candidate_id
)
reranked_hybrid_metrics = metrics_from_first_positive_ranks(
    final_first_positive_rank_by_candidate_id
)

reranking_impact_report = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "retrieval_configuration": (
        "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
    ),
    "reranker_model": arvan_chat_model,
    "candidate": best_candidate_name,
    "reranker_scope": {
        "all_query_count": 553,
        "reranked_query_count": len(reranked_positive_rank_by_candidate_id),
        "reranking_input_top_k": 10,
    },
    "baseline_metrics": baseline_hybrid_metrics,
    "reranked_metrics": reranked_hybrid_metrics,
    "metric_deltas": {
        metric_name: (
            reranked_hybrid_metrics[metric_name]
            - baseline_hybrid_metrics[metric_name]
        )
        for metric_name in baseline_hybrid_metrics
    },
}

reranking_impact_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_reranking_full_metric_impact.json"
)

reranking_impact_report_path.write_text(
    json.dumps(
        reranking_impact_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== FULL-QUERY RERANKING METRIC IMPACT ===")
print("Queries evaluated:", len(final_first_positive_rank_by_candidate_id))
print("Queries reranked:", len(reranked_positive_rank_by_candidate_id))

for metric_name in (
    "recall_at_1",
    "recall_at_3",
    "recall_at_5",
    "recall_at_10",
    "mrr",
    "mean_first_positive_rank",
):
    print(
        f"\n{metric_name}"
        f"\n- baseline: {baseline_hybrid_metrics[metric_name]:.4f}"
        f"\n- reranked: {reranked_hybrid_metrics[metric_name]:.4f}"
        f"\n- delta: "
        f"{reranked_hybrid_metrics[metric_name] - baseline_hybrid_metrics[metric_name]:+.4f}"
    )

print("\nReport:", reranking_impact_report_path)
print("\nFULL-QUERY RERANKING METRIC IMPACT: PASS")

<div dir="rtl" align="right">

## اثر Reranking بر نگه‌داشتن شواهد در کانتکست نهایی

در سل قبل دیدیم که Reranking می‌تواند جایگاه شواهد درست را در میان ده Chunk بازیابی‌شده بهتر کند، اما نمی‌تواند شواهدی را که اصلاً در ده نتیجهٔ اول نیستند پیدا کند.

در این سل بررسی می‌کنیم که آیا جابه‌جایی Chunkها توسط Reranking باعث می‌شود در سیاست انتخاب‌شدهٔ کانتکست والد–فرزند، Parent حاوی شواهد درست با احتمال بیشتری وارد کانتکست نهایی شود یا نه. این تحلیل فقط از خروجی‌های ذخیره‌شده استفاده می‌کند و هیچ فراخوانی جدیدی به Arvan ندارد.

</div>

In [ ]:
# Code Cell 79
# NB13 — Reranking impact on expansive parent-context evidence retention
# Uses cached reranking responses only; makes no Arvan API requests.

selected_context_policy = next(
    policy
    for policy in context_budget_policies
    if policy["policy_id"] == "expansive"
)

reranking_result_by_candidate_id = {
    result["candidate_id"]: result
    for result in parsed_conditional_results
}

assert len(reranking_result_by_candidate_id) == 38, (
    "Expected 38 parsed conditional reranking results."
)

def assemble_parent_context_from_child_indexes(
    ranked_child_indexes,
    policy,
):
    selected_parent_ids = []
    current_token_total = 0
    missing_parent_references = 0

    for child_index in ranked_child_indexes[:policy["retrieved_child_k"]]:
        child_record = best_candidate_records[int(child_index)]
        parent_chunk_id = child_record["parent_chunk_id"]

        if parent_chunk_id not in parent_record_by_chunk_id:
            missing_parent_references += 1
            continue

        if parent_chunk_id in selected_parent_ids:
            continue

        parent_record = parent_record_by_chunk_id[parent_chunk_id]
        parent_token_count = parent_record["token_count"]

        if len(selected_parent_ids) >= policy["max_parent_chunks"]:
            continue

        if (
            current_token_total + parent_token_count
            > policy["max_context_tokens"]
        ):
            continue

        selected_parent_ids.append(parent_chunk_id)
        current_token_total += parent_token_count

    return {
        "selected_parent_chunk_ids": selected_parent_ids,
        "token_count": current_token_total,
        "missing_parent_references": missing_parent_references,
    }

reranking_context_rows = []

for query_index, query_record in enumerate(arvan_hybrid_query_records):
    candidate_id = query_record["candidate_id"]
    seed_id = query_record["source_seed_id"]

    baseline_ranked_child_indexes = best_rrf_ranked_indexes[query_index]
    final_ranked_child_indexes = baseline_ranked_child_indexes.copy()

    reranking_applied = candidate_id in reranking_result_by_candidate_id

    if reranking_applied:
        reranking_result = reranking_result_by_candidate_id[candidate_id]

        successful_attempt = next(
            attempt
            for attempt in reranking_result["attempts"]
            if attempt["parsed"]
        )

        reranked_labels = extract_reranked_labels(
            successful_attempt["raw_response"],
            allowed_labels={
                f"P{rank:02d}"
                for rank in range(1, conditional_reranker_top_k + 1)
            },
        )

        assert reranked_labels is not None, (
            f"Cached reranking response cannot be parsed: {candidate_id}"
        )

        baseline_top_k_indexes = baseline_ranked_child_indexes[
            :conditional_reranker_top_k
        ]

        reranked_top_k_indexes = np.array(
            [
                int(
                    baseline_top_k_indexes[
                        int(label.removeprefix("P")) - 1
                    ]
                )
                for label in reranked_labels
            ],
            dtype=int,
        )

        assert len(set(reranked_top_k_indexes)) == conditional_reranker_top_k
        assert set(reranked_top_k_indexes) == set(baseline_top_k_indexes)

        final_ranked_child_indexes = np.concatenate(
            [
                reranked_top_k_indexes,
                baseline_ranked_child_indexes[
                    conditional_reranker_top_k:
                ],
            ]
        )

    positive_child_indexes = best_positive_indexes_by_seed[seed_id]
    positive_parent_ids = {
        best_candidate_records[int(child_index)]["parent_chunk_id"]
        for child_index in positive_child_indexes
    }

    baseline_context = assemble_parent_context_from_child_indexes(
        baseline_ranked_child_indexes,
        selected_context_policy,
    )
    reranked_context = assemble_parent_context_from_child_indexes(
        final_ranked_child_indexes,
        selected_context_policy,
    )

    baseline_parent_hit = bool(
        positive_parent_ids.intersection(
            baseline_context["selected_parent_chunk_ids"]
        )
    )
    reranked_parent_hit = bool(
        positive_parent_ids.intersection(
            reranked_context["selected_parent_chunk_ids"]
        )
    )

    reranking_context_rows.append(
        {
            "candidate_id": candidate_id,
            "source_seed_id": seed_id,
            "reranking_applied": reranking_applied,
            "baseline_parent_evidence_retained": baseline_parent_hit,
            "reranked_parent_evidence_retained": reranked_parent_hit,
            "baseline_parent_chunk_count": len(
                baseline_context["selected_parent_chunk_ids"]
            ),
            "reranked_parent_chunk_count": len(
                reranked_context["selected_parent_chunk_ids"]
            ),
            "baseline_context_tokens": baseline_context["token_count"],
            "reranked_context_tokens": reranked_context["token_count"],
            "missing_parent_references": (
                baseline_context["missing_parent_references"]
                + reranked_context["missing_parent_references"]
            ),
        }
    )

assert len(reranking_context_rows) == 553
assert sum(
    row["missing_parent_references"]
    for row in reranking_context_rows
) == 0, "A selected Child referenced a missing Parent."

baseline_parent_retention = float(
    np.mean(
        [
            row["baseline_parent_evidence_retained"]
            for row in reranking_context_rows
        ]
    )
)

reranked_parent_retention = float(
    np.mean(
        [
            row["reranked_parent_evidence_retained"]
            for row in reranking_context_rows
        ]
    )
)

reranked_scope_rows = [
    row
    for row in reranking_context_rows
    if row["reranking_applied"]
]

baseline_scope_retention = float(
    np.mean(
        [
            row["baseline_parent_evidence_retained"]
            for row in reranked_scope_rows
        ]
    )
)

reranked_scope_retention = float(
    np.mean(
        [
            row["reranked_parent_evidence_retained"]
            for row in reranked_scope_rows
        ]
    )
)

improved_parent_retention_count = sum(
    (
        not row["baseline_parent_evidence_retained"]
        and row["reranked_parent_evidence_retained"]
    )
    for row in reranking_context_rows
)

worsened_parent_retention_count = sum(
    (
        row["baseline_parent_evidence_retained"]
        and not row["reranked_parent_evidence_retained"]
    )
    for row in reranking_context_rows
)

reranking_context_retention_report = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "retrieval_configuration": (
        "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
    ),
    "reranker_model": arvan_chat_model,
    "context_policy": selected_context_policy,
    "query_count": len(reranking_context_rows),
    "reranked_query_count": len(reranked_scope_rows),
    "all_query_parent_evidence_retention": {
        "baseline": baseline_parent_retention,
        "reranked": reranked_parent_retention,
        "delta": reranked_parent_retention - baseline_parent_retention,
    },
    "reranked_scope_parent_evidence_retention": {
        "baseline": baseline_scope_retention,
        "reranked": reranked_scope_retention,
        "delta": reranked_scope_retention - baseline_scope_retention,
    },
    "improved_parent_retention_count": improved_parent_retention_count,
    "worsened_parent_retention_count": worsened_parent_retention_count,
    "rows": reranking_context_rows,
}

reranking_context_retention_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_reranking_parent_context_retention.json"
)

reranking_context_retention_report_path.write_text(
    json.dumps(
        reranking_context_retention_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== RERANKING IMPACT ON PARENT-CONTEXT RETENTION ===")
print("Context policy:", selected_context_policy["policy_id"])
print("Queries evaluated:", len(reranking_context_rows))
print("Queries reranked:", len(reranked_scope_rows))

print("\nAll-query parent-evidence retention:")
print(f"- baseline: {baseline_parent_retention:.4f}")
print(f"- reranked: {reranked_parent_retention:.4f}")
print(
    "- delta: "
    f"{reranked_parent_retention - baseline_parent_retention:+.4f}"
)

print("\nReranked-query scope parent-evidence retention:")
print(f"- baseline: {baseline_scope_retention:.4f}")
print(f"- reranked: {reranked_scope_retention:.4f}")
print(
    "- delta: "
    f"{reranked_scope_retention - baseline_scope_retention:+.4f}"
)

print("\nContext evidence changes:")
print("- improved:", improved_parent_retention_count)
print("- worsened:", worsened_parent_retention_count)

print("\nReport:", reranking_context_retention_report_path)
print("\nRERANKING IMPACT ON PARENT-CONTEXT RETENTION: PASS")

<div dir="rtl" align="right">

## امکان‌سنجی اجرایی Reranking

نتایج قبل نشان دادند که Reranking جایگاه شواهد درست را بهتر می‌کند و در بخشی از موارد باعث می‌شود Parent درست وارد کانتکست نهایی شود.

اما ارزیابی قبلی فقط روی پرسش‌هایی انجام شد که در دادهٔ ارزیابی می‌دانستیم شواهد درست از قبل در ده نتیجهٔ اول هستند. در محصول واقعی چنین اطلاعی نداریم؛ بنابراین اگر Reranking وارد محصول شود، باید برای همهٔ پرسش‌ها یا بر اساس یک قانون مستقل و قابل‌اجرا اعمال شود.

در این سل، با استفاده از زمان و هزینهٔ واقعی درخواست‌های ثبت‌شده، هزینه و زمان تقریبی اجرای کامل روی همهٔ پرسش‌های Golden Dataset تخمین زده می‌شود. این تخمین تصمیم اجرایی نیست و هیچ درخواست جدیدی به سرویس ارسال نمی‌کند.

</div>

In [ ]:
# Code Cell 80
# NB13 — Reranking feasibility profile from cached live executions
# No Arvan API requests are made in this cell.

assert len(parsed_conditional_results) == 38, (
    "Expected the 38 parsed conditional reranking results."
)

reranking_execution_rows = []

for result in parsed_conditional_results:
    attempt_seconds = [
        float(attempt["seconds"])
        for attempt in result["attempts"]
        if attempt.get("seconds") is not None
    ]
    attempt_costs = [
        float(attempt["cost"])
        for attempt in result["attempts"]
        if attempt.get("cost") is not None
    ]

    reranking_execution_rows.append(
        {
            "candidate_id": result["candidate_id"],
            "source_seed_id": result["source_seed_id"],
            "attempt_count": len(result["attempts"]),
            "total_seconds": float(sum(attempt_seconds)),
            "total_cost": float(sum(attempt_costs)),
            "response_parse_status": result["response_parse_status"],
        }
    )

assert len(reranking_execution_rows) == 38
assert all(
    row["response_parse_status"] == "PASS"
    for row in reranking_execution_rows
)

observed_seconds = np.array(
    [row["total_seconds"] for row in reranking_execution_rows],
    dtype=float,
)
observed_costs = np.array(
    [row["total_cost"] for row in reranking_execution_rows],
    dtype=float,
)

full_query_count = len(arvan_hybrid_query_records)

reranking_feasibility_profile = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "reranker_model": arvan_chat_model,
    "measured_query_count": len(reranking_execution_rows),
    "full_golden_query_count": full_query_count,
    "measured_execution": {
        "parse_success_rate": 1.0,
        "mean_seconds_per_query": float(np.mean(observed_seconds)),
        "p95_seconds_per_query": float(
            np.percentile(observed_seconds, 95)
        ),
        "mean_cost_per_query": float(np.mean(observed_costs)),
        "p95_cost_per_query": float(
            np.percentile(observed_costs, 95)
        ),
        "total_observed_cost": float(np.sum(observed_costs)),
    },
    "sequential_full_evaluation_estimate": {
        "estimated_seconds": float(
            np.mean(observed_seconds) * full_query_count
        ),
        "estimated_minutes": float(
            np.mean(observed_seconds) * full_query_count / 60
        ),
        "estimated_cost": float(
            np.mean(observed_costs) * full_query_count
        ),
    },
    "interpretation": [
        (
            "This is an estimate from cached live executions, not a "
            "guaranteed production latency or cost."
        ),
        (
            "The prior reranking evaluation was conditional on a "
            "positive Child already being in the top 10."
        ),
        (
            "A production policy cannot use Golden labels to decide "
            "whether reranking should run."
        ),
    ],
    "next_decision": (
        "Decide whether to run an unbiased full-query reranking "
        "evaluation on all active Golden Dataset questions."
    ),
}

reranking_feasibility_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_reranking_feasibility_profile.json"
)

reranking_feasibility_report_path.write_text(
    json.dumps(
        reranking_feasibility_profile,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

measured = reranking_feasibility_profile["measured_execution"]
estimated = reranking_feasibility_profile[
    "sequential_full_evaluation_estimate"
]

print("=== RERANKING FEASIBILITY PROFILE ===")
print("Measured cached queries:", len(reranking_execution_rows))
print("Full Golden query count:", full_query_count)

print("\nObserved per-query execution:")
print(f"- Parse success: {measured['parse_success_rate']:.4f}")
print(f"- Mean seconds: {measured['mean_seconds_per_query']:.2f}")
print(f"- P95 seconds: {measured['p95_seconds_per_query']:.2f}")
print(f"- Mean cost: {measured['mean_cost_per_query']:.6f}")
print(f"- P95 cost: {measured['p95_cost_per_query']:.6f}")
print(f"- Total observed cost: {measured['total_observed_cost']:.6f}")

print("\nSequential estimate for all Golden queries:")
print(f"- Estimated minutes: {estimated['estimated_minutes']:.1f}")
print(f"- Estimated cost: {estimated['estimated_cost']:.4f}")

print("\nImportant boundary:")
for item in reranking_feasibility_profile["interpretation"]:
    print("-", item)

print("\nReport:", reranking_feasibility_report_path)
print("\nRERANKING FEASIBILITY PROFILE: PASS")

<div dir="rtl" align="right">

## ارزیابی کامل و قابل‌ادامهٔ Reranking

در این مرحله Reranking برای همهٔ پرسش‌های Golden Dataset ارزیابی می‌شود، نه فقط پرسش‌هایی که در ارزیابی قبلی می‌دانستیم شواهد درست در ده نتیجهٔ اول دارند.

نتیجهٔ هر پرسش بلافاصله در یک کش ذخیره می‌شود. شناسهٔ این کش بر اساس هش Golden Dataset، مدل، Chunk candidate و تنظیمات Reranking است؛ بنابراین تغییر نام Notebook یا Restart کردن Kernel باعث تکرار درخواست‌های انجام‌شده نمی‌شود. اگر اجرا قطع شود، اجرای بعدی فقط پرسش‌های باقی‌مانده را ارسال می‌کند.

</div>

In [ ]:
# Code Cell 81
# NB13 — Full resumable Arvan Chat reranking evaluation
# Reuses the prior 38 conditional results and sends only missing requests.

import json
import time

full_reranker_top_k = 10
full_reranker_max_tokens = 1536
full_reranker_retry_max_tokens = 2048
full_reranker_excerpt_char_limit = 1000

full_reranker_cache_path = (
    active_benchmark_run_directory
    / "arvan_chat_full_reranking_cache.json"
)

full_reranker_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_full_reranking_evaluation.json"
)

allowed_full_reranker_labels = {
    f"P{rank:02d}"
    for rank in range(1, full_reranker_top_k + 1)
}

def get_successful_reranked_labels(result):
    if isinstance(result.get("reranked_labels"), list):
        labels = result["reranked_labels"]

        if (
            len(labels) == full_reranker_top_k
            and set(labels) == allowed_full_reranker_labels
        ):
            return labels

    for attempt in result.get("attempts", []):
        if not attempt.get("parsed"):
            continue

        labels = extract_reranked_labels(
            attempt.get("raw_response", ""),
            allowed_labels=allowed_full_reranker_labels,
        )

        if labels is not None:
            return labels

    return None

if full_reranker_cache_path.is_file():
    full_reranker_cache = json.loads(
        full_reranker_cache_path.read_text(encoding="utf-8")
    )
else:
    full_reranker_cache = {
        "active_golden_dataset_sha256": golden_dataset_context["sha256"],
        "model": arvan_chat_model,
        "candidate": best_candidate_name,
        "top_k": full_reranker_top_k,
        "excerpt_char_limit": full_reranker_excerpt_char_limit,
        "results_by_candidate_id": {},
    }

assert (
    full_reranker_cache["active_golden_dataset_sha256"]
    == golden_dataset_context["sha256"]
), "The full reranking cache belongs to a different Golden Dataset."

assert full_reranker_cache["model"] == arvan_chat_model, (
    "The full reranking cache belongs to a different chat model."
)

assert full_reranker_cache["candidate"] == best_candidate_name, (
    "The full reranking cache belongs to a different Chunk candidate."
)

assert full_reranker_cache["top_k"] == full_reranker_top_k, (
    "The full reranking cache uses a different top-k setting."
)

# Reuse the 38 already-paid conditional runs.
conditional_reuse_count = 0

for conditional_result in parsed_conditional_results:
    candidate_id = conditional_result["candidate_id"]

    if candidate_id in full_reranker_cache["results_by_candidate_id"]:
        continue

    reranked_labels = get_successful_reranked_labels(
        conditional_result
    )

    assert reranked_labels is not None, (
        "A prior conditional result has no usable reranked labels: "
        f"{candidate_id}"
    )

    full_reranker_cache["results_by_candidate_id"][candidate_id] = {
        **conditional_result,
        "reranked_labels": reranked_labels,
        "result_origin": "REUSED_FROM_CONDITIONAL_EVALUATION",
    }
    conditional_reuse_count += 1

if conditional_reuse_count:
    full_reranker_cache_path.write_text(
        json.dumps(
            full_reranker_cache,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

print("=== FULL RESUMABLE ARVAN CHAT RERANKING ===")
print("Golden queries:", len(arvan_hybrid_query_records))
print(
    "Previously cached:",
    len(full_reranker_cache["results_by_candidate_id"]),
)
print(
    "Reused from conditional evaluation:",
    conditional_reuse_count,
)
print("Top-k passages:", full_reranker_top_k)
print("Maximum completion tokens:", full_reranker_max_tokens)

for query_position, query_record in enumerate(
    arvan_hybrid_query_records,
    start=1,
):
    candidate_id = query_record["candidate_id"]

    if candidate_id in full_reranker_cache["results_by_candidate_id"]:
        print(
            f"Query {query_position}/{len(arvan_hybrid_query_records)} "
            f"| cached | candidate_id={candidate_id}"
        )
        continue

    query_index = query_position - 1
    top_k_indexes = best_rrf_ranked_indexes[query_index][
        :full_reranker_top_k
    ]

    passage_labels = [
        f"P{rank:02d}"
        for rank in range(1, full_reranker_top_k + 1)
    ]

    passage_by_label = {
        label: best_candidate_records[int(chunk_index)]
        for label, chunk_index in zip(
            passage_labels,
            top_k_indexes,
        )
    }

    prompt_passages = "\n\n".join(
        (
            f"[{label}]\n"
            f"{passage_by_label[label]['content'][:full_reranker_excerpt_char_limit]}"
        )
        for label in passage_labels
    )

    reranker_prompt = f"""
فقط Passageهای زیر را برای پاسخ‌گویی به پرسش رتبه‌بندی کن.

قواعد:
- فقط بر اساس متن Passageها تصمیم بگیر.
- از دانش بیرونی استفاده نکن.
- همهٔ برچسب‌ها را دقیقاً یک‌بار بیاور.
- پاسخ را فوراً و فقط به شکل JSON معتبر بده.
- هیچ توضیح، تحلیل یا متن دیگری ننویس.

ساختار دقیق خروجی:
{{"ranking":["P01","P02","P03","P04","P05","P06","P07","P08","P09","P10"]}}

پرسش:
{query_record["question"]}

Passageها:
{prompt_passages}
""".strip()

    attempts = []
    reranked_labels = None

    for attempt_number, max_tokens in enumerate(
        (
            full_reranker_max_tokens,
            full_reranker_retry_max_tokens,
        ),
        start=1,
    ):
        started_at = time.perf_counter()

        response = arvan_chat_client.chat.completions.create(
            model=arvan_chat_model,
            messages=[
                {
                    "role": "user",
                    "content": reranker_prompt,
                }
            ],
            temperature=0,
            max_tokens=max_tokens,
        )

        elapsed_seconds = time.perf_counter() - started_at
        choice = response.choices[0]
        raw_response_text = (choice.message.content or "").strip()

        reranked_labels = extract_reranked_labels(
            raw_response_text,
            allowed_labels=allowed_full_reranker_labels,
        )

        attempts.append(
            {
                "attempt_number": attempt_number,
                "max_tokens": max_tokens,
                "finish_reason": choice.finish_reason,
                "seconds": round(elapsed_seconds, 3),
                "cost": getattr(
                    getattr(response, "usage", None),
                    "cost",
                    None,
                ),
                "raw_response": raw_response_text,
                "parsed": reranked_labels is not None,
            }
        )

        if reranked_labels is not None:
            break

    result = {
        "candidate_id": candidate_id,
        "source_seed_id": query_record["source_seed_id"],
        "question": query_record["question"],
        "response_parse_status": (
            "PASS"
            if reranked_labels is not None
            else "FAIL"
        ),
        "reranked_labels": reranked_labels,
        "attempts": attempts,
        "result_origin": "FULL_UNBIASED_EVALUATION",
    }

    full_reranker_cache["results_by_candidate_id"][candidate_id] = result

    # Save after every request: safe to restart or rename the notebook.
    full_reranker_cache_path.write_text(
        json.dumps(
            full_reranker_cache,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    total_cost = sum(
        float(attempt["cost"])
        for attempt in attempts
        if attempt.get("cost") is not None
    )

    print(
        f"Query {query_position}/{len(arvan_hybrid_query_records)} | "
        f"seed={query_record['source_seed_id']} | "
        f"parse={result['response_parse_status']} | "
        f"cost={total_cost:.6f}"
    )

full_reranker_results = list(
    full_reranker_cache["results_by_candidate_id"].values()
)

assert len(full_reranker_results) == len(arvan_hybrid_query_records), (
    "The full reranking cache is incomplete."
)

full_reranker_parse_success_count = sum(
    result["response_parse_status"] == "PASS"
    for result in full_reranker_results
)

full_reranker_total_cost = sum(
    float(attempt["cost"])
    for result in full_reranker_results
    for attempt in result["attempts"]
    if attempt.get("cost") is not None
)

full_reranker_report = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "model": arvan_chat_model,
    "candidate": best_candidate_name,
    "top_k": full_reranker_top_k,
    "excerpt_char_limit": full_reranker_excerpt_char_limit,
    "query_count": len(full_reranker_results),
    "parse_success_count": full_reranker_parse_success_count,
    "parse_success_rate": (
        full_reranker_parse_success_count
        / len(full_reranker_results)
    ),
    "total_observed_cost": full_reranker_total_cost,
    "results": full_reranker_results,
}

full_reranker_report_path.write_text(
    json.dumps(
        full_reranker_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nCompleted:", len(full_reranker_results))
print(
    "Parse success:",
    f"{full_reranker_report['parse_success_rate']:.4f}",
)
print(
    "Total observed cost:",
    f"{full_reranker_total_cost:.6f}",
)
print("Cache:", full_reranker_cache_path)
print("Report:", full_reranker_report_path)
print("\nFULL RESUMABLE ARVAN CHAT RERANKING: PASS")

<div dir="rtl" align="right">

## ترمیم پاسخ‌های نامعتبر Reranking

اجرای کامل Reranking برای همهٔ پرسش‌ها انجام شد. اگر تعداد کمی پاسخ به‌علت ناقص یا نامعتبر بودن ساختار JSON قابل‌خواندن نباشند، فقط همان موارد با تنظیم خروجی بزرگ‌تر دوباره اجرا می‌شوند.

نتیجهٔ ترمیم در همان کش اصلی ثبت می‌شود. این مرحله هیچ پرسش موفقی را دوباره ارسال نمی‌کند.

</div>

In [ ]:
# Code Cell 82
# NB13 — Retry only failed full-reranking responses.
# Successful cached requests are never sent again.

failed_full_reranker_results = [
    result
    for result in full_reranker_cache["results_by_candidate_id"].values()
    if result["response_parse_status"] != "PASS"
]

query_index_by_candidate_id = {
    query_record["candidate_id"]: query_index
    for query_index, query_record in enumerate(
        arvan_hybrid_query_records
    )
}

print("=== FULL RERANKING FAILED-RESPONSE REPAIR ===")
print("Failed responses before repair:", len(failed_full_reranker_results))

for repair_position, result in enumerate(
    failed_full_reranker_results,
    start=1,
):
    candidate_id = result["candidate_id"]
    query_index = query_index_by_candidate_id[candidate_id]
    query_record = arvan_hybrid_query_records[query_index]

    top_k_indexes = best_rrf_ranked_indexes[query_index][
        :full_reranker_top_k
    ]

    passage_labels = [
        f"P{rank:02d}"
        for rank in range(1, full_reranker_top_k + 1)
    ]

    passage_by_label = {
        label: best_candidate_records[int(chunk_index)]
        for label, chunk_index in zip(
            passage_labels,
            top_k_indexes,
        )
    }

    prompt_passages = "\n\n".join(
        (
            f"[{label}]\n"
            f"{passage_by_label[label]['content'][:full_reranker_excerpt_char_limit]}"
        )
        for label in passage_labels
    )

    repair_prompt = f"""
فقط Passageهای زیر را برای پاسخ‌گویی به پرسش رتبه‌بندی کن.

قواعد:
- فقط بر اساس متن Passageها تصمیم بگیر.
- از دانش بیرونی استفاده نکن.
- همهٔ برچسب‌ها را دقیقاً یک‌بار بیاور.
- پاسخ را فوراً و فقط به شکل JSON معتبر بده.
- هیچ توضیح، تحلیل یا متن دیگری ننویس.

ساختار دقیق خروجی:
{{"ranking":["P01","P02","P03","P04","P05","P06","P07","P08","P09","P10"]}}

پرسش:
{query_record["question"]}

Passageها:
{prompt_passages}
""".strip()

    repair_attempts = []
    repaired_labels = None

    for attempt_number, max_tokens in enumerate(
        (3072, 4096),
        start=1,
    ):
        started_at = time.perf_counter()

        response = arvan_chat_client.chat.completions.create(
            model=arvan_chat_model,
            messages=[
                {
                    "role": "user",
                    "content": repair_prompt,
                }
            ],
            temperature=0,
            max_tokens=max_tokens,
        )

        elapsed_seconds = time.perf_counter() - started_at
        choice = response.choices[0]
        raw_response_text = (choice.message.content or "").strip()

        repaired_labels = extract_reranked_labels(
            raw_response_text,
            allowed_labels=allowed_full_reranker_labels,
        )

        repair_attempts.append(
            {
                "repair_attempt_number": attempt_number,
                "max_tokens": max_tokens,
                "finish_reason": choice.finish_reason,
                "seconds": round(elapsed_seconds, 3),
                "cost": getattr(
                    getattr(response, "usage", None),
                    "cost",
                    None,
                ),
                "raw_response": raw_response_text,
                "parsed": repaired_labels is not None,
            }
        )

        if repaired_labels is not None:
            break

    result.setdefault("repair_attempts", []).extend(repair_attempts)

    if repaired_labels is not None:
        result["response_parse_status"] = "PASS"
        result["reranked_labels"] = repaired_labels
        result["result_origin"] = "REPAIRED_FULL_EVALUATION"

    full_reranker_cache["results_by_candidate_id"][candidate_id] = result

    full_reranker_cache_path.write_text(
        json.dumps(
            full_reranker_cache,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        f"Repair {repair_position}/{len(failed_full_reranker_results)} | "
        f"candidate_id={candidate_id} | "
        f"parse={result['response_parse_status']}"
    )

full_reranker_results = list(
    full_reranker_cache["results_by_candidate_id"].values()
)

full_reranker_parse_success_count = sum(
    result["response_parse_status"] == "PASS"
    for result in full_reranker_results
)

full_reranker_total_cost = sum(
    float(attempt["cost"])
    for result in full_reranker_results
    for attempt_group in (
        result.get("attempts", []),
        result.get("repair_attempts", []),
    )
    for attempt in attempt_group
    if attempt.get("cost") is not None
)

full_reranker_report = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "model": arvan_chat_model,
    "candidate": best_candidate_name,
    "top_k": full_reranker_top_k,
    "excerpt_char_limit": full_reranker_excerpt_char_limit,
    "query_count": len(full_reranker_results),
    "parse_success_count": full_reranker_parse_success_count,
    "parse_success_rate": (
        full_reranker_parse_success_count
        / len(full_reranker_results)
    ),
    "total_observed_cost": full_reranker_total_cost,
    "results": full_reranker_results,
}

full_reranker_report_path.write_text(
    json.dumps(
        full_reranker_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

assert full_reranker_parse_success_count == 553, (
    "One or more reranking responses remain unparseable after repair."
)

print("\nParse success after repair:")
print(
    f"{full_reranker_parse_success_count}/"
    f"{len(full_reranker_results)}"
)
print(f"Total observed cost: {full_reranker_total_cost:.6f}")
print("Report:", full_reranker_report_path)
print("\nFULL RERANKING FAILED-RESPONSE REPAIR: PASS")

<div dir="rtl" align="right">

## اثر کامل Reranking بر بازیابی

اکنون برای همهٔ پرسش‌های Golden Dataset یک ترتیب جدید از ده Chunk اول داریم. در این سل، ترتیب اولیه و ترتیب پس از Reranking مقایسه می‌شوند تا مشخص شود شواهد درست چند بار به رتبه‌های بالاتر منتقل شده‌اند.

Reranking فقط ترتیب ده نتیجهٔ اول را تغییر می‌دهد؛ بنابراین انتظار نداریم Recall@10 افزایش یابد، اما معیارهایی مانند Recall@1، Recall@3 و MRR می‌توانند بهتر شوند.

</div>

In [ ]:
# Code Cell 83
# NB13 — Full-query metric impact of complete cached reranking
# No Arvan API requests are made in this cell.

assert len(full_reranker_results) == 553, (
    "Expected a completed full reranking evaluation."
)
assert all(
    result["response_parse_status"] == "PASS"
    for result in full_reranker_results
), "All full reranking responses must pass before metric evaluation."

full_reranking_result_by_candidate_id = {
    result["candidate_id"]: result
    for result in full_reranker_results
}

assert len(full_reranking_result_by_candidate_id) == 553

full_baseline_rank_by_candidate_id = {}
full_reranked_rank_by_candidate_id = {}
full_reranked_child_indexes_by_candidate_id = {}

for query_index, query_record in enumerate(arvan_hybrid_query_records):
    candidate_id = query_record["candidate_id"]
    seed_id = query_record["source_seed_id"]

    baseline_ranked_indexes = best_rrf_ranked_indexes[query_index]
    baseline_top_k_indexes = baseline_ranked_indexes[
        :full_reranker_top_k
    ]

    reranked_labels = full_reranking_result_by_candidate_id[
        candidate_id
    ]["reranked_labels"]

    assert (
        len(reranked_labels) == full_reranker_top_k
        and set(reranked_labels) == allowed_full_reranker_labels
    ), f"Invalid reranked labels for {candidate_id}."

    reranked_top_k_indexes = np.array(
        [
            int(
                baseline_top_k_indexes[
                    int(label.removeprefix("P")) - 1
                ]
            )
            for label in reranked_labels
        ],
        dtype=int,
    )

    assert set(reranked_top_k_indexes) == set(baseline_top_k_indexes)

    reranked_indexes = np.concatenate(
        [
            reranked_top_k_indexes,
            baseline_ranked_indexes[full_reranker_top_k:],
        ]
    )

    positive_indexes = best_positive_indexes_by_seed[seed_id]

    baseline_positive_positions = np.flatnonzero(
        np.isin(
            baseline_ranked_indexes,
            list(positive_indexes),
        )
    )
    reranked_positive_positions = np.flatnonzero(
        np.isin(
            reranked_indexes,
            list(positive_indexes),
        )
    )

    assert len(baseline_positive_positions) > 0
    assert len(reranked_positive_positions) > 0

    full_baseline_rank_by_candidate_id[candidate_id] = (
        int(baseline_positive_positions[0]) + 1
    )
    full_reranked_rank_by_candidate_id[candidate_id] = (
        int(reranked_positive_positions[0]) + 1
    )
    full_reranked_child_indexes_by_candidate_id[candidate_id] = (
        reranked_indexes
    )

assert len(full_baseline_rank_by_candidate_id) == 553
assert len(full_reranked_rank_by_candidate_id) == 553

def full_reranking_metrics(rank_by_candidate_id):
    ranks = list(rank_by_candidate_id.values())

    return {
        "recall_at_1": float(np.mean([rank <= 1 for rank in ranks])),
        "recall_at_3": float(np.mean([rank <= 3 for rank in ranks])),
        "recall_at_5": float(np.mean([rank <= 5 for rank in ranks])),
        "recall_at_10": float(np.mean([rank <= 10 for rank in ranks])),
        "mrr": float(np.mean([1 / rank for rank in ranks])),
        "mean_first_positive_rank": float(np.mean(ranks)),
    }

full_baseline_metrics = full_reranking_metrics(
    full_baseline_rank_by_candidate_id
)
full_reranked_metrics = full_reranking_metrics(
    full_reranked_rank_by_candidate_id
)

full_reranking_metric_impact = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "retrieval_configuration": (
        "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
    ),
    "reranker_model": arvan_chat_model,
    "candidate": best_candidate_name,
    "query_count": len(full_reranked_rank_by_candidate_id),
    "reranking_input_top_k": full_reranker_top_k,
    "baseline_metrics": full_baseline_metrics,
    "reranked_metrics": full_reranked_metrics,
    "metric_deltas": {
        metric_name: (
            full_reranked_metrics[metric_name]
            - full_baseline_metrics[metric_name]
        )
        for metric_name in full_baseline_metrics
    },
}

full_reranking_metric_impact_path = (
    active_benchmark_run_directory
    / "arvan_chat_full_reranking_metric_impact.json"
)

full_reranking_metric_impact_path.write_text(
    json.dumps(
        full_reranking_metric_impact,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== FULL-QUERY RERANKING METRIC IMPACT ===")
print("Queries evaluated:", len(full_reranked_rank_by_candidate_id))
print("Reranking input top-k:", full_reranker_top_k)

for metric_name in (
    "recall_at_1",
    "recall_at_3",
    "recall_at_5",
    "recall_at_10",
    "mrr",
    "mean_first_positive_rank",
):
    print(
        f"\n{metric_name}"
        f"\n- baseline: {full_baseline_metrics[metric_name]:.4f}"
        f"\n- reranked: {full_reranked_metrics[metric_name]:.4f}"
        f"\n- delta: "
        f"{full_reranked_metrics[metric_name] - full_baseline_metrics[metric_name]:+.4f}"
    )

print("\nReport:", full_reranking_metric_impact_path)
print("\nFULL-QUERY RERANKING METRIC IMPACT: PASS")

<div dir="rtl" align="right">

## اثر کامل Reranking بر کانتکست والد–فرزند

در این سل، برای همهٔ پرسش‌ها بررسی می‌شود که ترتیب جدیدِ Chunkها چه اثری بر ورود Parent حاوی شواهد درست به کانتکست نهایی دارد.

این ارزیابی دیگر وابسته به دانستن پاسخ درست در زمان اجرا نیست: Reranking برای همهٔ پرسش‌ها انجام شده و ترتیب خروجی آن برای همهٔ آن‌ها در کش ثبت شده است.

</div>

In [ ]:
# Code Cell 84
# NB13 — Full reranking impact on expansive parent-context retention
# No Arvan API requests are made in this cell.

selected_context_policy = next(
    policy
    for policy in context_budget_policies
    if policy["policy_id"] == "expansive"
)

full_reranking_context_rows = []

for query_index, query_record in enumerate(arvan_hybrid_query_records):
    candidate_id = query_record["candidate_id"]
    seed_id = query_record["source_seed_id"]

    baseline_ranked_child_indexes = best_rrf_ranked_indexes[query_index]
    reranked_child_indexes = (
        full_reranked_child_indexes_by_candidate_id[candidate_id]
    )

    positive_child_indexes = best_positive_indexes_by_seed[seed_id]
    positive_parent_ids = {
        best_candidate_records[int(child_index)]["parent_chunk_id"]
        for child_index in positive_child_indexes
    }

    baseline_context = assemble_parent_context_from_child_indexes(
        baseline_ranked_child_indexes,
        selected_context_policy,
    )
    reranked_context = assemble_parent_context_from_child_indexes(
        reranked_child_indexes,
        selected_context_policy,
    )

    baseline_parent_hit = bool(
        positive_parent_ids.intersection(
            baseline_context["selected_parent_chunk_ids"]
        )
    )
    reranked_parent_hit = bool(
        positive_parent_ids.intersection(
            reranked_context["selected_parent_chunk_ids"]
        )
    )

    full_reranking_context_rows.append(
        {
            "candidate_id": candidate_id,
            "source_seed_id": seed_id,
            "baseline_parent_evidence_retained": baseline_parent_hit,
            "reranked_parent_evidence_retained": reranked_parent_hit,
            "baseline_parent_chunk_count": len(
                baseline_context["selected_parent_chunk_ids"]
            ),
            "reranked_parent_chunk_count": len(
                reranked_context["selected_parent_chunk_ids"]
            ),
            "baseline_context_tokens": baseline_context["token_count"],
            "reranked_context_tokens": reranked_context["token_count"],
            "missing_parent_references": (
                baseline_context["missing_parent_references"]
                + reranked_context["missing_parent_references"]
            ),
        }
    )

assert len(full_reranking_context_rows) == 553
assert sum(
    row["missing_parent_references"]
    for row in full_reranking_context_rows
) == 0, "A selected Child referenced a missing Parent."

baseline_parent_retention = float(
    np.mean(
        [
            row["baseline_parent_evidence_retained"]
            for row in full_reranking_context_rows
        ]
    )
)

reranked_parent_retention = float(
    np.mean(
        [
            row["reranked_parent_evidence_retained"]
            for row in full_reranking_context_rows
        ]
    )
)

improved_parent_retention_count = sum(
    (
        not row["baseline_parent_evidence_retained"]
        and row["reranked_parent_evidence_retained"]
    )
    for row in full_reranking_context_rows
)

worsened_parent_retention_count = sum(
    (
        row["baseline_parent_evidence_retained"]
        and not row["reranked_parent_evidence_retained"]
    )
    for row in full_reranking_context_rows
)

unchanged_parent_retention_count = (
    len(full_reranking_context_rows)
    - improved_parent_retention_count
    - worsened_parent_retention_count
)

full_reranking_context_impact = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "retrieval_configuration": (
        "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
    ),
    "reranker_model": arvan_chat_model,
    "context_policy": selected_context_policy,
    "query_count": len(full_reranking_context_rows),
    "baseline_parent_evidence_retention": baseline_parent_retention,
    "reranked_parent_evidence_retention": reranked_parent_retention,
    "parent_evidence_retention_delta": (
        reranked_parent_retention - baseline_parent_retention
    ),
    "improved_parent_retention_count": improved_parent_retention_count,
    "worsened_parent_retention_count": worsened_parent_retention_count,
    "unchanged_parent_retention_count": unchanged_parent_retention_count,
    "rows": full_reranking_context_rows,
}

full_reranking_context_impact_path = (
    active_benchmark_run_directory
    / "arvan_chat_full_reranking_parent_context_impact.json"
)

full_reranking_context_impact_path.write_text(
    json.dumps(
        full_reranking_context_impact,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== FULL RERANKING PARENT-CONTEXT IMPACT ===")
print("Context policy:", selected_context_policy["policy_id"])
print("Queries evaluated:", len(full_reranking_context_rows))

print("\nParent-evidence retention:")
print(f"- baseline: {baseline_parent_retention:.4f}")
print(f"- reranked: {reranked_parent_retention:.4f}")
print(
    "- delta: "
    f"{reranked_parent_retention - baseline_parent_retention:+.4f}"
)

print("\nContext evidence changes:")
print("- improved:", improved_parent_retention_count)
print("- worsened:", worsened_parent_retention_count)
print("- unchanged:", unchanged_parent_retention_count)

print("\nReport:", full_reranking_context_impact_path)
print("\nFULL RERANKING PARENT-CONTEXT IMPACT: PASS")

<div dir="rtl" align="right">

## پایلوت بازنویسی پرسش برای بازیابی

در این آزمایش، مدل فقط خودِ پرسش را به یک عبارت جست‌وجوی فارسی دقیق‌تر تبدیل می‌کند. مدل به منبع، شواهد Golden Dataset و پاسخ مورد انتظار دسترسی ندارد و اجازه ندارد واقعیت، نام، عدد یا مفهوم تازه‌ای به پرسش اضافه کند.

هدف این است که ببینیم حذف عبارت‌های کلی و تبدیل پرسش به زبان مناسب بازیابی، می‌تواند شواهد درست را بیشتر وارد نتایج اولیه کند یا نه. برای پوشش متوازن، از هر Seed قابل‌ارزیابی یک پرسش به‌صورت قطعی انتخاب می‌شود.

</div>

In [ ]:
# Code Cell 85
# NB13 — Resumable query-rewriting pilot for retrieval
# The model sees only the question, never Golden evidence or answers.

import json
import re
import time

query_rewrite_pilot_cache_path = (
    active_benchmark_run_directory
    / "arvan_chat_query_rewrite_pilot_cache.json"
)

query_rewrite_pilot_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_query_rewrite_pilot.json"
)

# One deterministic question per eligible Seed.
query_rewrite_pilot_records = []

for seed_id in common_eligible_seed_ids:
    seed_records = sorted(
        [
            record
            for record in arvan_hybrid_query_records
            if record["source_seed_id"] == seed_id
        ],
        key=lambda record: record["candidate_id"],
    )

    assert seed_records, f"No eligible query was found for seed {seed_id}."

    query_rewrite_pilot_records.append(seed_records[0])

assert len(query_rewrite_pilot_records) == len(common_eligible_seed_ids)

def extract_search_query(raw_response_text):
    raw_response_text = (raw_response_text or "").strip()

    json_candidates = [raw_response_text]
    json_candidates.extend(
        re.findall(
            r"\{(?:[^{}]|(?R))*\}",
            raw_response_text,
        )
        if False
        else re.findall(r"\{.*?\}", raw_response_text, flags=re.S)
    )

    for candidate_text in json_candidates:
        try:
            parsed = json.loads(candidate_text)
        except json.JSONDecodeError:
            continue

        search_query = parsed.get("search_query")

        if isinstance(search_query, str) and search_query.strip():
            return search_query.strip()

    return None

if query_rewrite_pilot_cache_path.is_file():
    query_rewrite_pilot_cache = json.loads(
        query_rewrite_pilot_cache_path.read_text(encoding="utf-8")
    )
else:
    query_rewrite_pilot_cache = {
        "active_golden_dataset_sha256": golden_dataset_context["sha256"],
        "model": arvan_chat_model,
        "selection_policy": (
            "one_lexicographically_first_candidate_id_per_eligible_seed"
        ),
        "results_by_candidate_id": {},
    }

assert (
    query_rewrite_pilot_cache["active_golden_dataset_sha256"]
    == golden_dataset_context["sha256"]
), "The query-rewrite cache belongs to a different Golden Dataset."

assert query_rewrite_pilot_cache["model"] == arvan_chat_model, (
    "The query-rewrite cache belongs to a different chat model."
)

print("=== QUERY-REWRITE RETRIEVAL PILOT ===")
print("Eligible Seeds:", len(common_eligible_seed_ids))
print("Pilot queries:", len(query_rewrite_pilot_records))
print(
    "Previously successful:",
    sum(
        result["response_parse_status"] == "PASS"
        for result in query_rewrite_pilot_cache[
            "results_by_candidate_id"
        ].values()
    ),
)

for pilot_position, query_record in enumerate(
    query_rewrite_pilot_records,
    start=1,
):
    candidate_id = query_record["candidate_id"]

    cached_result = query_rewrite_pilot_cache[
        "results_by_candidate_id"
    ].get(candidate_id)

    if (
        cached_result is not None
        and cached_result["response_parse_status"] == "PASS"
    ):
        print(
            f"Pilot {pilot_position}/{len(query_rewrite_pilot_records)} "
            f"| cached | seed={query_record['source_seed_id']}"
        )
        continue

    rewrite_prompt = f"""
پرسش زیر را فقط برای جست‌وجو در یک پایگاه فارسیِ آموزش مالی بازنویسی کن.

قواعد قطعی:
- فقط منظور موجود در خود پرسش را حفظ کن.
- هیچ واقعیت، عدد، نام، تاریخ، مثال یا مفهوم تازه‌ای اضافه نکن.
- عبارت‌های کلی مانند «برای یک فرد آگاه مالی» را حذف کن.
- خروجی باید کوتاه، دقیق و مناسب جست‌وجوی معنایی و واژگانی باشد.
- فقط یک JSON معتبر و بدون هیچ متن دیگری برگردان.

ساختار دقیق خروجی:
{{"search_query":"..."}}

پرسش اصلی:
{query_record["question"]}
""".strip()

    attempts = []
    rewritten_query = None

    for attempt_number, max_tokens in enumerate((512, 1024), start=1):
        started_at = time.perf_counter()

        response = arvan_chat_client.chat.completions.create(
            model=arvan_chat_model,
            messages=[
                {
                    "role": "user",
                    "content": rewrite_prompt,
                }
            ],
            temperature=0,
            max_tokens=max_tokens,
        )

        elapsed_seconds = time.perf_counter() - started_at
        choice = response.choices[0]
        raw_response_text = (choice.message.content or "").strip()

        rewritten_query = extract_search_query(raw_response_text)

        attempts.append(
            {
                "attempt_number": attempt_number,
                "max_tokens": max_tokens,
                "finish_reason": choice.finish_reason,
                "seconds": round(elapsed_seconds, 3),
                "cost": getattr(
                    getattr(response, "usage", None),
                    "cost",
                    None,
                ),
                "raw_response": raw_response_text,
                "parsed": rewritten_query is not None,
            }
        )

        if rewritten_query is not None:
            break

    result = {
        "candidate_id": candidate_id,
        "source_seed_id": query_record["source_seed_id"],
        "original_question": query_record["question"],
        "rewritten_query": rewritten_query,
        "response_parse_status": (
            "PASS"
            if rewritten_query is not None
            else "FAIL"
        ),
        "attempts": attempts,
    }

    query_rewrite_pilot_cache["results_by_candidate_id"][
        candidate_id
    ] = result

    query_rewrite_pilot_cache_path.write_text(
        json.dumps(
            query_rewrite_pilot_cache,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        f"Pilot {pilot_position}/{len(query_rewrite_pilot_records)} | "
        f"seed={query_record['source_seed_id']} | "
        f"parse={result['response_parse_status']}"
    )

query_rewrite_pilot_results = [
    query_rewrite_pilot_cache["results_by_candidate_id"][
        record["candidate_id"]
    ]
    for record in query_rewrite_pilot_records
]

query_rewrite_pilot_successes = [
    result
    for result in query_rewrite_pilot_results
    if result["response_parse_status"] == "PASS"
]

query_rewrite_pilot_total_cost = sum(
    float(attempt["cost"])
    for result in query_rewrite_pilot_results
    for attempt in result["attempts"]
    if attempt.get("cost") is not None
)

query_rewrite_pilot_report = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "model": arvan_chat_model,
    "eligible_seed_count": len(common_eligible_seed_ids),
    "pilot_query_count": len(query_rewrite_pilot_results),
    "parse_success_count": len(query_rewrite_pilot_successes),
    "parse_success_rate": (
        len(query_rewrite_pilot_successes)
        / len(query_rewrite_pilot_results)
    ),
    "total_observed_cost": query_rewrite_pilot_total_cost,
    "results": query_rewrite_pilot_results,
}

query_rewrite_pilot_report_path.write_text(
    json.dumps(
        query_rewrite_pilot_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

assert len(query_rewrite_pilot_successes) == len(
    query_rewrite_pilot_records
), "One or more query rewrites could not be parsed."

print("\nParse success:")
print(
    f"{len(query_rewrite_pilot_successes)}/"
    f"{len(query_rewrite_pilot_results)}"
)
print(f"Total observed cost: {query_rewrite_pilot_total_cost:.6f}")
print("Cache:", query_rewrite_pilot_cache_path)
print("Report:", query_rewrite_pilot_report_path)
print("\nQUERY-REWRITE RETRIEVAL PILOT: PASS")

In [ ]:
# Code Cell 86
# NB13 — Retry only failed query-rewrite pilot responses.

failed_query_rewrite_results = [
    result
    for result in query_rewrite_pilot_cache[
        "results_by_candidate_id"
    ].values()
    if result["response_parse_status"] != "PASS"
]

pilot_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in query_rewrite_pilot_records
}

print("=== QUERY-REWRITE FAILED-RESPONSE REPAIR ===")
print("Failed responses before repair:", len(failed_query_rewrite_results))

for repair_position, failed_result in enumerate(
    failed_query_rewrite_results,
    start=1,
):
    candidate_id = failed_result["candidate_id"]
    query_record = pilot_record_by_candidate_id[candidate_id]

    repair_prompt = f"""
پرسش زیر را فقط برای جست‌وجو در یک پایگاه فارسیِ آموزش مالی بازنویسی کن.

قواعد قطعی:
- فقط منظور موجود در خود پرسش را حفظ کن.
- هیچ واقعیت، عدد، نام، تاریخ، مثال یا مفهوم تازه‌ای اضافه نکن.
- عبارت‌های کلی مانند «برای یک فرد آگاه مالی» را حذف کن.
- خروجی باید کوتاه، دقیق و مناسب جست‌وجوی معنایی و واژگانی باشد.
- فقط یک JSON معتبر و بدون هیچ متن دیگری برگردان.

ساختار دقیق خروجی:
{{"search_query":"..."}}

پرسش اصلی:
{query_record["question"]}
""".strip()

    repair_attempts = []
    rewritten_query = None

    for attempt_number, max_tokens in enumerate((2048, 3072), start=1):
        started_at = time.perf_counter()

        response = arvan_chat_client.chat.completions.create(
            model=arvan_chat_model,
            messages=[
                {
                    "role": "user",
                    "content": repair_prompt,
                }
            ],
            temperature=0,
            max_tokens=max_tokens,
        )

        elapsed_seconds = time.perf_counter() - started_at
        choice = response.choices[0]
        raw_response_text = (choice.message.content or "").strip()

        rewritten_query = extract_search_query(raw_response_text)

        repair_attempts.append(
            {
                "repair_attempt_number": attempt_number,
                "max_tokens": max_tokens,
                "finish_reason": choice.finish_reason,
                "seconds": round(elapsed_seconds, 3),
                "cost": getattr(
                    getattr(response, "usage", None),
                    "cost",
                    None,
                ),
                "raw_response": raw_response_text,
                "parsed": rewritten_query is not None,
            }
        )

        if rewritten_query is not None:
            break

    failed_result.setdefault("repair_attempts", []).extend(
        repair_attempts
    )

    if rewritten_query is not None:
        failed_result["rewritten_query"] = rewritten_query
        failed_result["response_parse_status"] = "PASS"

    query_rewrite_pilot_cache["results_by_candidate_id"][
        candidate_id
    ] = failed_result

    query_rewrite_pilot_cache_path.write_text(
        json.dumps(
            query_rewrite_pilot_cache,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        f"Repair {repair_position}/{len(failed_query_rewrite_results)} | "
        f"seed={failed_result['source_seed_id']} | "
        f"parse={failed_result['response_parse_status']}"
    )

query_rewrite_pilot_results = [
    query_rewrite_pilot_cache["results_by_candidate_id"][
        record["candidate_id"]
    ]
    for record in query_rewrite_pilot_records
]

query_rewrite_pilot_successes = [
    result
    for result in query_rewrite_pilot_results
    if result["response_parse_status"] == "PASS"
]

query_rewrite_pilot_total_cost = sum(
    float(attempt["cost"])
    for result in query_rewrite_pilot_results
    for attempt_group in (
        result.get("attempts", []),
        result.get("repair_attempts", []),
    )
    for attempt in attempt_group
    if attempt.get("cost") is not None
)

query_rewrite_pilot_report = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "model": arvan_chat_model,
    "eligible_seed_count": len(common_eligible_seed_ids),
    "pilot_query_count": len(query_rewrite_pilot_results),
    "parse_success_count": len(query_rewrite_pilot_successes),
    "parse_success_rate": (
        len(query_rewrite_pilot_successes)
        / len(query_rewrite_pilot_results)
    ),
    "total_observed_cost": query_rewrite_pilot_total_cost,
    "results": query_rewrite_pilot_results,
}

query_rewrite_pilot_report_path.write_text(
    json.dumps(
        query_rewrite_pilot_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

assert len(query_rewrite_pilot_successes) == 27, (
    "One or more query rewrites remain unparseable after repair."
)

print("\nParse success after repair:")
print(
    f"{len(query_rewrite_pilot_successes)}/"
    f"{len(query_rewrite_pilot_results)}"
)
print(f"Total observed cost: {query_rewrite_pilot_total_cost:.6f}")
print("Report:", query_rewrite_pilot_report_path)
print("\nQUERY-REWRITE FAILED-RESPONSE REPAIR: PASS")

<div dir="rtl" align="right">

## ارزیابی بازیابی با پرسش‌های بازنویسی‌شده

در این سل، ۲۷ پرسش بازنویسی‌شده با همان مدل embedding اروان و همان TF‑IDF فارسیِ منتخب ارزیابی می‌شوند. بنابراین تنها متغیرِ تغییرکرده، خود عبارت جست‌وجو است.

Embedding این ۲۷ عبارت نیز کش می‌شود؛ تغییر نام Notebook یا Restart باعث تکرار آن نمی‌شود.

</div>

In [ ]:
# Code Cell 87
# NB13 — Retrieval evaluation for the cached query-rewrite pilot
# Sends at most one small Arvan embedding request for 27 rewritten queries.

query_rewrite_embedding_cache_path = (
    active_benchmark_run_directory
    / "arvan_query_rewrite_pilot_embeddings.npz"
)

query_rewrite_embedding_manifest_path = (
    active_benchmark_run_directory
    / "arvan_query_rewrite_pilot_embeddings_manifest.json"
)

query_rewrite_pilot_result_by_candidate_id = {
    result["candidate_id"]: result
    for result in query_rewrite_pilot_results
}

pilot_candidate_ids = [
    record["candidate_id"]
    for record in query_rewrite_pilot_records
]

pilot_rewritten_queries = [
    query_rewrite_pilot_result_by_candidate_id[candidate_id][
        "rewritten_query"
    ]
    for candidate_id in pilot_candidate_ids
]

assert len(pilot_candidate_ids) == 27
assert all(
    isinstance(query, str) and query.strip()
    for query in pilot_rewritten_queries
)

pilot_rewrite_input_sha256 = hashlib.sha256(
    "\n".join(
        f"{candidate_id}\t{query}"
        for candidate_id, query in zip(
            pilot_candidate_ids,
            pilot_rewritten_queries,
        )
    ).encode("utf-8")
).hexdigest()

pilot_embedding_cache_reused = False

if (
    query_rewrite_embedding_cache_path.is_file()
    and query_rewrite_embedding_manifest_path.is_file()
):
    pilot_embedding_manifest = json.loads(
        query_rewrite_embedding_manifest_path.read_text(
            encoding="utf-8"
        )
    )

    cached_arrays = np.load(
        query_rewrite_embedding_cache_path,
        allow_pickle=True,
    )

    cached_candidate_ids = [
        str(candidate_id)
        for candidate_id in cached_arrays["candidate_ids"].tolist()
    ]

    pilot_embedding_cache_reused = (
        pilot_embedding_manifest.get("golden_dataset_sha256")
        == golden_dataset_context["sha256"]
        and pilot_embedding_manifest.get("input_sha256")
        == pilot_rewrite_input_sha256
        and cached_candidate_ids == pilot_candidate_ids
        and cached_arrays["vectors"].shape
        == (len(pilot_candidate_ids), 3072)
    )

if pilot_embedding_cache_reused:
    rewritten_query_vectors = cached_arrays["vectors"].astype(
        np.float32
    )
    print("Existing rewritten-query embedding cache loaded.")
else:
    response = arvan_embedding_client.embeddings.create(
        model=arvan_embedding_model_id,
        input=pilot_rewritten_queries,
    )

    response_items = sorted(
        response.data,
        key=lambda item: item.index,
    )

    assert len(response_items) == len(pilot_rewritten_queries)

    rewritten_query_vectors = np.asarray(
        [item.embedding for item in response_items],
        dtype=np.float32,
    )

    rewritten_query_norms = np.linalg.norm(
        rewritten_query_vectors,
        axis=1,
        keepdims=True,
    )

    assert np.all(rewritten_query_norms > 0), (
        "A rewritten query received a zero embedding vector."
    )

    rewritten_query_vectors = (
        rewritten_query_vectors / rewritten_query_norms
    ).astype(np.float32)

    np.savez_compressed(
        query_rewrite_embedding_cache_path,
        candidate_ids=np.asarray(pilot_candidate_ids, dtype=object),
        vectors=rewritten_query_vectors,
    )

    query_rewrite_embedding_manifest = {
        "artifact": query_rewrite_embedding_cache_path.name,
        "golden_dataset_sha256": golden_dataset_context["sha256"],
        "embedding_model": arvan_embedding_model_id,
        "embedding_dimension": 3072,
        "input_sha256": pilot_rewrite_input_sha256,
        "query_count": len(pilot_candidate_ids),
    }

    query_rewrite_embedding_manifest_path.write_text(
        json.dumps(
            query_rewrite_embedding_manifest,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

assert rewritten_query_vectors.shape == (27, 3072)

pilot_lexical_query_matrix = best_lexical_vectorizer.transform(
    pilot_rewritten_queries
)

pilot_dense_scores = (
    rewritten_query_vectors @ best_arvan_candidate_matrix.T
)
pilot_lexical_scores = (
    pilot_lexical_query_matrix @ best_lexical_candidate_matrix.T
).toarray()

pilot_dense_ranked_indexes = np.argsort(
    -pilot_dense_scores,
    axis=1,
)
pilot_lexical_ranked_indexes = np.argsort(
    -pilot_lexical_scores,
    axis=1,
)

pilot_dense_rank_positions = np.empty_like(
    pilot_dense_ranked_indexes
)
pilot_lexical_rank_positions = np.empty_like(
    pilot_lexical_ranked_indexes
)

pilot_row_indexes = np.arange(len(query_rewrite_pilot_records))[:, None]
pilot_candidate_rank_positions = (
    np.arange(len(best_candidate_records))[None, :] + 1
)

pilot_dense_rank_positions[
    pilot_row_indexes,
    pilot_dense_ranked_indexes,
] = pilot_candidate_rank_positions

pilot_lexical_rank_positions[
    pilot_row_indexes,
    pilot_lexical_ranked_indexes,
] = pilot_candidate_rank_positions

pilot_rrf_scores = (
    1.0 / (rrf_constant + pilot_dense_rank_positions)
    + 1.0 / (rrf_constant + pilot_lexical_rank_positions)
)

pilot_rewritten_ranked_indexes = np.argsort(
    -pilot_rrf_scores,
    axis=1,
)

pilot_baseline_ranked_indexes = np.vstack(
    [
        best_rrf_ranked_indexes[
            next(
                index
                for index, record in enumerate(
                    arvan_hybrid_query_records
                )
                if record["candidate_id"] == pilot_record["candidate_id"]
            )
        ]
        for pilot_record in query_rewrite_pilot_records
    ]
)

pilot_baseline_metrics = calculate_retrieval_metrics(
    pilot_baseline_ranked_indexes,
    query_rewrite_pilot_records,
    best_positive_indexes_by_seed,
)

pilot_rewritten_metrics = calculate_retrieval_metrics(
    pilot_rewritten_ranked_indexes,
    query_rewrite_pilot_records,
    best_positive_indexes_by_seed,
)

query_rewrite_retrieval_pilot_report = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "retrieval_configuration": (
        "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
    ),
    "candidate": best_candidate_name,
    "pilot_query_count": len(query_rewrite_pilot_records),
    "embedding_cache_reused": pilot_embedding_cache_reused,
    "baseline_metrics": pilot_baseline_metrics,
    "rewritten_query_metrics": pilot_rewritten_metrics,
    "metric_deltas": {
        metric_name: (
            pilot_rewritten_metrics[metric_name]
            - pilot_baseline_metrics[metric_name]
        )
        for metric_name in (
            "recall_at_1",
            "recall_at_3",
            "recall_at_5",
            "recall_at_10",
            "mrr",
            "mean_first_positive_rank",
        )
    },
    "queries": [
        {
            "candidate_id": record["candidate_id"],
            "source_seed_id": record["source_seed_id"],
            "original_question": record["question"],
            "rewritten_query": query_rewrite_pilot_result_by_candidate_id[
                record["candidate_id"]
            ]["rewritten_query"],
        }
        for record in query_rewrite_pilot_records
    ],
}

query_rewrite_retrieval_pilot_report_path = (
    active_benchmark_run_directory
    / "arvan_query_rewrite_retrieval_pilot.json"
)

query_rewrite_retrieval_pilot_report_path.write_text(
    json.dumps(
        query_rewrite_retrieval_pilot_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== QUERY-REWRITE RETRIEVAL PILOT ===")
print("Pilot queries:", len(query_rewrite_pilot_records))
print("Embedding cache reused:", pilot_embedding_cache_reused)

for metric_name in (
    "recall_at_1",
    "recall_at_3",
    "recall_at_5",
    "recall_at_10",
    "mrr",
    "mean_first_positive_rank",
):
    print(
        f"\n{metric_name}"
        f"\n- baseline: {pilot_baseline_metrics[metric_name]:.4f}"
        f"\n- rewritten: {pilot_rewritten_metrics[metric_name]:.4f}"
        f"\n- delta: "
        f"{pilot_rewritten_metrics[metric_name] - pilot_baseline_metrics[metric_name]:+.4f}"
    )

print("\nEmbedding cache:", query_rewrite_embedding_cache_path)
print("Report:", query_rewrite_retrieval_pilot_report_path)
print("\nQUERY-REWRITE RETRIEVAL PILOT: PASS")

<div dir="rtl" align="right">

## ترکیب پرسش اصلی و پرسش بازنویسی‌شده

نتیجهٔ قبلی نشان داد جایگزین‌کردن کامل پرسش اصلی با نسخهٔ بازنویسی‌شده تصمیم خوبی نیست؛ چون ممکن است بخشی از اطلاعات مفید پرسش اصلی از بین برود.

در این آزمایش، هر دو پرسش حفظ می‌شوند. رتبه‌بندی حاصل از پرسش اصلی و پرسش بازنویسی‌شده با چند وزن مختلف ترکیب می‌شود تا مشخص شود آیا بازنویسی، در کنار پرسش اصلی، به بازیابی کمک می‌کند یا نه. این سل فقط از embeddingها و خروجی‌های موجود استفاده می‌کند و هیچ فراخوانی جدیدی ندارد.

</div>

In [ ]:
# Code Cell 88
# NB13 — Query-rewrite augmentation pilot:
# original query + rewritten query, fused without new API requests.

pilot_query_indexes_in_full_benchmark = np.array(
    [
        next(
            index
            for index, record in enumerate(arvan_hybrid_query_records)
            if record["candidate_id"] == pilot_record["candidate_id"]
        )
        for pilot_record in query_rewrite_pilot_records
    ],
    dtype=int,
)

assert len(pilot_query_indexes_in_full_benchmark) == 27
assert len(set(pilot_query_indexes_in_full_benchmark)) == 27

pilot_original_questions = [
    record["question"]
    for record in query_rewrite_pilot_records
]

pilot_original_dense_scores = (
    best_arvan_query_matrix[
        pilot_query_indexes_in_full_benchmark
    ]
    @ best_arvan_candidate_matrix.T
)

pilot_original_lexical_query_matrix = best_lexical_vectorizer.transform(
    pilot_original_questions
)

pilot_original_lexical_scores = (
    pilot_original_lexical_query_matrix
    @ best_lexical_candidate_matrix.T
).toarray()

def rank_positions_from_scores(scores):
    ranked_indexes = np.argsort(-scores, axis=1)
    rank_positions = np.empty_like(ranked_indexes)

    row_indexes = np.arange(scores.shape[0])[:, None]
    candidate_rank_positions = (
        np.arange(scores.shape[1])[None, :] + 1
    )

    rank_positions[
        row_indexes,
        ranked_indexes,
    ] = candidate_rank_positions

    return ranked_indexes, rank_positions

(
    pilot_original_dense_ranked_indexes,
    pilot_original_dense_rank_positions,
) = rank_positions_from_scores(pilot_original_dense_scores)

(
    pilot_original_lexical_ranked_indexes,
    pilot_original_lexical_rank_positions,
) = rank_positions_from_scores(pilot_original_lexical_scores)

# These scores were created in Code Cell 87 from rewritten questions.
(
    pilot_rewritten_dense_ranked_indexes,
    pilot_rewritten_dense_rank_positions,
) = rank_positions_from_scores(pilot_dense_scores)

(
    pilot_rewritten_lexical_ranked_indexes,
    pilot_rewritten_lexical_rank_positions,
) = rank_positions_from_scores(pilot_lexical_scores)

rewrite_weight_candidates = (0.0, 0.25, 0.50, 0.75, 1.0)
query_augmentation_results = {}

print("=== QUERY-REWRITE AUGMENTATION PILOT ===")
print("Pilot queries:", len(query_rewrite_pilot_records))
print("Fusion: original dense + original lexical + weighted rewrite")

for rewrite_weight in rewrite_weight_candidates:
    fused_rrf_scores = (
        1.0 / (rrf_constant + pilot_original_dense_rank_positions)
        + 1.0 / (rrf_constant + pilot_original_lexical_rank_positions)
        + rewrite_weight
        * (
            1.0
            / (
                rrf_constant
                + pilot_rewritten_dense_rank_positions
            )
        )
        + rewrite_weight
        * (
            1.0
            / (
                rrf_constant
                + pilot_rewritten_lexical_rank_positions
            )
        )
    )

    fused_ranked_indexes = np.argsort(
        -fused_rrf_scores,
        axis=1,
    )

    metrics = calculate_retrieval_metrics(
        fused_ranked_indexes,
        query_rewrite_pilot_records,
        best_positive_indexes_by_seed,
    )

    query_augmentation_results[str(rewrite_weight)] = {
        "rewrite_weight": rewrite_weight,
        **metrics,
    }

    print(f"\nRewrite weight: {rewrite_weight:.2f}")
    print(f"- Recall@1: {metrics['recall_at_1']:.4f}")
    print(f"- Recall@3: {metrics['recall_at_3']:.4f}")
    print(f"- Recall@5: {metrics['recall_at_5']:.4f}")
    print(f"- Recall@10: {metrics['recall_at_10']:.4f}")
    print(f"- MRR: {metrics['mrr']:.4f}")
    print(
        "- Mean first-positive rank: "
        f"{metrics['mean_first_positive_rank']:.2f}"
    )

best_query_augmentation_result = sorted(
    query_augmentation_results.values(),
    key=lambda result: (
        -result["recall_at_10"],
        -result["mrr"],
        result["mean_first_positive_rank"],
    ),
)[0]

query_augmentation_report = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "retrieval_configuration": (
        "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
    ),
    "pilot_query_count": len(query_rewrite_pilot_records),
    "baseline_original_only": query_augmentation_results["0.0"],
    "results_by_rewrite_weight": query_augmentation_results,
    "best_result": best_query_augmentation_result,
    "interpretation": (
        "This is a small pilot only. A full evaluation is justified "
        "only if augmentation improves the selected pilot metric."
    ),
}

query_augmentation_report_path = (
    active_benchmark_run_directory
    / "arvan_query_rewrite_augmentation_pilot.json"
)

query_augmentation_report_path.write_text(
    json.dumps(
        query_augmentation_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nBest pilot configuration:")
print(
    f"- rewrite_weight="
    f"{best_query_augmentation_result['rewrite_weight']:.2f}"
)
print(
    f"- Recall@10="
    f"{best_query_augmentation_result['recall_at_10']:.4f}"
)
print(f"- MRR={best_query_augmentation_result['mrr']:.4f}")

print("\nReport:", query_augmentation_report_path)
print("\nQUERY-REWRITE AUGMENTATION PILOT: PASS")

<div dir="rtl" align="right">

## تصمیم نهاییِ ارزیابی بازیابی با Golden Dataset فعال

تمام روش‌های قابل‌اجرا در این Notebook بررسی شدند: embeddingهای محلی و اروان، بازیابی واژگانی فارسی، ترکیب رتبه‌ها، Reranking کامل و بازنویسی پرسش.

Reranking ترتیب نتایجی را که از قبل در ده نتیجهٔ اول بودند بهتر می‌کند، اما شواهدی را که وارد ده نتیجهٔ اول نشده‌اند پیدا نمی‌کند. بازنویسی پرسش نیز در پایلوت نتوانست Recall@10 را بهتر کند.

بنابراین با Golden Dataset فعال، هیچ پیکربندی برای استفادهٔ Production تأیید نمی‌شود. این نتیجه به معنی توقف پروژه نیست: اجزای ارزیابی، کش‌ها و قراردادهای داده آماده‌اند و پس از ساخت Golden Dataset جدید، همین Notebook بدون بازنویسی از ابتدا با Dataset فعال جدید اجرا می‌شود.

</div>

In [ ]:
# Code Cell 89
# NB13 — Formal retrieval-remediation decision for the active Golden Dataset
# No external calls are made in this cell.

assert best_hybrid_metrics["recall_at_10"] == full_baseline_metrics[
    "recall_at_10"
], "The baseline metric contracts are inconsistent."

assert full_reranked_metrics["recall_at_10"] == full_baseline_metrics[
    "recall_at_10"
], "Reranking must not change Recall@10 when it reranks only top 10."

assert query_augmentation_results["1.0"]["recall_at_10"] == 0.0, (
    "The query-rewrite augmentation pilot result is unexpected."
)

retrieval_remediation_decision = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "status": "NO_PRODUCTION_APPROVAL_WITH_ACTIVE_GOLDEN_DATASET",
    "evaluated_retrieval_configuration": {
        "dense_model": "arvan_text_embedding_3_large",
        "lexical_model": "persian_tfidf_word_unigram_bigram",
        "fusion_method": "reciprocal_rank_fusion",
        "rrf_constant": rrf_constant,
        "retrieval_candidate": best_candidate_name,
        "context_policy": "expansive",
    },
    "baseline_retrieval": {
        "recall_at_10": full_baseline_metrics["recall_at_10"],
        "mrr": full_baseline_metrics["mrr"],
        "parent_evidence_retention": baseline_parent_retention,
    },
    "full_reranking_result": {
        "model": arvan_chat_model,
        "reranking_input_top_k": full_reranker_top_k,
        "recall_at_10_delta": (
            full_reranked_metrics["recall_at_10"]
            - full_baseline_metrics["recall_at_10"]
        ),
        "mrr_delta": (
            full_reranked_metrics["mrr"]
            - full_baseline_metrics["mrr"]
        ),
        "parent_evidence_retention_delta": (
            reranked_parent_retention
            - baseline_parent_retention
        ),
        "observed_total_cost": full_reranker_total_cost,
        "interpretation": (
            "Reranking improves ordering within the retrieved top 10 "
            "but does not recover evidence absent from that top 10."
        ),
    },
    "query_rewrite_pilot_result": {
        "pilot_query_count": len(query_rewrite_pilot_records),
        "replacement_recall_at_10": (
            pilot_rewritten_metrics["recall_at_10"]
        ),
        "best_augmentation_weight": (
            best_query_augmentation_result["rewrite_weight"]
        ),
        "best_augmentation_recall_at_10": (
            best_query_augmentation_result["recall_at_10"]
        ),
        "interpretation": (
            "Neither replacement nor augmentation improved pilot "
            "Recall@10; full query rewriting is rejected."
        ),
    },
    "rejected_actions": [
        "full_query_rewrite_execution",
        "production_deployment_of_current_retrieval_configuration",
        "claiming_final_chunking_or_model_selection",
    ],
    "accepted_artifacts_for_future_rerun": [
        "frozen_chunking_candidates",
        "embedding_caches",
        "hybrid_retrieval_evaluation_contract",
        "full_reranking_cache",
        "parent_child_context_budget_policy",
        "query_rewrite_pilot_artifacts",
    ],
    "required_next_work": [
        (
            "Complete author-feedback integration and create the next "
            "active Golden Dataset."
        ),
        (
            "Run NB13 again against the new active Golden Dataset; "
            "existing version-agnostic loader will select it."
        ),
        (
            "Compare new results with this active-dataset decision "
            "before any production selection."
        ),
    ],
}

retrieval_remediation_decision_path = (
    active_benchmark_run_directory
    / "retrieval_remediation_decision.json"
)

retrieval_remediation_decision_path.write_text(
    json.dumps(
        retrieval_remediation_decision,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== RETRIEVAL REMEDIATION DECISION ===")
print("Status:", retrieval_remediation_decision["status"])

print("\nBaseline:")
print(
    "- Recall@10:",
    f"{retrieval_remediation_decision['baseline_retrieval']['recall_at_10']:.4f}",
)
print(
    "- MRR:",
    f"{retrieval_remediation_decision['baseline_retrieval']['mrr']:.4f}",
)

print("\nFull reranking:")
print(
    "- Recall@10 delta:",
    f"{retrieval_remediation_decision['full_reranking_result']['recall_at_10_delta']:+.4f}",
)
print(
    "- MRR delta:",
    f"{retrieval_remediation_decision['full_reranking_result']['mrr_delta']:+.4f}",
)
print(
    "- Parent-retention delta:",
    f"{retrieval_remediation_decision['full_reranking_result']['parent_evidence_retention_delta']:+.4f}",
)

print("\nQuery-rewrite pilot:")
print(
    "- Best augmentation Recall@10:",
    f"{retrieval_remediation_decision['query_rewrite_pilot_result']['best_augmentation_recall_at_10']:.4f}",
)
print("- Decision: REJECT_FULL_QUERY_REWRITE")

print("\nNext work:")
for item in retrieval_remediation_decision["required_next_work"]:
    print("-", item)

print("\nReport:", retrieval_remediation_decision_path)
print("\nRETRIEVAL REMEDIATION DECISION: PASS")

<div dir="rtl" align="right">

## Handoff نهایی NB13

این Notebook با Golden Dataset فعال اجرا و نتایج آن ثبت شد. هیچ پیکربندی فعلی برای Production تأیید نشده است، اما همهٔ قراردادهای ارزیابی، نتایج، کش‌ها و مسیر اجرای دوباره آماده‌اند.

پس از ساخت Golden Dataset جدید، Loader ابتدای Notebook به‌صورت خودکار Dataset فعال جدید را انتخاب می‌کند و اجرای مجدد NB13، نتایج جدید را در پوشه‌ای جدا بر اساس هش Dataset ثبت خواهد کرد. نام Notebook در این سازوکار هیچ نقشی ندارد.

</div>

In [ ]:
# Code Cell 90
# NB13 — Final reproducibility handoff for the active Golden Dataset.

from datetime import datetime, timezone

nb13_required_artifacts = {
    "benchmark_manifest": embedding_benchmark_manifest_path,
    "e5_instructional_cache": e5_instructional_cache_path,
    "full_reranking_cache": full_reranker_cache_path,
    "full_reranking_report": full_reranker_report_path,
    "full_reranking_metric_impact": (
        full_reranking_metric_impact_path
    ),
    "full_reranking_context_impact": (
        full_reranking_context_impact_path
    ),
    "query_rewrite_pilot_report": (
        query_rewrite_pilot_report_path
    ),
    "query_rewrite_augmentation_report": (
        query_augmentation_report_path
    ),
    "remediation_decision": retrieval_remediation_decision_path,
}

missing_nb13_artifacts = [
    artifact_name
    for artifact_name, artifact_path in nb13_required_artifacts.items()
    if not artifact_path.is_file()
]

assert not missing_nb13_artifacts, (
    "NB13 handoff is incomplete. Missing artifacts: "
    f"{missing_nb13_artifacts}"
)

nb13_final_handoff = {
    "notebook": "13_embedding_benchmark",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "active_golden_dataset": {
        "path": str(golden_dataset_path),
        "sha256": golden_dataset_context["sha256"],
        "record_count": golden_dataset_context["record_count"],
    },
    "notebook_status": (
        "COMPLETE_WITH_NO_PRODUCTION_APPROVAL"
    ),
    "decision": {
        "status": retrieval_remediation_decision["status"],
        "baseline_recall_at_10": full_baseline_metrics["recall_at_10"],
        "baseline_mrr": full_baseline_metrics["mrr"],
        "reranked_mrr": full_reranked_metrics["mrr"],
        "reranking_cost": full_reranker_total_cost,
        "full_query_rewrite_status": "REJECTED_BY_PILOT",
    },
    "reproducibility_policy": {
        "dataset_selection": (
            "The loader selects the highest valid versioned "
            "Golden Dataset file."
        ),
        "cache_scope": (
            "All benchmark artifacts are stored under the active "
            "Golden Dataset SHA-256."
        ),
        "notebook_filename_affects_cache": False,
        "rerun_after_new_active_dataset": (
            "Required. A new dataset hash intentionally creates a "
            "separate benchmark run directory."
        ),
    },
    "required_next_work": (
        retrieval_remediation_decision["required_next_work"]
    ),
    "artifacts": {
        artifact_name: str(artifact_path)
        for artifact_name, artifact_path
        in nb13_required_artifacts.items()
    },
}

nb13_final_handoff_path = (
    active_benchmark_run_directory
    / "nb13_final_handoff.json"
)

nb13_final_handoff_path.write_text(
    json.dumps(
        nb13_final_handoff,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== NB13 FINAL HANDOFF ===")
print("Notebook status:", nb13_final_handoff["notebook_status"])
print(
    "Active Golden Dataset records:",
    nb13_final_handoff["active_golden_dataset"]["record_count"],
)
print(
    "Active Golden Dataset SHA-256:",
    nb13_final_handoff["active_golden_dataset"]["sha256"],
)
print(
    "Production approval:",
    nb13_final_handoff["decision"]["status"],
)
print(
    "Notebook filename affects cache:",
    nb13_final_handoff["reproducibility_policy"][
        "notebook_filename_affects_cache"
    ],
)
print("Artifacts verified:", len(nb13_required_artifacts))
print("Handoff:", nb13_final_handoff_path)
print("\nNB13 FINAL HANDOFF: PASS")

<div style="direction: rtl; white-space: normal; line-height: 1;">
NB13 درست اجرا شد و کامل بسته شد؛ از نظر فنی خطای بازمانده نداریم.

اما نتیجهٔ علمی‌اش این است که با Golden Dataset فعلی، سیستم هنوز برای Production تأیید نمی‌شود:

* بهترین بازیابی فعلی: **Recall@10 = 6.87٪**
  یعنی از هر ۱۰۰ پرسش، فقط حدود ۷ بار شواهد درست در ده نتیجهٔ اول می‌آیند.
* Reranking کامل تست شد: ترتیب نتایج را بهتر کرد و MRR را از **0.0342** به **0.0426** رساند، اما چیزی را که در ده نتیجهٔ اول نبود پیدا نکرد؛ پس Recall@10 تغییر نکرد.
* بازنویسی پرسش هم در پایلوت کمکی به ورود شواهد به Top‑10 نکرد؛ بنابراین آگاهانه رد شد.
* همهٔ آزمایش‌ها، کش‌ها، گزارش‌ها و Handoff ذخیره شده‌اند؛ هزینهٔ reranking کامل هم حدود **0.638 دلار** شد.

پس «مشکل اجرایی» نداریم؛ مشکل اصلی کیفیت/تناسب پرسش‌های Golden Dataset فعال با بازیابی است. قدم بعدی درست این است که در `12f` پس از بازخورد دکتر، Golden Dataset جدید بسازیم و سپس همین NB13 را دوباره اجرا کنیم. هیچ نوت‌بوکی از صفر نوشته نمی‌شود.

</div>